# Building Mini Chatbot RAG untuk Riset Saham
- Nama: Gidion Depari
- Batch: DSML 42

Saya membuat Bot untuk menjawab berdasarkan dokumen Analisa IHSG 2026 & Riset saham Unggulan, untuk membantu mencari informasi terkait analisa, pergerakan IHSG selama 2026 (prediksi bottom dan kapan recovery) serta saham unggulan ketika IHSG Rebound.

> **Catatan menjalankan notebook revisi P0**
> Jalankan cell secara berurutan dari atas. Khusus bagian P0, urutannya adalah **Trace Test 1 pertanyaan → periksa retrieved chunks/context → sanity check → benchmark kecil 3 pertanyaan → baru benchmark penuh 125 pertanyaan**. Jangan menjalankan benchmark penuh sebelum Trace Test dan checkpoint kecil berhasil.

## 0. Setup & Instalasi

### Instalasi library

In [1]:
# Import dan setup yang dibutuhkan.

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "langchain": "langchain",
    "langchain_openai": "langchain-openai",
    "langchain_huggingface": "langchain-huggingface",
    "langchain_community": "langchain-community",
    "langchain_chroma": "langchain-chroma",
    "langchain_text_splitters": "langchain-text-splitters",
    "faiss": "faiss-cpu",
    "pypdf": "pypdf",
    "pandas": "pandas",
    "torch": "torch",
    "dotenv": "python-dotenv",
    "sentence_transformers": "sentence-transformers",
    "yfinance": "yfinance",
}

missing = [
    package
    for module, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Package belum tersedia:", ", ".join(missing))
    print("Meng-install package yang hilang...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", *missing])
    print("✅ Instalasi selesai. Jika kernel meminta restart, restart kernel lalu Run All.")
else:
    print("✅ Semua library utama sudah tersedia.")


✅ Semua library utama sudah tersedia.


**Interpretasi output:** Output menunjukkan ✅ Semua library utama sudah tersedia.. Jadi proses pada tahap ini berjalan sesuai alurnya.


### Setup Project

In [2]:
# Import dan setup yang dibutuhkan.

import torch

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Device embedding: {DEVICE}")


Device embedding: mps


**Interpretasi output:** Output menunjukkan Device embedding: mps. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [3]:
# Import dan setup yang dibutuhkan.

import os
from dotenv import load_dotenv

                  
load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

if deepseek_api_key:
    print("DeepSeek API Key berhasil di-load!")
else:
    print("Gagal! DEEPSEEK_API_KEY tidak ditemukan di file .env atau nama variabel salah.")


DeepSeek API Key berhasil di-load!


**Interpretasi output:** Output menunjukkan DeepSeek API Key berhasil di-load!. Jadi proses pada tahap ini berjalan sesuai alurnya.


## 1. Penentuan Topik & Knowledge Base
### 1.1 Topik yang dipilih
Saya memilih topic mengenai Financial Market khusunya di Saham Indonesia. Isinya adalah analisis penurunan IHSG tahun 2026 sebagai peluang major bottom pasar dengan memproyeksikan pembalikan arah yang agresif seiring stabilisasi makroekonomi. Riset ini secara spesifik mengcompare beberapa saham pilihan di sektor perbankan dan energy, untuk mencari saham unggulan dengan return paling tinggi ketika pembalikan arah terjadi (Bullish).

### 1.2 Ruang lingkup knowledge base
**Knowledge base mencakup informasi dan analisis terkait IHSG 2026, saham unggulan, serta riset equity/portfolio tambahan yang dimasukkan ke folder `data/knowledge_base/additional`, meliputi:**

- **Kondisi IHSG dan makroekonomi Indonesia 2026,** termasuk pergerakan IHSG, nilai tukar rupiah, BI Rate, pertumbuhan PDB, cadangan devisa, yield US Treasury, harga minyak, serta kebijakan fiskal.

- **Analisis teknikal IHSG multi-timeframe (monthly, weekly, daily)**, support/resistance, falling wedge, Stochastic RSI, MACD, bullish divergence, Fibonacci, serta metode harga-waktu W.D. Gann.

- **Proyeksi skenario IHSG 2026** meliputi final flush ke area 6.057, immediate bullish reversal, hingga bearish breakdown di bawah 6.000 beserta probabilitas dan target masing-masing.

- **Kebijakan pemerintah dan dampaknya terhadap pasar**, khususnya pembentukan Danantara Sumberdaya Indonesia (DSI), kebijakan ekspor satu pintu, serta aturan Devisa Hasil Ekspor (DHE) SDA dan dampaknya terhadap perbankan Himbara.

- **Rotasi sektor**, terutama perpindahan perhatian dari sektor energi/basic materials menuju sektor keuangan/perbankan sebagai penerima manfaat aturan DHE.

- **Analisis historis rebound IHSG**, dengan membandingkan beberapa periode krisis besar seperti 2008, 2015, 2018, 2020, dan kondisi 2026.

- **Screening saham menggunakan parameter** likuiditas, beta rebound, historical rebound, teknikal, sector leadership, fundamental, kebijakan, foreign flow/institutional interest, dan risk control.

- **10 saham kandidat:** BBRI, BMRI, BBNI, BBCA, MEDC, ADRO, AKRA, PGAS, PTBA, dan AMMN, dengan BBRI sebagai final pick berdasarkan skor tertinggi.

- **Analisis mendalam BBRI**, termasuk valuasi PBV/PE, ROE, dividen, kepemilikan, dampak DHE, serta potensi rebound.

- **Saham cadangan**, yaitu BMRI dan MEDC, beserta alasan pemilihannya.

- **Trading plan BBRI**, mencakup area entry, add position, stop-loss, invalidation, take profit, risk-reward ratio, dan strategi berdasarkan timeframe mingguan, bulanan, hingga akhir 2026.

- **Key dates dan red flags Mei–Desember 2026** termasuk MSCI rebalancing, RDG BI, laporan keuangan bank, implementasi DSI, RAPBN 2027, pelemahan rupiah, gangguan ekspor, dan risiko geopolitik/energi.

- **Deep Research Trading Saham Indonesia — Analisis Strategis Portofolio BBRI dan Energy Winner**, yang menambahkan analisis BBRI, ENRG, BUMI, portfolio construction, scenario analysis, entry/TP/SL, risk-reward, trade management, thesis invalidation, serta horizon 31 Desember 2026.


### 1.3 Batasan

- Tidak mencakup seluruh saham dan fokus pada 10 saham kandidat yang disaring dalam dokumen.
- Tidak mencakup rekomendasi investasi yang dipersonalisasi berdasarkan profil dan kondisi keuangan pengguna.
- Tidak mencakup mekanisme teknis investasi seperti broker, biaya transaksi, dan pajak.
- Tidak menjamin akurasi atau ketercapaian proyeksi harga dan IHSG.
- Tidak mencakup pembaruan data pasar secara real-time setelah periode penelitian dokumen.
- Tidak mencakup seluruh risiko pasar,tetapi hanya membahas risiko utama yang diidentifikasi dalam dokumen.

## 2. Penyusunan Knowledge Base

### 2.1 Load dokumen PDF utama & Cek Metadata

Knowledge base awal hanya mengambil **1 PDF utama yang berada langsung di folder `data/`**. Folder `data/knowledge_base/additional/` tidak ikut di-load pada tahap ini.

In [4]:
# Fungsi: cari_file_utama.

import os
import re
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
DATA_KB_DIR = DATA_DIR / "knowledge_base"
DATA_PRIMARY_DIR = DATA_KB_DIR / "primary"
DATA_ADDITIONAL_DIR = DATA_KB_DIR / "additional"
DATASETS_DIR = DATA_DIR / "datasets"
INDEXES_DIR = DATA_DIR / "indexes"
FAISS_INDEX_DIR = INDEXES_DIR / "stock_research_faiss"
EVALUATION_DIR = DATA_DIR / "evaluation"
GOLD_DIR = EVALUATION_DIR / "gold"
GOLD_CANDIDATES_DIR = GOLD_DIR / "candidates"
GOLD_REVIEW_DIR = GOLD_DIR / "review"
GOLD_VERIFIED_DIR = GOLD_DIR / "verified"
RESULTS_DIR = EVALUATION_DIR / "results"
RESULTS_RETRIEVAL_DIR = RESULTS_DIR / "retrieval"
RESULTS_PHASE2_DIR = RESULTS_DIR / "phase2"
RESULTS_PHASE3_DIR = RESULTS_DIR / "phase3"
RESULTS_FINAL_DIR = RESULTS_DIR / "final"
CHECKPOINTS_DIR = EVALUATION_DIR / "checkpoints"

                                                                                   
for _dir in [
    DATA_PRIMARY_DIR, DATA_ADDITIONAL_DIR, DATASETS_DIR, FAISS_INDEX_DIR,
    GOLD_CANDIDATES_DIR, GOLD_REVIEW_DIR, GOLD_VERIFIED_DIR,
    RESULTS_RETRIEVAL_DIR, RESULTS_PHASE2_DIR, RESULTS_PHASE3_DIR,
    RESULTS_FINAL_DIR, CHECKPOINTS_DIR,
]:
    _dir.mkdir(parents=True, exist_ok=True)


def cari_file_utama(nama_file=None):






    nama_dikenal = [
        "Analisis IHSG 2026 & Saham Unggulan.pdf",
        "Analisis IHSG 2026 & Saham Unggulan(1).pdf",
    ]
    if nama_file:
        nama_dikenal.insert(0, nama_file)
    nama_dikenal = list(dict.fromkeys(nama_dikenal))

    for nama in nama_dikenal:
        p = (DATA_PRIMARY_DIR / nama).resolve()
        if p.is_file():
            return str(p)

    if DATA_PRIMARY_DIR.exists():
        pdf_di_primary = sorted(
            [p.resolve() for p in DATA_PRIMARY_DIR.iterdir()
             if p.is_file() and p.suffix.lower() == ".pdf"],
            key=lambda p: p.name.lower(),
        )
        if len(pdf_di_primary) == 1:
            return str(pdf_di_primary[0])

                                                                                      
    mnt_candidates = [Path("/mnt/data") / nama for nama in nama_dikenal]
    for p in mnt_candidates:
        p = p.expanduser().resolve()
        if p.is_file():
            return str(p)

    pdf_names = [
        p.name for p in DATA_PRIMARY_DIR.iterdir()
        if p.is_file() and p.suffix.lower() == ".pdf"
    ] if DATA_PRIMARY_DIR.exists() else []

    raise FileNotFoundError(
        "PDF utama tidak ditemukan di data/knowledge_base/primary/. "
        f"PDF yang terdeteksi: {pdf_names}. "
        "Pastikan folder primary memiliki tepat satu PDF knowledge utama."
    )


PDF_PATH = cari_file_utama()
NAMA_SUMBER = "riset-ihsg-2026"

document_pdf = PyPDFLoader(PDF_PATH).load()
if not document_pdf:
    raise ValueError("PDF utama terbaca tetapi tidak menghasilkan halaman.")

for d in document_pdf:
    d.metadata["source"] = NAMA_SUMBER

print(f"PDF utama berhasil di-load: {len(document_pdf)} halaman")
print("Path utama:", PDF_PATH)
print("Folder KB tambahan (belum dimuat otomatis):", DATA_ADDITIONAL_DIR)
print("Folder index FAISS:", FAISS_INDEX_DIR)
print("Folder evaluasi:", EVALUATION_DIR)
print(f"\nCuplikan halaman 1:\n{document_pdf[0].page_content[:200]}…")


/var/folders/7j/1_v4mdn15nn4ry80ckw0xy9w0000gn/T/ipykernel_29939/2940989574.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF utama berhasil di-load: 17 halaman
Path utama: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/knowledge_base/primary/Analisis IHSG 2026 & Saham Unggulan.pdf
Folder KB tambahan (belum dimuat otomatis): /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/knowledge_base/additional
Folder index FAISS: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/indexes/stock_research_faiss
Folder evaluasi: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation

Cuplikan halaman 1:
POTENSI
 
MAJOR
 
BOTTOM
 
IHSG
 
2026:
 
ANALISIS
 
KOMPREHENSIF
 
DAN
 
STRATEGI
 
REBOUND
 
SAHAM
 
TERBAIK
 
Executive
 
Summary
 
Pasar
 
modal
 
Indonesia
 
sepanjang
 
paruh
 
pertama
 
tahun
 …


**Interpretasi output:** Output menunjukkan /var/folders/7j/1_v4mdn15nn4ry80ckw0xy9w0000gn/T/ipykernel_75537/2473367095.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/lan.... Jadi proses pada tahap ini berjalan sesuai alurnya.


### 2.2 Pembersihan teks

In [5]:
# Fungsi utama dan helper pada bagian ini.

import re

PENANDA_REFERENSI = [
    "Works cited", "Karya yang dikutip", "Daftar Pustaka", "Bibliography", "Referensi",
]
PANJANG_MINIMUM = 50
URL_RE = re.compile(r"https?://\S+", re.IGNORECASE)


def bersihkan(teks):
    teks = (teks or "").replace("\xa0", " ")
    teks = re.sub(r"\s*\n\s*", " ", teks)
    teks = re.sub(r" {2,}", " ", teks)
    return teks.strip()


def potong_tail_referensi(teks):




    urls = list(URL_RE.finditer(teks))
    if len(urls) < 2:
        return teks, False, False

    posisi_awal = urls[0].start()
    rasio_awal = posisi_awal / max(len(teks), 1)

                                               
    if rasio_awal <= 0.10:
        return "", True, True

                                       
    kandidat = teks[:posisi_awal].strip(" .;:-")
    if len(kandidat) >= PANJANG_MINIMUM:
        return kandidat, True, False

    return "", True, True


def dominan_referensi(teks):
    urls = list(URL_RE.finditer(teks))
    if len(urls) < 2:
        return False

    posisi_awal = urls[0].start()
    rasio_awal = posisi_awal / max(len(teks), 1)
    numbered = len(re.findall(r"(?:^|\s)\d{1,3}\.\s+", teks))
    url_chars = sum(len(m.group(0)) for m in urls)
    density = url_chars / max(len(teks), 1)

    if rasio_awal <= 0.10:
        return True
    if len(urls) >= 7 and (density >= 0.18 or numbered >= 5):
        return True
    return False


def kurasi_halaman(halaman, nama_sumber, metadata_tambahan=None, verbose=True):
    bersih, catatan = [], []

    for d in halaman:
        no_halaman = d.metadata.get("page", 0) + 1
        teks = bersihkan(d.page_content)

                                                                                
                                                                                    
        posisi_marker = min(
            (
                teks.lower().find(marker.lower())
                for marker in PENANDA_REFERENSI
                if teks.lower().find(marker.lower()) >= 0
            ),
            default=-1,
        )

        if posisi_marker == 0:
            if dominan_referensi(teks) or len(URL_RE.findall(teks)) >= 5:
                catatan.append(f"hal.{no_halaman}: dibuang (halaman referensi)")
                continue
        elif posisi_marker > 0:
            tail = teks[posisi_marker:]
            if len(URL_RE.findall(tail)) >= 3 or len(tail) < 180:
                teks = teks[:posisi_marker].strip()
                catatan.append(f"hal.{no_halaman}: dipotong di marker referensi")

        teks, dipotong, full_ref = potong_tail_referensi(teks)
        if full_ref:
            catatan.append(f"hal.{no_halaman}: dibuang (daftar URL/reference)")
            continue
        if dipotong:
            catatan.append(f"hal.{no_halaman}: dipotong sebelum tail URL/reference")

        if len(teks) < PANJANG_MINIMUM:
            catatan.append(f"hal.{no_halaman}: dibuang (teks terlalu pendek)")
            continue
        if dominan_referensi(teks):
            catatan.append(f"hal.{no_halaman}: dibuang (dominan URL/reference)")
            continue

        d.page_content = teks
        d.metadata["source"] = nama_sumber
        if metadata_tambahan:
            d.metadata.update(metadata_tambahan)
        bersih.append(d)

    if verbose:
        karakter = sum(len(d.page_content) for d in bersih)
        print(
            f"{nama_sumber}: {len(halaman)} halaman → "
            f"{len(bersih)} halaman konten · {karakter:,} karakter"
        )
        for c in catatan:
            print(f"   • {c}")

    if not bersih:
        raise ValueError(
            f"KURASI GAGAL: '{nama_sumber}' menghasilkan 0 halaman konten. "
            "Periksa path PDF atau aturan filter referensi."
        )

    return bersih


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


### 2.3 Struktur knowledge base

In [6]:
# Fungsi: summary_knowledge_base, semua_dokumen.

dokumen_bersih = kurasi_halaman(document_pdf, NAMA_SUMBER)
if not dokumen_bersih:
    raise RuntimeError("PDF utama tidak menghasilkan dokumen bersih.")
print(f"Dokumen utama siap dimasukkan ke KNOWLEDGE_BASE: {len(dokumen_bersih)} halaman konten")

                                                                        
KNOWLEDGE_BASE = {
    NAMA_SUMBER: dokumen_bersih,
}


def summary_knowledge_base():
    total_bagian = sum(len(daftar) for daftar in KNOWLEDGE_BASE.values())
    total_karakter = sum(
        len(d.page_content)
        for daftar in KNOWLEDGE_BASE.values()
        for d in daftar
    )
    print(
        f"Knowledge base: {len(KNOWLEDGE_BASE)} sumber · "
        f"{total_bagian} bagian · {total_karakter:,} karakter:"
    )
    for nama, daftar in KNOWLEDGE_BASE.items():
        karakter = sum(len(d.page_content) for d in daftar)
        print(
            f"   • {nama:<28} {len(daftar):>3} bagian · "
            f"{karakter:>7,} karakter"
        )


def semua_dokumen():

    hasil = []
    for daftar in KNOWLEDGE_BASE.values():
        hasil.extend(daftar)
    return hasil

summary_knowledge_base()


riset-ihsg-2026: 17 halaman → 15 halaman konten · 32,803 karakter
   • hal.15: dipotong di marker referensi
   • hal.16: dibuang (daftar URL/reference)
   • hal.17: dibuang (daftar URL/reference)
Dokumen utama siap dimasukkan ke KNOWLEDGE_BASE: 15 halaman konten
Knowledge base: 1 sumber · 15 bagian · 32,803 karakter:
   • riset-ihsg-2026               15 bagian ·  32,803 karakter


**Interpretasi output:** Output menunjukkan riset-ihsg-2026: 17 halaman → 15 halaman konten · 32,803 karakter • hal.15: dipotong di marker referensi. Jadi proses pada tahap ini berjalan sesuai alurnya.


### 2.4 Mekanisme pembaruan knowledge base

In [7]:
# Fungsi utama dan helper pada bagian ini.

def tambah_dokumen(nama_sumber, isi_teks, metadata_tambahan=None):
    if nama_sumber in KNOWLEDGE_BASE:
        raise ValueError(f"'{nama_sumber}' sudah ada. Gunakan ganti_dokumen().")
    meta = {"source": nama_sumber}
    if metadata_tambahan:
        meta.update(metadata_tambahan)
    teks = isi_teks.strip()
    if len(teks) < PANJANG_MINIMUM:
        raise ValueError(f"Isi dokumen '{nama_sumber}' terlalu pendek untuk di-index.")
    KNOWLEDGE_BASE[nama_sumber] = [Document(page_content=teks, metadata=meta)]
    print(f"Sumber '{nama_sumber}' berhasil ditambahkan.")


                                           
def tambah_dokumen_pdf(nama_sumber, path_pdf, metadata_tambahan=None):
    if nama_sumber in KNOWLEDGE_BASE:
        raise ValueError(f"'{nama_sumber}' sudah ada. Gunakan ganti_dokumen().")
    if not os.path.isfile(path_pdf):
        raise FileNotFoundError(f"File tidak ditemukan: {path_pdf}")

    halaman = PyPDFLoader(path_pdf).load()
    if not halaman:
        raise ValueError(
            f"'{path_pdf}' tidak menghasilkan halaman — kemungkinan PDF kosong/scan."
        )

    hasil = kurasi_halaman(halaman, nama_sumber, metadata_tambahan)
    if not hasil:
        raise ValueError(f"'{nama_sumber}' gagal dikurasi: 0 bagian.")

    KNOWLEDGE_BASE[nama_sumber] = hasil
    print(f"Sumber '{nama_sumber}' berhasil ditambahkan: {len(hasil)} bagian.")


                                 
def ganti_dokumen(nama_sumber, isi_teks_baru):
    if nama_sumber not in KNOWLEDGE_BASE:
        raise ValueError(
            f"'{nama_sumber}' belum ada. Gunakan tambah_dokumen() terlebih dahulu."
        )
    if not KNOWLEDGE_BASE[nama_sumber]:
        raise ValueError(f"'{nama_sumber}' kosong sehingga tidak bisa diambil metadata lamanya.")

    metadata_lama = KNOWLEDGE_BASE[nama_sumber][0].metadata.copy()
    jumlah_lama = len(KNOWLEDGE_BASE[nama_sumber])
    teks = isi_teks_baru.strip()
    if len(teks) < PANJANG_MINIMUM:
        raise ValueError("Isi dokumen baru terlalu pendek.")
    KNOWLEDGE_BASE[nama_sumber] = [
        Document(page_content=teks, metadata=metadata_lama)
    ]
    print(f"Sumber '{nama_sumber}' diganti ({jumlah_lama} bagian → 1 bagian).")


                  
def hapus_dokumen(nama_sumber):
    if nama_sumber not in KNOWLEDGE_BASE:
        raise ValueError(f"'{nama_sumber}' tidak ditemukan di knowledge base.")
    jumlah = len(KNOWLEDGE_BASE.pop(nama_sumber))
    print(f"Sumber '{nama_sumber}' dihapus ({jumlah} bagian).")


TAMBAHAN_SOURCE_MAP = {
    "Analisis Investasi BIPI Geopolitik & Fundamental.pdf": "riset-bipi-2026",
    "Analisis Mendalam Saham Barito Group.pdf": "riset-barito-2026",
    "Blueprint Investasi Presisi Chaos Scenario.pdf": "riset-blueprint-2026",
    "Indonesian Equity Trading Research.pdf": "riset-equity-2026",
}

                                                              
                                                                      
                                                                       
                                                              

def daftar_pdf_tambahan():
    if not DATA_ADDITIONAL_DIR.exists():
        return []
    return sorted(
        [
            p for p in DATA_ADDITIONAL_DIR.iterdir()
            if p.is_file() and p.suffix.lower() == ".pdf"
        ],
        key=lambda p: p.name.lower(),
    )


def nama_sumber_dari_file(path_pdf):
    nama = Path(path_pdf).stem.lower()
    nama = re.sub(r"[^a-z0-9]+", "-", nama).strip("-")
    return nama or "dokumen-tambahan"


def tambah_pdf_tambahan(
    nama_file,
    nama_sumber=None,
    metadata_tambahan=None,
    rebuild=True,
):

    path_pdf = (DATA_ADDITIONAL_DIR / nama_file).resolve()
    if not path_pdf.is_file():
        raise FileNotFoundError(
            f"PDF tambahan tidak ditemukan: {path_pdf}\n"
            "PDF harus berada di folder data/knowledge_base/additional/."
        )

    source = nama_sumber or TAMBAHAN_SOURCE_MAP.get(
        path_pdf.name, nama_sumber_dari_file(path_pdf)
    )
    meta = dict(metadata_tambahan or {})
    meta.setdefault("tipe", "riset-tambahan")
    meta.setdefault("nama_file", path_pdf.name)

    tambah_dokumen_pdf(
        source,
        str(path_pdf),
        metadata_tambahan=meta,
    )

    if rebuild:
        bangun_ulang_index()
                                                                                   
    refresh_dynamic_ticker_registry()

    return source


def tambah_semua_pdf_tambahan(rebuild=True):

    files = daftar_pdf_tambahan()
    if not files:
        print("Tidak ada PDF di data/knowledge_base/additional/.")
        return []

    added, skipped = [], []
    for path_pdf in files:
        source = TAMBAHAN_SOURCE_MAP.get(
            path_pdf.name, nama_sumber_dari_file(path_pdf)
        )
        if source in KNOWLEDGE_BASE:
            skipped.append(source)
            continue

        tambah_dokumen_pdf(
            source,
            str(path_pdf),
            metadata_tambahan={
                "tipe": "riset-tambahan",
                "nama_file": path_pdf.name,
            },
        )
        added.append(source)

    if rebuild and added:
        bangun_ulang_index()
    if added:
                                                                                 
        refresh_dynamic_ticker_registry()

    print("PDF tambahan ditambahkan:", added or "tidak ada")
    if skipped:
        print("PDF yang sudah ada, dilewati:", skipped)
    return added


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


Knowledge base berhasil disimpan dalam dictionary `KNOWLEDGE_BASE` yang memetakan nama sumber ke daftar Document. Dictionary ini adalah satu-satunya sumber kebenaran untuk chatbot. Lalu untuk *vector index* dan *vector database* akan dibuat berdasarkan isi dari dictionary *knowledge base*.

Metode ini saya gunakan karena *vector database* FAISS tidak menyediakan operasi *edit in-place* untuk mengubah satu *chunk*, sehingga harus membangun ulang *index* dari awal. Oleh karena itu, metode saya ini menyimpan dokumen asli secara terpisah, sehingga *index*-nya dapat selalu direkonstruksi kapan pun tanpa risiko adanya *chunk* usang yang tertinggal di *index*.

Kekurangan metode ini adalah tidak *scalable*. Ketika diberikan jutaan dokumen, proses *embedding* ulang akan memakan waktu dan komputasi yang sangat besar.

## 3. Create Sistem dan Implementasi RAG

### 3.1 Text Splitter (Chunking)

In [8]:
# Import dan setup yang dibutuhkan.

from langchain_text_splitters import RecursiveCharacterTextSplitter

                                                                      
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 80
)
                                    
dokumen_gabungan = semua_dokumen()

          
chunks = splitter.split_documents(dokumen_gabungan)
print(f"{len(dokumen_gabungan)} Dokumen menghasilkan: {len(chunks)} chunk")
print("Contoh 3 chunk pertama:\n")
for i, c in enumerate(chunks[:3], 1):
    print(f"->chunk {i} · {c.metadata} · {len(c.page_content)} karakter")
    print(f"{c.page_content[:200]}\n")


15 Dokumen menghasilkan: 84 chunk
Contoh 3 chunk pertama:

->chunk 1 · {'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Analisis IHSG 2026 & Saham Unggulan', 'source': 'riset-ihsg-2026', 'total_pages': 17, 'page': 0, 'page_label': '1'} · 499 karakter
POTENSI MAJOR BOTTOM IHSG 2026: ANALISIS KOMPREHENSIF DAN STRATEGI REBOUND SAHAM TERBAIK Executive Summary Pasar modal Indonesia sepanjang paruh pertama tahun 2026 diuji oleh tekanan koreksi yang sign

->chunk 2 · {'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Analisis IHSG 2026 & Saham Unggulan', 'source': 'riset-ihsg-2026', 'total_pages': 17, 'page': 0, 'page_label': '1'} · 498 karakter
pada penutupan perdagangan 20 Mei 2026. 1 Kejatuhan tajam ini menobatkan IHSG sebagai indeks dengan kinerja terburuk di kawasan Asia sepanjang tahun berjalan. 1 Sentimen negatif digerakkan oleh kombin

->chunk 3 · {'producer': 'Skia/PDF m153 Google Do

**Interpretasi output:** Output menunjukkan 15 Dokumen menghasilkan: 84 chunk Contoh 3 chunk pertama:. Jadi proses pada tahap ini berjalan sesuai alurnya.


Sudah berhasil melakukan text splitter dari 15 Dokumen yang menghasilkan 84 chunk. Hasilnya juga sangat bagus, terlihat dari potongan 3 chunk pertama yg berhasil memotong setiap kata dengan tepat. Tetapi angka ini sebelum dilakukan penambahan document di akhir utk memperluas Knowledge base.

### 3.2 Teks to Vector (Embedding)

In [9]:
# Fungsi: simpan_metadata_index.

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import json
from datetime import datetime, timezone


embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": DEVICE},                                 
    encode_kwargs = {"normalize_embeddings": True}
)

                                                      
store_vektor = FAISS.from_documents(documents=chunks, embedding=embeddings)

                      
store_vektor.save_local(str(FAISS_INDEX_DIR))
print(f"Index FAISS: {store_vektor.index.ntotal} vektor × {store_vektor.index.d} dimensi")

def simpan_metadata_index():
    metadata = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "embedding_model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "normalize_embeddings": True,
        "chunk_size": int(CHUNK_SIZE) if "CHUNK_SIZE" in globals() else None,
        "chunk_overlap": int(CHUNK_OVERLAP) if "CHUNK_OVERLAP" in globals() else None,
        "active_sources": sorted(KNOWLEDGE_BASE.keys()) if "KNOWLEDGE_BASE" in globals() else [],
        "chunk_count": len(chunks) if "chunks" in globals() else None,
        "vector_count": int(store_vektor.index.ntotal),
        "vector_dimension": int(store_vektor.index.d),
        "retrieval_final": "Similarity k=8",
    }
    path = FAISS_INDEX_DIR / "metadata.json"
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)
    return path

simpan_metadata_index()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index FAISS: 84 vektor × 384 dimensi


PosixPath('/Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/indexes/stock_research_faiss/metadata.json')

**Interpretasi output:** Output menunjukkan Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads. Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]. Jadi proses pada tahap ini berjalan sesuai alurnya.


**Pemilihan embedding model** 

Saya memakai `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` dengan tiga pertimbangan yang datang dari karakter dokumen saya sendiri:

1. **Multilingual(karena dokumen saya campuran).** Isi riset saya berbahasa Indonesia, tetapi istilah kuncinya berbahasa Inggris seperti PBV, forward PE, dividend yield, stop-loss, major bottom. Sehingga model yang hanya dilatih untuk bahasa Inggris akan kesulitan pada kalimat penjelasnya, sedangkan model yang hanya paham Bahasa Indonesia akan kesulitan pada istilah teknisnya. Model multilingual menempatkan keduanya di ruang vektor yang sama.

2. **Ringan dan bisa berjalan lokal.** Dimensinya hanya 384, sehingga cukup dijalankan di MacBook saya lewat akselerasi MPS tanpa perlu API embedding berbayar. Untuk knowledge base sekecil ini, model yang lebih besar hanya menambah cost tanpa manfaat yang sepadan.

3. **`normalize_embeddings=True`.** Vektor dinormalisasi ke panjang 1 supaya skor kemiripan berada pada skala yang konsisten dan bisa dibandingkan antar pertanyaan. Ini penting karena seluruh pengujian saya di bagian 3.3 memakai ambang dan selisih skor sebagai dasar keputusan.


**Pemilihan vector database**

Saya memilih menyimpan hasil vektor ke dalam vektor Database FAISS adalah karena:
1. chunk yang saya punya sangat kecil yaitu 84 chunk × 384 dimensi (mungkin sekitar ~147 KB), sehingga lebih ringan
2. Yang menggunakan hanya saya dilocal sehingga tidak butuh server ataupun jaringan
3. Berjalan di local
4. Gratis

Angka 84 chunk ini adalah kondisi knowledge base awal (1 dokumen). Setelah diperluas menjadi 4 dokumen pada bagian 4.2, jumlah chunk menjadi ±290 dan masih dalam skala yang sangat ringan untuk FAISS.

### 3.3 Retriever

In [10]:
# Fungsi: label_sumber, buat_retriever, periksa_retriever.

def label_sumber(d):
    sumber = d.metadata.get("source", "dokumen")
    halaman = d.metadata.get("page")
    return f"{sumber} hal.{halaman + 1}" if halaman is not None else sumber


                                  
def buat_retriever(k=3, score_threshold=0.3, fetch_k=None):
    if fetch_k is None:
        fetch_k = k*4                                                                                       

    return{
                                 
        'Similarity': store_vektor.as_retriever(
            search_kwargs={"k": k}
        ),
                                  
        'retriever_mmr': store_vektor.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k,"fetch_k": fetch_k},
        ),
                                              
        'threshold': store_vektor.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": score_threshold},
        )
    }

                                                                                                                    
def periksa_retriever(pertanyaan, k=3, score_threshold=0.3, fetch_k=None, panjang_cuplikan=300):
    
    daftar_retriever = buat_retriever(k=k, score_threshold=score_threshold, fetch_k=fetch_k)

                                                                                      
                                                                                              
    berskor = store_vektor.similarity_search_with_score(pertanyaan, k=k * 3)
    jarak_by_id = {d.id: s for d, s in berskor}

    print("=" * 78)
    print(f"❓ {pertanyaan}")
    print(f"   parameter: k={k} · score_threshold={score_threshold} · fetch_k={fetch_k or k * 4}")
    print("=" * 78)

    for nama, r in daftar_retriever.items():
        dokumen = r.invoke(pertanyaan)
        print(f"\n▼ {nama.upper()}  —  {len(dokumen)} chunk")

        if not dokumen:
            print("   (kosong — semua kandidat berada di bawah score_threshold)")
            continue

        for i, d in enumerate(dokumen, 1):
            jarak = jarak_by_id.get(d.id)
            jarak_teks = f"{jarak:.3f}" if jarak is not None else "n/a"
            print(f"   {i}. [{label_sumber(d)}]  jarak_FAISS={jarak_teks}  ({len(d.page_content)} karakter)")
            print(f"      {d.page_content[:panjang_cuplikan]}…")
    print()


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


In [11]:
# Proses utama pada bagian ini.

PERTANYAAN_UJI = [
    "Berapa harga Take Profit BBRI?",                  
    "Berapa PBV dan dividend yield BBRI?",             
    "Bagaimana prospek GOTO?",                                   
    "Resep rendang yang enak apa?",                              
]

for p in PERTANYAAN_UJI:
    periksa_retriever(p, k=5)


No relevant docs were retrieved using the relevance score threshold 0.3


❓ Berapa harga Take Profit BBRI?
   parameter: k=5 · score_threshold=0.3 · fetch_k=20

▼ SIMILARITY  —  5 chunk
   1. [riset-ihsg-2026 hal.13]  jarak_FAISS=0.590  (496 karakter)
      jual mekanis selesai. 1 Harga BBRI diproyeksikan akan keluar dari fase bearish harian dan memulai pembentukan tren naik baru menuju target pertama di level Rp3.400, didukung oleh rilis laporan bulanan perbankan yang menunjukkan perbaikan rasio dana murah (CASA). 7 ● End of 2026 Perspective (Akhir Ta…
   2. [riset-ihsg-2026 hal.11]  jarak_FAISS=0.787  (500 karakter)
      PT Bank Rakyat Indonesia (Persero) Tbk (BBRI) merupakan pilar utama sektor perbankan nasional yang berfokus pada pembiayaan segmen mikro dan Usaha Mikro, Kecil, dan Menengah (UMKM). 7 Kejatuhan harga saham BBRI sebesar 32% dari level puncaknya di Rp4.450 ke level Rp3.040 pada Mei 2026 membuka peluan…
   3. [riset-ihsg-2026 hal.11]  jarak_FAISS=0.862  (498 karakter)
      BBRI diperdagangkan pada trailing PE sebesar 7,8x dan forward PE tah

/opt/anaconda3/envs/env_assig_rag/lib/python3.10/site-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='39f32f2b-ef54-4274-b6a7-1a96a82ab96d', metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Analisis IHSG 2026 & Saham Unggulan', 'source': 'riset-ihsg-2026', 'total_pages': 17, 'page': 5, 'page_label': '6'}, page_content='Waktu Bottom Akhir Mei - Pertengahan Juni 2026. 1 Akhir Mei 2026. Kuartal III - Kuartal IV 2026. Proyeksi Akhir Tahun 2026 8.873 - 9.000 8.873 - 9.000 5.800 - 6.100 Pengujian Realisme Target Bullish Akhir 2026 (8.873 - 9.000) Target akhir tahun di level 8.873 - 9.000 dinilai sangat realistis dan berbasis fundamental kuat. 7 Meskipun saat ini IHSG mengalami tekanan likuiditas jangka pendek, rancangan kebijakan fiskal pemerintah terbaru (KEM PPKF RAPBN 2027) yang diajukan oleh Presiden Prabowo'), np.float32(-0.28794706)), (Document(id='b


▼ THRESHOLD  —  0 chunk
   (kosong — semua kandidat berada di bawah score_threshold)



**Interpretasi Hasil Retrieve Ketika K=5, chunk_size = 500, chunk_overlap = 80**
*note: No relevant docs were retrieved using the relevance score threshold 0.3, sehingga nilai threshold menjadi default saja*


Saya melakukan eksperimen dengan menaikkan nilai k dari 3 menjadi 5, dan ini berhasil menyelesaikan masalah pertanyaan majemuk. Pada pertanyaan "Berapa PBV dan dividend yield BBRI?", retriever `similarity` kini mengambil kedua chunk yang dibutuhkan sekaligus yaitu chunk dividen (Rp418 per saham, yield 13,75%) di peringkat 3 dan chunk PBV (BVPS Rp2.246, PBV 1,35x) di peringkat 4.

Tetapi untuk pertanyaan "Berapa harga Take Profit BBRI?", seluruh varian chunk yang hanya berhasil mengidentifikasi sampai Take Profit 1 saja, dan tidak sampai ke TP 3. Lalu kalimat yg dihasilkan juga terpotong, maka selanjutnya saya akan mengulanginya lagi tetapi dengan menaikkan Chunk_size dan chunk_overlap

Lalu, nilai `score_threshold = 0,3`  terbukti tepat untuk menolak pertanyaan di luar cakupan (GOTO dan resep rendang tetap menghasilkan 0 chunk
pada k = 5), tetapi sekaligus menyaring keluar chunk tabel Take Profit yang berskor 0,248 padahaal masih tetap relevan dengan pertanyaannya.

**Selanjutnya** saya akan mencari kombinasi Chunk_Size dan Chunk_overlap terbaik melalui grid search dengan hit-rate@k.

saya akan membuat 2 kolom sebagai metrik pengukurannya:
1. `hit_rate:`seberapa sering chunk berisi jawaban berhasil terambil. Makin tinggi makin baik. Ini metrik utama.
2. `celah:` jarak antara jarak in-scope terburuk dan jarak out-of-scope terbaik. Makin lebar, makin mudah menentukan score_threshold yang aman. Celah sempit berarti sistemnya rapuh sedikit saja pertanyaan tak terduga, batas antara "tahu" dan "tidak tahu" jadi kabur.

In [12]:
# Fungsi: evaluasi.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import pandas as pd

                                                                                
UJI = [
    {"pertanyaan": "Berapa harga Take Profit BBRI?",
     "kata_kunci": ["Rp3.400", "Rp3.800", "Rp4.250"], "in_scope": True},
    {"pertanyaan": "Berapa PBV dan dividend yield BBRI?",
     "kata_kunci": ["1,35x", "13,75%"], "in_scope": True},
    {"pertanyaan": "Berapa BVPS BBRI?",
     "kata_kunci": ["Rp2.246"], "in_scope": True},
    {"pertanyaan": "Di level berapa zona support konvergen IHSG?",
     "kata_kunci": ["6.050", "6.120"], "in_scope": True},
    {"pertanyaan": "Bagaimana prospek GOTO?",
     "kata_kunci": [], "in_scope": False},
    {"pertanyaan": "Resep rendang yang enak apa?",
     "kata_kunci": [], "in_scope": False},
]

KONFIGURASI = [
    (300, 45), (500, 80), (700, 105), (1000, 150), (1000, 250), (1500, 225),
]
K = 4


def evaluasi(chunk_size, chunk_overlap, k=K):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                              chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(semua_dokumen())
    store = FAISS.from_documents(chunks, embeddings)

    hit, total_in_scope = 0, 0
    skor_in, skor_out = [], []

    for u in UJI:
        hasil = store.similarity_search_with_score(u["pertanyaan"], k=k)
        gabungan = " ".join(d.page_content for d, _ in hasil)
        jarak_terdekat = hasil[0][1] if hasil else float("inf")

        if u["in_scope"]:
            total_in_scope += 1
            skor_in.append(jarak_terdekat)
                                                                         
            if all(kk in gabungan for kk in u["kata_kunci"]):
                hit += 1
        else:
            skor_out.append(jarak_terdekat)

    return {
        "chunk_size": chunk_size,
        "overlap": chunk_overlap,
        "jml_chunk": len(chunks),
        f"hit@{k}": f"{hit}/{total_in_scope}",
        "hit_rate": round(hit / total_in_scope, 2),
        "jarak_in_max": round(max(skor_in), 3),
        "jarak_out_min": round(min(skor_out), 3),
        "celah_jarak": round(min(skor_out) - max(skor_in), 3),
    }


hasil = [evaluasi(cs, ov) for cs, ov in KONFIGURASI]
pd.DataFrame(hasil)


,chunk_size,overlap,jml_chunk,hit@4,hit_rate,jarak_in_max,jarak_out_min,celah_jarak
0,300,45,135,1/4,0.25,0.800,1.205,0.404
1,500,80,84,3/4,0.75,0.749,1.165,0.416
2,700,105,61,3/4,0.75,0.873,1.211,0.338
3,1000,150,44,3/4,0.75,0.876,1.122,0.246
4,1000,250,47,3/4,0.75,0.876,1.211,0.334
5,1500,225,30,3/4,0.75,0.755,1.154,0.399


**Interpretasi Mencari nilai Chunk size dan overlap terbaik:**

Setelah dilakukan berbagai kombinasi, konfigurasi saya (500/80) tetap menjadi pilihan yang paling masuk akal dari eksperimen kecil ini.

`Temuan:`

* **300/45 kurang baik:** hit_rate 0,25. Artinya dari 4 pertanyaan in-scope, hanya 1 yang seluruh kata kuncinya berhasil ditemukan pada chunk terambil.
* **500/80 konsisten baik:** hit_rate 0,75 dan celah jarak 0,294 pada pengujian yang tersimpan.
* **Chunk lebih besar tidak otomatis lebih baik:** beberapa konfigurasi 700–1500 memang tetap mencapai 0,75, tetapi celah jaraknya tidak selalu lebih baik daripada 500/80.
* **Overlap yang lebih besar dapat membantu pada kondisi tertentu**, tetapi belum memberikan hasil yang lebih baik daripada konfigurasi 500/80 pada eksperimen kecil ini.

Jadi, untuk sementara saya **freeze chunk_size = 500 dan chunk_overlap = 80** sebagai baseline. Ini belum berarti konfigurasi tersebut pasti optimal untuk seluruh 125 pertanyaan karena pengujian ini masih menggunakan 4 pertanyaan uji.

In [13]:
# Tampilkan hasil proses.

for k in [4, 5, 6, 8,9,10,12]:
    print(k, evaluasi(500, 80, k=k))


4 {'chunk_size': 500, 'overlap': 80, 'jml_chunk': 84, 'hit@4': '3/4', 'hit_rate': 0.75, 'jarak_in_max': np.float32(0.749), 'jarak_out_min': np.float32(1.165), 'celah_jarak': np.float32(0.416)}
5 {'chunk_size': 500, 'overlap': 80, 'jml_chunk': 84, 'hit@5': '3/4', 'hit_rate': 0.75, 'jarak_in_max': np.float32(0.749), 'jarak_out_min': np.float32(1.165), 'celah_jarak': np.float32(0.416)}
6 {'chunk_size': 500, 'overlap': 80, 'jml_chunk': 84, 'hit@6': '3/4', 'hit_rate': 0.75, 'jarak_in_max': np.float32(0.749), 'jarak_out_min': np.float32(1.165), 'celah_jarak': np.float32(0.416)}
8 {'chunk_size': 500, 'overlap': 80, 'jml_chunk': 84, 'hit@8': '3/4', 'hit_rate': 0.75, 'jarak_in_max': np.float32(0.749), 'jarak_out_min': np.float32(1.165), 'celah_jarak': np.float32(0.416)}
9 {'chunk_size': 500, 'overlap': 80, 'jml_chunk': 84, 'hit@9': '3/4', 'hit_rate': 0.75, 'jarak_in_max': np.float32(0.749), 'jarak_out_min': np.float32(1.165), 'celah_jarak': np.float32(0.416)}
10 {'chunk_size': 500, 'overlap': 8

**Interpretasi output:** Output menunjukkan 4 {'chunk_size': 500, 'overlap': 80, 'jml_chunk': 84, 'hit@4': '3/4', 'hit_rate': 0.75, 'jarak_in_max': np.float32(0.749), 'jarak_out_min': np.float32(1.165), 'celah_jarak': np.float32(0.416)} 5 {'chunk_size': 500, 'o.... Jadi proses pada tahap ini berjalan sesuai alurnya.


In [14]:
# Fungsi: diagnosa.

def diagnosa(chunk_size=500, chunk_overlap=80, k=9):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    store = FAISS.from_documents(splitter.split_documents(semua_dokumen()), embeddings)

    for u in UJI:
        if not u["in_scope"]:
            continue
        hasil = store.similarity_search_with_relevance_scores(u["pertanyaan"], k=k)
        gabungan = " ".join(d.page_content for d, _ in hasil)
        ada = [kk for kk in u["kata_kunci"] if kk in gabungan]
        hilang = [kk for kk in u["kata_kunci"] if kk not in gabungan]
        status = "✅" if not hilang else "❌"
        print(f"{status} {u['pertanyaan']}")
        print(f"     ditemukan: {ada}")
        print(f"     HILANG   : {hilang}\n")

diagnosa()


❌ Berapa harga Take Profit BBRI?
     ditemukan: ['Rp3.400']
     HILANG   : ['Rp3.800', 'Rp4.250']

✅ Berapa PBV dan dividend yield BBRI?
     ditemukan: ['1,35x', '13,75%']
     HILANG   : []

✅ Berapa BVPS BBRI?
     ditemukan: ['Rp2.246']
     HILANG   : []

✅ Di level berapa zona support konvergen IHSG?
     ditemukan: ['6.050', '6.120']
     HILANG   : []



**Interpretasi output:** Output menunjukkan ❌ Berapa harga Take Profit BBRI? ditemukan: ['Rp3.400']. Jadi proses pada tahap ini berjalan sesuai alurnya.


Setelah saya coba mengubah nilai K dari 1 sampai 12, pada eksperimen similarity murni hasil **4/4 baru tercapai pada k=12**. Pada k=4 sampai k=10 hasil yang tersimpan masih 3/4.

Hal ini menunjukkan bahwa menaikkan k memang dapat membantu mengambil informasi yang tersebar, tetapi semakin besar k maka semakin banyak chunk yang masuk ke context.

**Maka selanjutnya saya membandingkannya dengan MMR** untuk melihat apakah keragaman retrieval dapat mencapai cakupan yang sama dengan jumlah chunk yang lebih kecil.

In [15]:
# Fungsi: evaluasi_mmr.

def evaluasi_mmr(k=5, fetch_k=None, chunk_size=500, chunk_overlap=80):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    store = FAISS.from_documents(splitter.split_documents(semua_dokumen()), embeddings)
    r = store.as_retriever(search_type="mmr",
                           search_kwargs={"k": k, "fetch_k": fetch_k or k * 4})
    hit = total = 0
    for u in UJI:
        if not u["in_scope"]:
            continue
        total += 1
        gabungan = " ".join(d.page_content for d in r.invoke(u["pertanyaan"]))
        if all(kk in gabungan for kk in u["kata_kunci"]):
            hit += 1
    return {"k": k, "hit": f"{hit}/{total}", "hit_rate": round(hit / total, 2)}

for k in [4, 5, 6, 8, 9]:
    print(evaluasi_mmr(k))


{'k': 4, 'hit': '2/4', 'hit_rate': 0.5}
{'k': 5, 'hit': '2/4', 'hit_rate': 0.5}
{'k': 6, 'hit': '4/4', 'hit_rate': 1.0}
{'k': 8, 'hit': '4/4', 'hit_rate': 1.0}
{'k': 9, 'hit': '4/4', 'hit_rate': 1.0}


**Interpretasi output:** Output menunjukkan {'k': 4, 'hit': '2/4', 'hit_rate': 0.5} {'k': 5, 'hit': '2/4', 'hit_rate': 0.5}. Jadi proses pada tahap ini berjalan sesuai alurnya.


**Penentuan Nilai k: Similarity vs MMR**

Setelah konfigurasi chunking ditetapkan (`chunk_size=500`, `overlap=80`), saya menguji nilai `k` pada empat pertanyaan in-scope.

Pada eksperimen yang tersimpan:

| Strategi | k=4 | k=5 | k=6 | k=8 | k=9 | k=10 | k=12 | k terkecil untuk 4/4 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Similarity | 0,75 | 0,75 | 0,75 | 0,75 | 0,75 | 0,75 | **1,00** | **12** |
| MMR | 0,50 | 0,50 | **1,00** | 1,00 | 1,00 | - | - | **6** |

MMR mencapai cakupan penuh pada **k=6**, sedangkan similarity murni baru mencapai 4/4 pada **k=12** dalam eksperimen kecil ini.

Karena MMR membutuhkan lebih sedikit chunk untuk mencapai cakupan penuh pada pengujian tersebut, saya memilih **MMR dengan k=6 dan fetch_k=24** sebagai konfigurasi baseline.

**Catatan penting:** angka di atas adalah hasil eksperimen kecil, bukan Recall@K benchmark 125 pertanyaan.

### 3.4 LLM

In [16]:
# Tampilkan hasil proses.

print("API key tersedia:", bool(deepseek_api_key))
print("Tipe:", type(deepseek_api_key))


API key tersedia: True
Tipe: <class 'str'>


**Interpretasi output:** Output menunjukkan API key tersedia: True Tipe: <class 'str'>. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [17]:
# Import dan setup yang dibutuhkan.

from langchain_openai import ChatOpenAI

if not deepseek_api_key:
    llm = None
    response_uji_llm = None
    print("⚠️ DEEPSEEK_API_KEY belum tersedia. LLM dilewati; siapkan .env sebelum menjalankan demo/benchmark.")
else:
    llm = ChatOpenAI(
        model="deepseek-v4-flash",
        api_key=deepseek_api_key,
        base_url="https://api.deepseek.com",
        temperature=0,
        max_tokens=2048,
        extra_body={"thinking": {"type": "disabled"}},
    )
    print("Object DeepSeek-V4-Flash berhasil dibuat.")

JALANKAN_UJI_LLM = False
if JALANKAN_UJI_LLM:
    if llm is None:
        raise RuntimeError("LLM belum tersedia. Isi DEEPSEEK_API_KEY terlebih dahulu.")
    response_uji_llm = llm.invoke("Jawab satu kalimat: apa itu RAG?")
    print("Koneksi DeepSeek berhasil.")
    print("Jawaban uji:", response_uji_llm.content)


Object DeepSeek-V4-Flash berhasil dibuat.


**Interpretasi output:** Output menunjukkan Object DeepSeek-V4-Flash berhasil dibuat.. Jadi proses pada tahap ini berjalan sesuai alurnya.


**Saya menggunakan DeepSeek-V4-Flash** sebagai LLM untuk sistem RAG ini. DeepSeek menyediakan API yang kompatibel dengan format OpenAI, sehingga saya dapat mengintegrasikannya ke LangChain melalui `ChatOpenAI` tanpa mengubah pipeline retrieval yang sudah dibuat.

Saya memilih mode non-thinking (`thinking=disabled`) karena tugas utama model di project ini adalah membaca konteks hasil retrieval lalu menghasilkan jawaban yang grounded dan bersitasi. Dengan begitu, proses generation lebih sederhana dan tidak perlu menggunakan reasoning tambahan untuk setiap pertanyaan.

Parameter `temperature=0` dipilih agar keluaran relatif konsisten/deterministik, sedangkan `max_tokens=2048` digunakan untuk memberi batas panjang jawaban. Pembatasan sumber jawaban tetap dilakukan oleh grounding prompt di bagian 3.5.

DeepSeek API menggunakan base URL `https://api.deepseek.com` dan model `deepseek-v4-flash`, sesuai dokumentasi API DeepSeek saat ini.

### 3.5 Grounding prompt

In [18]:
# Import dan setup yang dibutuhkan.

from langchain_core.prompts import ChatPromptTemplate

PROMPT_RAG = ChatPromptTemplate.from_template(
    """Kamu adalah asisten yang menjawab pertanyaan berdasarkan dokumen riset yang tersedia di knowledge base.

Aturan:
1. Jawab HANYA berdasarkan konteks di bawah. Dilarang menggunakan pengetahuan di luar konteks.
2. Jika jawabannya tidak ada di konteks, maka katakan:
   "Maaf, informasi itu tidak ada di dokumen saya."
3. Sebutkan sumber untuk setiap fakta yang kamu tulis, dalam format [sumber hal.N].
4. Sampaikan angka persis seperti tertulis di konteks, jangan dibulatkan atau dihitung ulang.
5. Jawaban ini merupakan ringkasan isi dokumen riset, bukan rekomendasi investasi.

Konteks:
{context}

Pertanyaan: {question}"""
)


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


Saya membuat prompt dengan mengambil tamplate dari langchain yaitu ChatPromptTemplate. Pada di awal saya memberikan role modelnya sebagai asisten yg menjawab pertanyaan mengenai dokumen riset. Lalu aturan lainya yg saya gunakan:

1. Membuat aturan 1 dan 2 agar chat bot hanya menjawab berdasarkan konteks dan tidak boleh diluar konteks, serta jika tidak ada di dalam konteks maka akan diberikan pesan.

2. Aturan 3 saya meminta chat bot untuk selalu memberikan sumber dari setiap informasi yg dihasilkan dalam format sumber hal.N

3. Aturan 4 untuk memastikan bahwa angka yg diebrikan sesuai dengan di dokumen tanpa di bulatkan atau dihitung ulang. Ini mencegah LLM menghitung sendiri.

4. Aturan 5 untuk menyampaikan disclaimer bahawa ini adalah dokumen riset dan bukan rekomendsi investasi.


Sehingga dengan aturan ini akan mencegah halusinasi sistem RAG.

### 3.6 Menyusun konteks + citation
Saya membuat label[...] yang akan membuat LLM bisa menulis citation. Tanpa ini LLM tidak tau informasi di dapat dari chunk berrasal dari halaman berapa.

In [19]:
# Fungsi: gabung_dokumen.

def gabung_dokumen(daftar_dokumen):

    return "\n\n".join(f"[{label_sumber(d)}] {d.page_content}" for d in daftar_dokumen)


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


### 3.7 Membuat Konfigurasi terbaik setelah melakukan pengujian Sebelumnya

In [20]:
# Fungsi: bangun_ulang_index.

CHUNK_SIZE, CHUNK_OVERLAP = 500, 80
K, FETCH_K = 6, 24

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

                                                                                                             
def bangun_ulang_index(simpan=True):

    global chunks, store_vektor, retriever

    chunks = splitter.split_documents(semua_dokumen())
    store_vektor = FAISS.from_documents(chunks, embeddings)
    retriever = buat_retriever(k=K, fetch_k=FETCH_K)["retriever_mmr"]

    if simpan:
        store_vektor.save_local(str(FAISS_INDEX_DIR))

    simpan_metadata_index()
    print(f"Index dibangun ulang: {len(chunks)} chunk · {store_vektor.index.ntotal} vektor")


                                            
bangun_ulang_index()
print(f"Retriever : MMR · k={K} · fetch_k={FETCH_K}")


Index dibangun ulang: 84 chunk · 84 vektor
Retriever : MMR · k=6 · fetch_k=24


**Interpretasi output:** Output menunjukkan Index dibangun ulang: 84 chunk · 84 vektor Retriever : MMR · k=6 · fetch_k=24. Jadi proses pada tahap ini berjalan sesuai alurnya.


Retriever final menggunakan MMR tanpa score_threshold. Keduanya tidak dapat digabungkan karena LangChain hanya menerima satu search_type. Pilihan ini diambil karena chunk tabel Take Profit berskor 0,248 akan tersaring keluar bila threshold 0,3 diterapkan di lapisan retrieval, sementara penolakan pertanyaan di luar cakupan sudah dapat ditangani oleh grounding prompt sebagaimana dibuktikan pada pengujian bagian 4.

### 3.8 Fungsi ask()

Fungsi yg akan dipanggil untuk menjawab pertanyaana user, ini menghubungkan seluruh komponen sebelumnya untuk membuat sistem RAG

In [21]:
# Fungsi: ask.

from langchain_core.output_parsers import StrOutputParser

if llm is not None:
    rantai_rag = PROMPT_RAG | llm | StrOutputParser()
else:
    rantai_rag = None

def ask(pertanyaan, retriever_pilihan=None, tampilkan_chunk=False):
    if rantai_rag is None:
        raise RuntimeError("LLM belum siap. Isi DEEPSEEK_API_KEY lalu jalankan ulang cell LLM.")
    r = retriever_pilihan or retriever
    dokumen = r.invoke(pertanyaan)
    if not dokumen:
        return "Maaf, informasi itu tidak ada di dokumen saya."
    if tampilkan_chunk:
        print(f"🔍 {len(dokumen)} chunk diambil: {[label_sumber(d) for d in dokumen]}\n")
    return rantai_rag.invoke({"context": gabung_dokumen(dokumen), "question": pertanyaan})


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


## 4. Test Sistem Real Case

### 4.1 Menjawab pertanyaan user dari Knowledge utama (analisis Saham 2026)

In [22]:
# Tampilkan hasil proses.

JALANKAN_DEMO_REAL_CASE = False
PERTANYAAN_UJI = [
    "Berapa harga Take Profit BBRI?",
    "Berapa PBV dan dividend yield BBRI?",
    "Bagaimana prospek GOTO?",
    "Resep rendang yang enak apa?",
]
if JALANKAN_DEMO_REAL_CASE:
    for p in PERTANYAAN_UJI:
        print(f"🧑 Question: {p}\n")
        print(f"🤖: {ask(p)}\n")
else:
    print("Demo real case dilewati. Ubah JALANKAN_DEMO_REAL_CASE = True untuk menjalankannya.")


Demo real case dilewati. Ubah JALANKAN_DEMO_REAL_CASE = True untuk menjalankannya.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


**Hasil Pengujian Sistem RAG End-to-End**

Sistem berhasil menjawab pertanyaan dalam cakupan knowledge base dengan
akurat dan lengkap. Pertanyaan "Berapa harga Take Profit BBRI?" dijawab
dengan ketiga level target sekaligus (TP1 Rp3.400, TP2 Rp3.800, TP3 Rp4.250)
beserta sumbernya. Ini membuktikan bahwa keputusan menggunakan MMR dengan
k = 6 berhasil mengumpulkan chunk tabel Trading Plan yang sebelumnya tidak
terjangkau oleh retriever similarity pada nilai k yang sama.

Pertanyaan mengenai PBV dan dividend yield dijawab dengan angka persis
seperti tertulis di dokumen (1,35x dan 13,75%), tanpa pembulatan maupun
perhitungan ulang. Ini sesuai aturan keempat pada grounding prompt yang
melarang model mengolah angka sendiri, ini adalah aturan yang penting untuk knowledge
base berisi data keuangan, karena kesalahan satu digit dapat mengubah makna
seluruh jawaban.

Kedua pertanyaan di luar cakupan ditolak dengan kalimat yang telah
ditentukan. Yang menarik, penolakan ini terjadi di lapisan generation, bukan
retrieval: retriever MMR yang digunakan tidak memiliki `score_threshold`
sehingga tetap mengembalikan enam chunk untuk pertanyaan "Resep rendang yang
enak apa?", namun grounding prompt membuat model menolak menjawab karena
konteks yang diterima tidak memuat informasi yang diminta.

#### 4.2.2 Menambahkan dokumen riset tambahan

Pada versi final project, knowledge base diperluas dengan empat dokumen riset tambahan:

1. Analisis Investasi BIPI Geopolitik & Fundamental.pdf
2. Analisis Mendalam Saham Barito Group.pdf
3. Blueprint Investasi Presisi Chaos Scenario.pdf
4. Indonesian Equity Trading Research.pdf

Dokumen keempat adalah riset strategis portofolio BBRI + BUMI dengan pembanding ENRG. Penambahannya penting karena berisi trading plan, scenario analysis, risk/reward, thesis invalidation, dan portfolio construction yang tidak identik dengan dokumen riset sebelumnya.

Setelah setiap penambahan, dictionary `KNOWLEDGE_BASE` harus diikuti dengan pembangunan ulang index FAISS. Dengan demikian dokumen baru benar-benar ikut dalam retrieval, bukan hanya tersimpan sebagai file.


#### 4.2.1 Kondisi sebelum pembaruan
Tiga pertanyaan berikut menyangkut saham dan skenario yang sama sekali tidak
dibahas dalam knowledge base saat ini. Jawaban penolakan di bawah menjadi
titik pembanding. Pertanyaan yang sama akan diajukan ulang setelah knowledge
base diperluas pada bagian 4.2.3.

In [23]:
# Tampilkan hasil proses.

PERTANYAAN_DOC_TAMBAHAN = [
    "Analisa Saham BIPI?",
    "Analisa Saham BREN?",
    "Berikan Summary Skenario Greenland?",
]
print("═══ SEBELUM PEMBARUAN ═══")
summary_knowledge_base()
print("\nDemo pertanyaan sebelum pembaruan dilewati agar Run All tidak memanggil API.")


═══ SEBELUM PEMBARUAN ═══
Knowledge base: 1 sumber · 15 bagian · 32,803 karakter:
   • riset-ihsg-2026               15 bagian ·  32,803 karakter

Demo pertanyaan sebelum pembaruan dilewati agar Run All tidak memanggil API.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


#### 4.2.2 Menambahkan dokumen riset tambahan — OPSIONAL

Folder `data/knowledge_base/additional/` hanya berfungsi sebagai **repository dokumen tambahan**. PDF di dalamnya tidak menjadi knowledge base otomatis ketika notebook dimulai.

Untuk menambahkan satu file, panggil:
```python
tambah_pdf_tambahan("Indonesian Equity Trading Research.pdf")
```

Untuk menambahkan semua PDF yang saat ini ada di folder:
```python
tambah_semua_pdf_tambahan()
```

Setelah penambahan, index FAISS dibangun ulang agar dokumen baru ikut digunakan oleh retriever.

In [24]:
# Tampilkan hasil proses.

AKTIFKAN_KB_TAMBAHAN = False

                                                            
FILE_TAMBAHAN_DIPILIH = []
         
                           
                                               
   

                                                                                              
TAMBAH_SEMUA_PDF = False

if AKTIFKAN_KB_TAMBAHAN:
    if TAMBAH_SEMUA_PDF:
        hasil_tambah = tambah_semua_pdf_tambahan(rebuild=True)
    elif FILE_TAMBAHAN_DIPILIH:
        hasil_tambah = []
        for nama_file in FILE_TAMBAHAN_DIPILIH:
            hasil_tambah.append(
                tambah_pdf_tambahan(nama_file, rebuild=False)
            )
        bangun_ulang_index()
    else:
        hasil_tambah = []
        print(
            "⚠️ AKTIFKAN_KB_TAMBAHAN=True tetapi belum ada file yang dipilih "
            "dan TAMBAH_SEMUA_PDF=False."
        )
else:
    hasil_tambah = []
    print("KB tambahan TIDAK dimuat otomatis.")
    print("KB aktif saat ini:", sorted(KNOWLEDGE_BASE.keys()))
    print("PDF tersedia di data/knowledge_base/additional/:")
    for p in daftar_pdf_tambahan():
        source_preview = TAMBAHAN_SOURCE_MAP.get(
            p.name, nama_sumber_dari_file(p)
        )
        print(f"  - {p.name}  → source key: {source_preview}")


KB tambahan TIDAK dimuat otomatis.
KB aktif saat ini: ['riset-ihsg-2026']
PDF tersedia di data/knowledge_base/additional/:
  - Analisis Investasi BIPI Geopolitik & Fundamental.pdf  → source key: riset-bipi-2026
  - Analisis Mendalam Saham Barito Group.pdf  → source key: riset-barito-2026
  - Blueprint Investasi Presisi Chaos Scenario.pdf  → source key: riset-blueprint-2026
  - Indonesian Equity Trading Research.pdf  → source key: riset-equity-2026


**Interpretasi output:** Output menunjukkan KB tambahan TIDAK dimuat otomatis. KB aktif saat ini: ['riset-ihsg-2026']. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [25]:
# Tampilkan hasil proses.

print("KB aktif saat ini:", sorted(KNOWLEDGE_BASE.keys()))
print("Jumlah sumber aktif:", len(KNOWLEDGE_BASE))
if "riset-equity-2026" in KNOWLEDGE_BASE:
    print("✅ riset-equity-2026 aktif karena sudah ditambahkan secara manual.")
else:
    print("ℹ️ riset-equity-2026 belum dimuat; ini normal pada Run All default.")


KB aktif saat ini: ['riset-ihsg-2026']
Jumlah sumber aktif: 1
ℹ️ riset-equity-2026 belum dimuat; ini normal pada Run All default.


**Interpretasi output:** Output menunjukkan KB aktif saat ini: ['riset-ihsg-2026'] Jumlah sumber aktif: 1. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [26]:
# Tampilkan hasil proses.

print("\nAUDIT KURASI & INDEX AKTIF")
sampah = [
    c for c in chunks
    if len(URL_RE.findall(c.page_content)) >= 2
]
print(f"Chunk mengandung ≥2 URL: {len(sampah)}   (target 0)")
if sampah:
    for c in sampah[:5]:
        print(
            "  -",
            c.metadata.get("source"),
            c.metadata.get("page"),
            c.page_content[:180],
        )
    raise RuntimeError(
        "Masih ada chunk dengan ≥2 URL. Perbaiki kurasi sebelum evaluasi."
    )

indexed_sources = {
    d.metadata.get("source")
    for d in chunks
    if d.metadata.get("source")
}

for nama, daftar in KNOWLEDGE_BASE.items():
    if not daftar:
        raise RuntimeError(f"Knowledge base source '{nama}' kosong.")
    if nama not in indexed_sources:
        raise RuntimeError(
            f"Source '{nama}' ada di KNOWLEDGE_BASE tetapi belum masuk index."
        )

print("Total chunk:", len(chunks))
print("Sumber ter-index:", sorted(indexed_sources))
print("✅ Semua sumber AKTIF berhasil masuk index.")



AUDIT KURASI & INDEX AKTIF
Chunk mengandung ≥2 URL: 0   (target 0)
Total chunk: 84
Sumber ter-index: ['riset-ihsg-2026']
✅ Semua sumber AKTIF berhasil masuk index.


**Interpretasi output:** Output menunjukkan AUDIT KURASI & INDEX AKTIF Chunk mengandung ≥2 URL: 0   (target 0). Jadi proses pada tahap ini berjalan sesuai alurnya.


Telah dilakukan perluasan knowledge base dengan empat PDF tambahan. Dokumen `riset-equity-2026` berisi 25 halaman dan menjadi sumber khusus untuk deep research portofolio BBRI + BUMI serta perbandingan ENRG.

Angka jumlah chunk/vektor tidak ditulis statis di markdown karena dapat berubah jika isi dokumen, versi loader, atau konfigurasi chunking berubah. Nilai aktual selalu diambil dari `summary_knowledge_base()`, `len(chunks)`, dan `store_vektor.index.ntotal`.

Dengan demikian, notebook tidak lagi mengunci narasi pada angka 299 chunk dari versi sebelumnya.


#### 4.2.3 Pengujian setelah pembaruan

In [27]:
# Tampilkan hasil proses.

JALANKAN_DEMO_SETELAH_UPDATE = False
if JALANKAN_DEMO_SETELAH_UPDATE:
    for p in PERTANYAAN_DOC_TAMBAHAN:
        print(f"🧑 {p}")
        print(f"🤖 {ask(p, tampilkan_chunk=True)}\n")
        print("-" * 78)
else:
    print("Demo setelah update dilewati. Ubah JALANKAN_DEMO_SETELAH_UPDATE = True untuk menjalankannya.")


Demo setelah update dilewati. Ubah JALANKAN_DEMO_SETELAH_UPDATE = True untuk menjalankannya.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


3 pertanyaan yg tidak terjawab di knowledge pada dokumen awal (riset-ihsg-2026), sudah berhasil di jawab setelah setelah saya menambah 3 dokumen tambahan terbaru yg menjawab pertanyaan sangat spesifik pada saham BIPI dan BREN, serta analisa skenario di Greenland. Semua output yg diberikan sudah sesuai dan akurat dengan dokumen aslinya.

#### 4.2.4 Mengganti isi dokumen

Requirement pembaruan knowledge base mencakup dua operasi yaitu menambahkan dan
mengganti. Bagian 4.2.2 telah membuktikan penambahan. Di sini diuji operasi
mengganti, yaitu memperbarui isi sumber yang sudah ada tanpa menambah sumber
baru, lalu memeriksa apakah perubahan tersebut sampai ke jawaban chatbot.

In [28]:
# Tampilkan hasil proses.

JALANKAN_DEMO_CATATAN = False

if JALANKAN_DEMO_CATATAN:
    tambah_dokumen(
        "catatan-pemantauan",
        """CATATAN PEMANTAUAN INTERNAL — versi 1 (20 Agustus 2026).
        Status posisi BBRI: belum dibuka. Menunggu konfirmasi harga masuk di zona
        Rp2.980 - Rp3.100 sesuai trading plan. Belum ada eksekusi pembelian.""",
        metadata_tambahan={"tipe": "catatan-internal", "versi": 1},
    )
    bangun_ulang_index()
    print("Catatan versi 1 ditambahkan dan index dibangun ulang.")
else:
    print("Demo catatan internal dilewati; KB tetap tidak berubah.")


Demo catatan internal dilewati; KB tetap tidak berubah.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


In [29]:
# Tampilkan hasil proses.

if JALANKAN_DEMO_CATATAN:
    ganti_dokumen(
        "catatan-pemantauan",
        """CATATAN PEMANTAUAN INTERNAL — versi 2 (21 Agustus 2026).
        Status posisi BBRI: tahap pertama SUDAH DIBUKA pada harga Rp3.050 dengan
        porsi 30% dari alokasi. Stop-loss aktif di Rp2.900. Sisa 70% menunggu
        konfirmasi penutupan gap.""",
    )
    bangun_ulang_index()
    print("Catatan versi 2 dipasang dan index dibangun ulang.")
else:
    print("Demo replace catatan dilewati; KB tetap tidak berubah.")


Demo replace catatan dilewati; KB tetap tidak berubah.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


In [30]:
# Tampilkan hasil proses.

if JALANKAN_DEMO_CATATAN:
    print(ask(PERTANYAAN_GANTI, tampilkan_chunk=True))
    print("\nskor relevansi 10 teratas:")
    for d, s in store_vektor.similarity_search_with_relevance_scores(PERTANYAAN_GANTI, k=10):
        print(f"   {s:+.3f}  [{label_sumber(d)}]  {d.page_content[:70]}…")
else:
    print("Diagnostik retrieval catatan dilewati.")


Diagnostik retrieval catatan dilewati.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


**Analisis Kegagalan Demo Penggantian Dokumen**
Pada percobaan pertama dan kedua dalam mengganti isi dokumen tidak mengubah jawaban sama sekali. Kedua versi catatan-pemantauan menghasilkan jawaban dengan isi yg sama dan sitasi yg sama yaitu [riset-ihsg-2026 hal.13]. Setelah saya melakukan pemeriksaan chunk yg di ambil, saya menemukan penyebab kegagalan bukan dari fungsi `ganti_dokumen()`, tetapi masalah retrieval. Dokumen catatan sebenarnya memperoleh skor relevansi +0,399 dan menempati peringkat kedua, hanya selisih 0,006 dari peringkat pertama (+0,405). Secara kemiripan, catatan tersebut sangat mirip dan menjadi kandidat terbaik.  Tetapi yang dipilih oleh mekanisme MMR adalah peringkat pertama.

**Maka selanjutnya,** saya akan melakukan perbaikan  dengan metadata filtering dengan retrieval untuk pertanyaan
mengenai catatan internal dibatasi pada dokumen bertipe `catatan-internal`, sehingga tidak perlu bersaing maupun tersaring oleh mekanisme keragaman MMR. Struktur metadata yang disiapkan sejak bagian 2.4 terbukti berguna untuk menangani kasus ini.

In [31]:
# Tampilkan hasil proses.

retriever_catatan = None
if "catatan-pemantauan" in KNOWLEDGE_BASE:
    retriever_catatan = store_vektor.as_retriever(
        search_kwargs={
            "k": 3,
            "filter": {"tipe": "catatan-internal"},
        }
    )
    print("Retriever metadata catatan siap.")
else:
    print("Catatan internal belum dimuat; retriever metadata catatan dilewati.")


Catatan internal belum dimuat; retriever metadata catatan dilewati.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


Operasi `tambah_dokumen()` dan `ganti_dokumen()` tetap tersedia sebagai mekanisme manual. Pada Run All default, knowledge base hanya berisi dokumen utama dari folder `data/`. PDF di `data/knowledge_base/additional/` baru masuk ke knowledge base setelah fungsi penambahan dipanggil.

## 5. Evaluasi & Analisis

### 5.1 Rancangan dan pelaksanaan evaluasi

In [32]:
# Fungsi: soal.

KATEGORI = {
    "riset-ihsg-2026":      "in-scope · IHSG 2026",
    "riset-barito-2026":    "in-scope · Barito Group",
    "riset-bipi-2026":      "in-scope · BIPI",
    "riset-blueprint-2026": "in-scope · Blueprint",
    "riset-equity-2026":   "in-scope · Indonesian Equity Trading Research",
}

def soal(pertanyaan, sumber, *kata_kunci):

    return {
        "pertanyaan": pertanyaan,
        "kategori": KATEGORI.get(sumber, "out-of-scope"),
        "sumber_benar": sumber,
        "kata_kunci": list(kata_kunci),
        "bisa_dijawab": sumber is not None,
    }


IHSG, BARITO, BIPI, BLUEPRINT, EQUITY = (
    "riset-ihsg-2026", "riset-barito-2026", "riset-bipi-2026", "riset-blueprint-2026", "riset-equity-2026",
)

EVALUASI = [
                                                             
    soal("Berapa level tertinggi historis IHSG pada 20 Januari 2026?", IHSG, "9.134,70"),
    soal("Berapa nilai Target Ambil Untung (TP) 1 BBRI?", IHSG, "3.400"),
    soal("Berapa nilai Target Ambil Untung (TP) 2 BBRI?", IHSG, "3.800"),
    soal("Berapa nilai Target Ambil Untung (TP) 3 BBRI?", IHSG, "4.250"),
    soal("Di level berapa IHSG ditutup pada 20 Mei 2026?", IHSG, "6.318,50"),
    soal("Berapa BI Rate per Mei 2026?", IHSG, "5,25"),
    soal("Berapa pertumbuhan ekonomi Indonesia pada kuartal I-2026?", IHSG, "5,61"),
    soal("Di level berapa gap bawah historis IHSG yang belum tertutup?", IHSG, "6.057"),
    soal("Apa nama BUMN khusus eksportir tunggal bentukan pemerintah?", IHSG, "danantara"),
    soal("Berapa tingkat dividend yield BBRI saat ini?", IHSG, "13,75"),
    soal("Berapa rasio PBV BBRI saat ini?", IHSG, "1,35"),
    soal("Berapa probabilitas terjadinya Skenario A?", IHSG, "65%"),
    soal("Berapa probabilitas terjadinya Skenario B?", IHSG, "25%"),
    soal("Berapa probabilitas terjadinya Skenario C?", IHSG, "10%"),
    soal("Berapa rentang target bullish IHSG akhir tahun 2026?", IHSG, "8.873", "9.000"),
    soal("Berapa total cadangan devisa RI per April 2026?", IHSG, "146,2"),
    soal("Kapan tanggal efektif keluarnya saham-saham dari indeks MSCI?", IHSG, "juni 2026"),
    soal("Apa saham cadangan utama pertama (Backup Pick 1)?", IHSG, "bmri"),
    soal("Apa saham cadangan kedua (Backup Pick 2)?", IHSG, "medc"),
    soal("Berapa batas harga stop-loss BBRI?", IHSG, "2.900"),
    soal("Berapa rentang harga untuk pembelian awal (initial entry) BBRI?", IHSG, "3.000", "3.100"),
    soal("Berapa tingkat ROE BBRI?", IHSG, "18,1"),
    soal("Berapa target defisit APBN tahun 2027?", IHSG, "1,8", "2,4"),
    soal("Berapa tingkat imbal hasil (yield) US Treasury 10 tahun?", IHSG, "4,66"),
    soal("Sektor apa yang menjadi penerima manfaat paling masif dari kebijakan DHE SDA?", IHSG, "perbankan"),

                                                              
    soal("Siapa Ultimate Beneficiary Owner (UBO) Barito Group?", BARITO, "prajogo"),
    soal("Siapa pembeli listrik jangka panjang BREN?", BARITO, "pln"),
    soal("Berapa pendapatan konsolidasian BREN pada kuartal I 2026?", BARITO, "165"),
    soal("Berapa pendapatan CUAN pada kuartal I 2026?", BARITO, "371,33"),
    soal("Apa sektor operasional utama CUAN?", BARITO, "tambang"),
    soal("Berapa rasio utang terhadap ekuitas (DER) CUAN pada kuartal I 2026?", BARITO, "3,30"),
    soal("Kapan efektif delisting BREN dan CUAN dari MSCI Global Standard Index?", BARITO, "29 mei"),
    soal("Berapa nilai functional free float BREN menurut analisis HSC April 2026?", BARITO, "2,69"),
    soal("Berapa margin EBITDA BREN pada kuartal I 2026?", BARITO, "87,6"),
    soal("Berapa harga IPO BREN pada Oktober 2023?", BARITO, "780"),
    soal("Berapa rasio pemecahan saham (stock split) CUAN?", BARITO, "10:1"),
    soal("Apa bentuk natural hedging yang dimiliki BREN?", BARITO, "usd"),
    soal("Berapa arus kas operasional CUAN pada kuartal I 2026?", BARITO, "45,32"),
    soal("Berapa rasio utang bersih terhadap ekuitas (net debt-to-equity) BREN?", BARITO, "1,77"),
    soal("Siapa pemberi utang sindikasi CUAN senilai USD 609,65 juta?", BARITO, "mandiri", "bni"),
    soal("Berapa area support terkuat (HVN) CUAN?", BARITO, "464"),
    soal("Berapa batas proteksi (stop loss) untuk perdagangan saham BREN?", BARITO, "2.900"),
    soal("Berapa target profit jangka pendek untuk saham CUAN?", BARITO, "850"),
    soal("Berapa harga IPO CUAN yang disesuaikan pasca stock split?", BARITO, "20 per saham"),
    soal("Apa tren yang terjadi pasca penutupan rebalancing MSCI pada 29 Mei 2026?", BARITO, "rebound"),
    soal("Apa risiko utama dari struktur kepemilikan yang terkonsentrasi?", BARITO, "indeks"),
    soal("Apa yang terjadi pada aktivitas broker asing terhadap BREN dan CUAN?", BARITO, "net sell"),
    soal("Berapa target harga jangka menengah untuk BREN?", BARITO, "5.800"),
    soal("Apa pola formasi yang sedang dibangun CUAN pada Juni 2026?", BARITO, "double bottom"),
    soal("Mengapa BREN dinilai memiliki kinerja operasional lebih stabil dibanding CUAN?", BARITO, "defensif"),

                                                 
    soal("Berapa total aset BIPI menurut laporan kuartal ketiga tahun 2025?", BIPI, "27,32"),
    soal("Berapa nilai total kewajiban (liabilitas) BIPI?", BIPI, "17,51"),
    soal("Berapa posisi kas dan setara kas BIPI pada September 2025?", BIPI, "49,1"),
    soal("Berapa nilai restrukturisasi utang pada anak usaha BIPI, Nixon Investments Pte Ltd?", BIPI, "235"),
    soal("Apa segmen utama yang menopang pendapatan BIPI?", BIPI, "pertambangan"),
    soal("Berapa persentase marjin EBITDA BIPI pada pertengahan tahun 2025?", BIPI, "22,3"),
    soal("Berapa beban bunga BIPI pada periode dengan EBITDA Rp535,3 miliar?", BIPI, "573,0"),
    soal("Berapa kerugian bersih BIPI pada kuartal II 2025?", BIPI, "160,7"),
    soal("Apa indikator bahwa kas bersih BIPI tidak mampu menutupi kewajibannya?", BIPI, "arus kas"),
    soal("Berapa nilai Debt-to-Equity Ratio (DER) BIPI pada kuartal III 2025?", BIPI, "1,79"),
    soal("Apa red flag utama dalam analisis kredit korporasi BIPI terkait beban bunga?", BIPI, "1,0"),
    soal("Berapa nilai Price-to-Book Value (PBV) saham BIPI?", BIPI, "0,55"),
    soal("Rasio valuasi apa yang paling representatif untuk perusahaan infrastruktur berutang tinggi?", BIPI, "ebitda"),
    soal("Berapa rasio EV/EBITDA BIPI saat ini?", BIPI, "41,36"),
    soal("Apa titik jepit geopolitik yang memfasilitasi 20% pasokan minyak dunia?", BIPI, "hormuz"),
    soal("Apa sebutan untuk kepanikan pengadaan pasokan yang memicu kenaikan harga batu bara?", BIPI, "restocking"),
    soal("Berapa target produksi batu bara nasional tahun 2026 yang direvisi?", BIPI, "790"),
    soal("Berapa harga terendah sepanjang masa Rupiah per Dolar AS pada April 2026?", BIPI, "17.138"),
    soal("Berapa imbal hasil obligasi pemerintah 10 tahun per April 2026?", BIPI, "6,57"),
    soal("Berapa harga tertinggi sepanjang masa saham BIPI pada 26 Februari 2026?", BIPI, "342"),
    soal("Siapa investor strategis yang masuk ke BIPI pada 2026?", BIPI, "bakrie"),
    soal("Perusahaan apa yang menjadi mitra BIPI dalam pengembangan infrastruktur LNG?", BIPI, "indogas"),
    soal("Berapa jumlah saham beredar BIPI saat ini?", BIPI, "63,71"),
    soal("Berapa target harga konservatif dari institusi untuk BIPI?", BIPI, "290"),
    soal("Apa risiko jika RUPSLB gagal mengesahkan restrukturisasi HMETD?", BIPI, "default"),

                                                     
    soal("Berapa total modal investasi yang tersedia dalam blueprint?", BLUEPRINT, "27.000.000"),
    soal("Apa nama skenario utama dalam laporan blueprint?", BLUEPRINT, "chaos"),
    soal("Apa aset ofensif yang direkomendasikan?", BLUEPRINT, "mp materials"),
    soal("Apa aset defensif yang direkomendasikan?", BLUEPRINT, "antm"),
    soal("Berapa persen alokasi modal untuk cadangan tunai?", BLUEPRINT, "20%"),
    soal("Apa kepanjangan dari REE?", BLUEPRINT, "rare earth"),
    soal("Wilayah mana yang menjadi pusat perhatian geopolitik dalam dokumen?", BLUEPRINT, "greenland"),
    soal("Strategi portofolio apa yang digunakan dalam blueprint?", BLUEPRINT, "barbell"),
    soal("Apa sebutan untuk institusi besar penyedia likuiditas yang menciptakan volatilitas?", BLUEPRINT, "shadow hands"),
    soal("Kapan target keluar dari pasar (exit) menurut dokumen?", BLUEPRINT, "mei 2026"),
    soal("Berapa persen kenaikan saham Molycorp pada krisis 2010?", BLUEPRINT, "465"),
    soal("Berapa drawdown ANTM saat perang dagang 2018?", BLUEPRINT, "34,5"),
    soal("Apa peran utama MP Materials dalam portofolio?", BLUEPRINT, "ofensif"),
    soal("Apa peran utama ANTM dalam portofolio?", BLUEPRINT, "lindung nilai"),
    soal("Di mana lokasi tambang MP Materials?", BLUEPRINT, "mountain pass"),
    soal("Berapa prediksi pelemahan nilai tukar Rupiah terhadap USD?", BLUEPRINT, "17.500"),
    soal("Kapan zona waktu Sniper Zone untuk eksekusi penuh?", BLUEPRINT, "februari 2026"),
    soal("Berapa target harga jual TP 1 untuk MP Materials?", BLUEPRINT, "79"),
    soal("Berapa target harga jual TP 1 untuk ANTM?", BLUEPRINT, "4.200"),
    soal("Berapa zona harga All-In untuk MP Materials?", BLUEPRINT, "51.50"),
    soal("Berapa zona harga All-In untuk ANTM?", BLUEPRINT, "3.450"),
    soal("Apa nama metodologi trading yang dipetakan dalam dokumen?", BLUEPRINT, "time trading"),
    soal("Apa risiko yang ditakuti trader dengan leverage tinggi?", BLUEPRINT, "margin call"),
    soal("Berapa persen alokasi modal untuk MP Materials?", BLUEPRINT, "40%"),
    soal("Apa indikator teknikal yang digunakan sebagai support dinamis di MP Materials?", BLUEPRINT, "ma50"),

                                                            
    soal("Berapa total modal kerja dalam portofolio utama?", EQUITY, "14.000.000"),
    soal("Berapa bobot alokasi BBRI dalam portofolio utama?", EQUITY, "55%"),
    soal("Berapa bobot alokasi BUMI dalam portofolio utama?", EQUITY, "45%"),
    soal("Berapa batas maksimum risiko portofolio?", EQUITY, "1.000.800"),
    soal("Berapa tingkat confidence final portfolio BBRI + BUMI?", EQUITY, "83%"),
    soal("Saham energi mana yang dipilih sebagai pasangan final BBRI?", EQUITY, "BUMI"),
    soal("Berapa pertumbuhan laba bersih BUMI pada H1 2026?", EQUITY, "188,4%"),
    soal("Berapa besar pengurangan utang BUMI?", EQUITY, "60%"),
    soal("Berapa valuasi EV/Revenue BUMI?", EQUITY, "0,03x"),
    soal("Berapa target harga analis BUMI menurut dokumen?", EQUITY, "Rp300"),
    soal("Berapa area major support BUMI?", EQUITY, "Rp180", "Rp184"),
    soal("Berapa target breakout/minor resistance BUMI?", EQUITY, "Rp202", "Rp210"),
    soal("Berapa major resistance BUMI?", EQUITY, "Rp250", "Rp300"),
    soal("Berapa technical stop loss BUMI?", EQUITY, "Rp172"),
    soal("Berapa TP1 BBRI dalam final trading plan?", EQUITY, "Rp3.400"),
    soal("Berapa TP2 BBRI dalam final trading plan?", EQUITY, "Rp3.780"),
    soal("Berapa TP3 BBRI dalam final trading plan?", EQUITY, "Rp4.010"),
    soal("Berapa technical stop loss BBRI?", EQUITY, "Rp2.870"),
    soal("Berapa target BBRI pada bull case?", EQUITY, "Rp4.200"),
    soal("Berapa target BBRI pada base case?", EQUITY, "Rp3.780"),
    soal("Berapa target downside BBRI pada bear case?", EQUITY, "Rp2.700"),
    soal("Berapa target BUMI pada bull case?", EQUITY, "Rp300"),
    soal("Berapa target BUMI pada base case?", EQUITY, "Rp250"),
    soal("Berapa target downside BUMI pada bear case?", EQUITY, "Rp160"),
    soal("Berapa risk/reward BUMI menuju TP2?", EQUITY, "1 : 4,27"),

                                          
                                                            
    soal("Bagaimana prospek saham GOTO?", None),
    soal("Bagaimana kinerja saham UNVR tahun 2026?", None),
    soal("Berapa target harga saham ICBP?", None),
    soal("Apakah saham BUKA layak dibeli?", None),
    soal("Bagaimana analisis fundamental saham SIDO?", None),
    soal("Berapa dividend yield saham INDF?", None),
    soal("Bagaimana prospek saham ARTO?", None),
    soal("Berapa rasio PBV saham MYOR?", None),

                                         
    soal("Berapa harga penutupan IHSG hari ini?", None),
    soal("Bagaimana proyeksi IHSG untuk tahun 2030?", None),
    soal("Berapa harga saham BBRI pada Desember 2027?", None),

                                              
    soal("Saham apa yang cocok untuk profil risiko saya?", None),
    soal("Berapa modal minimal yang harus saya siapkan untuk mulai berinvestasi?", None),
    soal("Apakah saya sebaiknya menjual seluruh portofolio saya sekarang?", None),

                                                    
    soal("Broker saham mana yang biaya transaksinya paling murah?", None),
    soal("Berapa harga Bitcoin saat ini?", None),
    soal("Bagaimana cara membeli reksa dana pasar uang?", None),
    soal("Berapa suku bunga deposito BCA saat ini?", None),
    soal("Apa itu obligasi ritel ORI dan bagaimana cara membelinya?", None),
    soal("Berapa tarif pajak final atas dividen saham di Indonesia?", None),

                                   
    soal("Resep rendang yang enak apa?", None),
    soal("Bagaimana cara memasak nasi goreng?", None),
    soal("Bagaimana cuaca di Jakarta besok?", None),
    soal("Apa ibu kota Provinsi Papua?", None),
    soal("Bagaimana cara belajar bahasa Inggris dengan cepat?", None),
]

print(f"Total pertanyaan uji : {len(EVALUASI)}")
print(f"   in-scope          : {sum(1 for u in EVALUASI if u['bisa_dijawab'])}")
print(f"   out-of-scope      : {sum(1 for u in EVALUASI if not u['bisa_dijawab'])}")
assert len(EVALUASI) == 150, f"Dataset final harus 150 test case, saat ini {len(EVALUASI)}."
assert sum(1 for u in EVALUASI if u["bisa_dijawab"]) == 125, "In-scope final harus 125."
assert sum(1 for u in EVALUASI if not u["bisa_dijawab"]) == 25, "OOS final harus 25."


Total pertanyaan uji : 150
   in-scope          : 125
   out-of-scope      : 25


**Interpretasi output:** Output menunjukkan Total pertanyaan uji : 150 in-scope          : 125. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [33]:
# Tampilkan hasil proses.

JALANKAN_EVALUASI_LEGACY = False

if JALANKAN_EVALUASI_LEGACY:
    print("⚠️ Evaluasi legacy dinonaktifkan pada versi final. Gunakan benchmark P0.")
else:
    hasil = []
    tabel = pd.DataFrame()
    skor = 0
    print("Evaluasi legacy dilewati. Tidak ada request LLM dari cell ini.")


Evaluasi legacy dilewati. Tidak ada request LLM dari cell ini.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


In [34]:
# Fungsi: fakta_penuh.

if hasil:
    ins = [h for h in hasil if h.get("kategori") != "out-of-scope"]
    oos = [h for h in hasil if h.get("kategori") == "out-of-scope"]

    def fakta_penuh(h):
        a, b = h["fakta benar"].split("/")
        return a == b

    ret_ok = sum(h.get("retrieval") == "✅" for h in ins)
    fakta_ok = sum(fakta_penuh(h) for h in ins if h.get("fakta benar") != "—")
    tolak_ins = sum(h.get("menolak") == "ya" for h in ins)
    tolak_oos = sum(h.get("menolak") == "ya" for h in oos)

    print("Rekapitulasi dua lapis")
    print("=" * 60)
    print(f"Lapis retrieval  — dokumen benar terambil : {ret_ok}/{len(ins)}")
    print(f"Lapis generation — seluruh fakta tepat    : {fakta_ok}/{len(ins)}")
    print(f"In-scope yang justru ditolak sistem       : {tolak_ins}/{len(ins)}")
    print(f"Out-of-scope terdeteksi menolak           : {tolak_oos}/{len(oos)}")
    print(f"Skor gabungan                             : {skor}/{len(hasil)}")
else:
    print("Rekap legacy dilewati karena evaluasi legacy tidak dijalankan.")


Rekap legacy dilewati karena evaluasi legacy tidak dijalankan.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


#### 5.2 Hasil Evaluasi Legacy (historis — bukan baseline P0 resmi)

Angka pada bagian ini berasal dari evaluasi legacy sebelum harness P0 diperbaiki. Metodenya masih menggunakan retrieval source-level dan penilaian generation berbasis keyword, sehingga **tidak boleh dipakai sebagai baseline kuantitatif resmi**.

Bagian ini dipertahankan sebagai dokumentasi masalah yang ditemukan. Baseline resmi selanjutnya harus menggunakan raw trace P0, chunk ID, gold chunk terverifikasi, dan metrik yang dapat diaudit.


#### 5.2b Perbandingan Legacy (historis)

Pengujian ulang di bagian ini tetap berguna sebagai observasi perilaku sistem, tetapi bukan pengganti benchmark P0. P0 memisahkan retrieval dan generation dengan trace yang sama dan menyimpan artefak mentah agar hasil dapat diaudit.


In [35]:
# Tampilkan hasil proses.

JALANKAN_RETTEST_LAMA = False
if JALANKAN_RETTEST_LAMA:
    for p in ["Berapa harga Take Profit BBRI?", "Bagaimana prospek GOTO?", "Resep rendang yang enak apa?"]:
        print(f"🧑 {p}\n🤖 {ask(p)}\n{'-' * 74}")
else:
    print("Retest legacy dilewati. Gunakan Trace Test P0 untuk evaluasi yang dapat diaudit.")


Retest legacy dilewati. Gunakan Trace Test P0 untuk evaluasi yang dapat diaudit.


**Interpretasi arsitektur KB.** Knowledge base awal sengaja hanya berisi satu PDF utama dari folder `data/`. Folder `data/knowledge_base/additional/` bukan bagian dari proses loading awal, melainkan tempat menyimpan kandidat PDF yang dapat ditambahkan kemudian. Ketika dokumen tambahan benar-benar dimasukkan, index dibangun ulang sehingga retriever menggunakan sumber baru tersebut.

### 5.3 Analisis

Selama mengerjakan project ini, saya menentukan konfigurasi sistem lewat pengujian, bukan hanya menebak. Dari eksperimen chunking kecil, konfigurasi `500/80` menjadi baseline yang saya freeze. Saat menguji nilai `k`, similarity murni membutuhkan sampai `k=12` untuk mencapai 4/4 pada empat pertanyaan uji, sedangkan MMR mencapai 4/4 pada `k=6`.

Temuan ini membuat saya memilih MMR karena mampu mencapai cakupan penuh dengan jumlah chunk yang lebih kecil pada eksperimen tersebut. Tetapi hasil ini masih berasal dari 4 pertanyaan, sehingga belum boleh disebut sebagai performa retrieval final.

Temuan penting lain muncul saat knowledge base diperluas. Pertanyaan yang sebelumnya mendapatkan beberapa informasi penting bisa kehilangan sebagian informasi ketika jumlah dokumen bertambah karena kuota `k` harus memilih kandidat dari knowledge base yang lebih besar. Ini menunjukkan bahwa konfigurasi retrieval perlu dievaluasi kembali setelah knowledge base berubah.

Saya juga menemukan kasus metadata filtering pada catatan internal. MMR dapat menganggap dokumen yang sangat mirip sebagai redundan, padahal dokumen tersebut justru merupakan sumber yang ingin diprioritaskan. Karena itu, metadata filtering menjadi salah satu jalur yang penting untuk jenis pertanyaan tertentu.

Dari sini saya menyimpulkan bahwa kualitas RAG tidak cukup dinilai dari apakah chatbot bisa menjawab. Saya harus melihat **chunk apa yang diambil, context apa yang dikirim ke LLM, dan apakah jawaban benar-benar berasal dari context tersebut**.

### 5.4 Keterbatasan Sistem

Saat saya mengerjakan sistem ini, **keterbatasan sistem yg saya sadari pertama adalah dari kualitas sumber datanya sendiri**. Saat memeirksa hasil ekstraksi PDF, saya menemukan beberapa angka hilang dari lapisan teks, misalnya kalimat "koreksi harian mengarah tepat ke level..." yang kehilangan nilainya. Selain itu tabel pada beberapa halaman rusak saat diekstrak seperti struktur kolomnya hilang dan sebagian kata terpotong seperti "perbanka n" atau "Rekomen dasi", sehingga kata-kata itu tidak akan pernah cocok saat pencarian semantik. Sehingga sebagus apa pun pipeline yang saya bangun, ada bagian dokumen yang memang tidak bisa dijangkau sistem.


**Keterbatasan kedua ada pada sistemnya sendiri.** Saat mengerjakan bagian retrieval, ternyata tidak ada satu strategi retrieval yang cocok untuk semua pertanyaan, seperti pada MMR bagus untuk pertanyaan yang jawabannya tersebar, tetapi bisa membuang dokumen paling relevan kalau isinya mirip dengan chunk peringkat pertama, seperti yang terjadi pada demo penggantian dokumen (pada 4.2.4).

**Di sisi generation,** grounding prompt saya berhasil mencegah bot menjawab di luar konteks, tapi tidak bisa mencegah bot salah menyebut sumber ketika konteksnya sendiri sudah keliru, dan format sitasinya juga belum terkontrol, karena bot sempat menulis "hal.1" untuk dokumen teks yang tidak punya metadata halaman. Dari sisi arsitektur, setiap kali knowledge base diperbarui saya harus membangun ulang seluruh index, sehingga biaya komputasi mengikuti ukuran knowledge base dan bukan besarnya perubahan, ini masih murah untuk ±300 chunk tetapi tidak akan scalable untuk jutaan dokumen. Terakhir, penilaian evaluasi saya masih berbasis pencocokan string dan sebagian dinilai manual, sehingga jawaban yang sebenarnya benar bisa saja dianggap salah hanya karena beda format penulisan.

## 6. Insight & Recommendation Action 

### 6.1 Insight
Saat mengerjakan project ini, **insight terbesar yg saya dapat adalah membangun RAG itu bukan hanya soal menyusun komponennya, tetapi menguji setiap keputusan yang saya ambil** untuk membangun sistem nya. Hampir semua dugaan awal saya meleset ketika di uji seperti:
- saya kira memperbesar chunk akan membuat jawaban lebih lengkap, ternyata celah skornya malah turun
- saya kira menaikkan k adalah solusi untuk jawaban yang kurang lengkap, ternyata yang lebih efektif adalah mengganti strategi retrievalnya
- saya kira demo mengganti dokumen akan berjalan mulus, tetapi malah justru gagal karena mekanisme MMR yang sebelumnya saya pilih sebagai yang terbaik

Kalau saya tidak mencetak chunk yang diambil setiap kali ada hasil aneh, semua kegagalan itu tidak akan pernah terlihat, Karena bot tetap menjawab dengan lancar dan meyakinkan meskipun sumbernya salah.

**Insight kedua,** tidak ada satu konfigurasi yang optimal untuk semua jenis pertanyaan. MMR unggul untuk pertanyaan yang jawabannya tersebar di beberapa halaman, tetapi gagal untuk pertanyaan yang jawabannya hanya ada di satu dokumen kecil. Threshold bagus untuk menolak pertanyaan di luar cakupan, tetapi ikut membuang chunk relevan yang skornya rendah. **Artinya sistem RAG yang baik bukan sistem dengan satu parameter paling sempurna, melainkan sistem yang punya beberapa jalur retrieval sesuai karakter pertanyaannya.** Di sisi lain, saya juga melihat langsung kelebihan RAG dibanding fine-tuning, yaitu dengan menambah tiga dokumen riset baru hanya butuh beberapa detik membangun ulang index, tanpa training ulang sama sekali, dan bot langsung bisa menjawab topik yang sebelumnya ditolak.

### 6.2 Recommendation Action
- Untuk pengembangan selanjutnya, prioritas pertama saya adalah memperbaiki kualitas knowledge base, karena dari situ sumber masalah terbesarnya berasal. Misalnya tabel pada beberapa halaman PDF perlu diekstrak dengan parser khusus tabel atau diketik ulang sebagai dokumen teks terpisah, supaya angka-angka penting tidak hilang.

- Selanjutnya saya ingin menambahkan hybrid search yang menggabungkan pencarian kata kunci dan vektor, karena knowledge base saya penuh kode saham seperti BBRI, BREN, BIPI dan sektor perbankan/energi lainnya yang lebih tepat dicari secara eksak daripada secara semantik.
Setelah itu, reranking dapat dipakai untuk memperbaiki urutan kandidat sebelum dikirim ke LLM.

- Dari sisi arsitektur, routing retrieval berbasis metadata yang sudah terbukti berhasil pada bagian 4.2.4 sebaiknya dibuat otomatis, sehingga sistem sendiri yang memilih jalur pencarian sesuai jenis pertanyaan, bukan ditentukan manual.

- Pembaruan knowledge base juga perlu diubah menjadi dengan `delete` dan `add` berbasis id dokumen, menggantikan pembangunan ulang seluruh index yang tidak akan scalable. Untuk evaluasi, saya ingin mengotomasi penilaiannya menggunakan RAGAS agar tidak lagi bergantung pada pencocokan string dan penilaian manual, atau saya coba pelajarin langfuse.

**Terakhir** chatbot ini bisa dikembangkan menjadi asisten riset pribadi dengan menambahkan riwayat percakapan dan query rewriting, sehingga saya bisa bertanya lanjutan tanpa harus mengulang konteks, dan tidak perlu membaca ulang puluhan halaman riset setiap kali butuh satu angka.

## 7. Reflection Question
**Setelah menyelesaikan assignment ini, jawablah pertanyaan berikut:**

1. Setelah membangun aplikasi RAG, bagian mana dari proses pengambilan informasi (retrieval) yang membuatmu paling memahami pentingnya kualitas knowledge base dalam menghasilkan jawaban yang relevan? Jelaskan bagaimana pengalaman ini mengubah caramu melihat proses pencarian informasi dalam sebuah chatbot.

**Jawaban:**
Setelah mengerjakan project tersebut, proses yg menyadarkan saya pentingnya kualitas knowledge base adalah saat mencetak chunk yang di ambil retriever. Sebelum itu saya selalu menilai sistem dari jawabannya, dan jawabannya selalu terlihat rapi dan meyakinkan. Ketika chunkknya saya lihat, saya sadar bot bisa menjawab dengan lancar dari potongan dokumen yg sebenarnya salah. Momen yg paling jelas terjadi ketika demo mengganti dokumen. Ketika jawaban versi lama dan versi baru sama persis, dan bot menulis "menurut catatan pemantauan internal riset-ihsg-2026 hal.13" padahal sumber yg disitasi adalah dokumen riset yg lain. Selain itu saat memeriksa hasil ekstraksi PDF, saya menemukan ada angka yg memang hilang dari dokumennya dan tabel yg rusak sampai datanya terpotong seperti "perbanka n", bagian itu tidaka akn di temukan sistem mau sebagus apapun pipeline yg saya buat.  

Dari pengalaman tersebut, mengubah cara saya membuat sistem RAG, dulu saya pikir kecerdasan chat bot ada di modelnya, jadi kalau jawabannya salah berarti modelnya kurang pintar. Ternyata, setelah mengerjakan projek ini saaya berpikir sebaliknya, bahwa LLM hanya menulis ulang apa yg di berikan sama dia, jadi kalau informasi yg diberikan salah maka hasilnya juga salah. Maka dari itu, bagian yg justru
paling menentukan kualitas sistem RAG adalah dokumennya bersih atau tidak, lalu tahap chunking nya sudah tepat atau belum, dan chunkingnya sudah tepat yg diambil atau tidak.

2. Dalam memilih LLM, embedding model, dan vector database, apa hal yang membuatmu menyadari bahwa arsitektur teknis sebuah sistem RAG harus disesuaikan dengan kebutuhan pengguna dan konteks data? Jelaskan bagaimana keputusan teknismu dipengaruhi oleh karakteristik topik knowledge base yang kamu gunakan.

**jawaban:**
Yang membuat saya sadar bahwa arsitektur sistem RAG harus sesuai dengan konteks datanya adalah ketika setiap keputusan teknis saya berdasarkan pertimbangan yg matang. Misalnya saya memilih model embedding multilingual karena knowledge base saya berbahasa Indonesia tetapi penuh istilah keuangan berbahasa Inggris seperti PBV, forward PE, dan stop-loss, sehingga model English-only maupun model yang hanya paham Bahasa Indonesia sama-sama tidak cukup. Tetapi kekurangannya model ini bukan di buat khusus untuk konteks berbahasa indonesia dan tidak spesifik di keuangan. Lalu pada pemilihan vektor database saya menggunakan FAISS karena knowledge base saya kecil, lalu di pakai sendiri di local tanpa perlu jaringan internet. Jika saya pakai Pinecone akan menambah biaya lagi. Lalu untuk LLM saya tidak memilih model dengan pengetauan yg seluas mungkin, tetapi yg saya cari adalah model yg patuh terhadap aturan dan konteks, karena pengetahuannya memang dari eksternal (dokumen pdf), dan pastinya model ini gratis.


Karena karakter topik saya isinya adalah research financial market, maka saya menambahkan aturan agar angka disalin persis tanpa dihitung ulang atau di kompres digit angkanya oleh LLM, karena jika 1 digit saja hilang maka akan mengubah seluruh strategi dan arti dari jawaban tersebut. Lalu di dokumen saya berisi rekomendasi saham, maka saya menambahkan disclaimer bahwa ini hanya ringkasan dokumen dan bukan saran investasi.

# P0 — Evaluation Harness V2 (Baseline yang dapat diaudit)

> **Tujuan bagian ini:** memperbaiki alat ukur terlebih dahulu tanpa mengubah konfigurasi RAG utama.

Evaluasi lama tetap dipertahankan sebagai catatan eksperimen, tetapi **skor 16,8% tidak diperlakukan sebagai baseline resmi** karena checkpoint lama tidak menyimpan raw answer/context dan retrieval dinilai pada level `source`, bukan level chunk.

Pada P0 ini kita memperbaiki tiga hal paling penting:

1. **Satu pertanyaan → satu retrieval → satu generation** agar context yang diukur sama dengan context yang dipakai LLM.
2. Menyimpan **raw answer, retrieved chunks, context, latency, error, dan konfigurasi** ke checkpoint JSON.
3. Memisahkan **retrieval**, **generation**, dan **out-of-scope refusal** sebagai lapisan evaluasi yang berbeda.

> **Catatan:** P0 ini belum mengganti embedding, chunking, MMR, hybrid search, reranker, atau model LLM. Konfigurasi RAG saat ini tetap dijadikan baseline.

## P0.1 — Freeze konfigurasi baseline

Konfigurasi di bawah mengikuti konfigurasi eksperimen yang sudah dipilih sebelumnya: chunk `500/80`, MMR `k=6`, dan `fetch_k=24`. Jangan mengubah nilai ini selama baseline sedang diukur.

In [36]:
# Proses utama pada bagian ini.

BASELINE_V1 = {
    "chunk_size": 500,
    "chunk_overlap": 80,
    "retriever": "MMR",
    "k": 6,
    "fetch_k": 24,
    "embedding": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "llm": "deepseek-v4-flash",
    "temperature": 0,
}

BASELINE_V1


{'chunk_size': 500,
 'chunk_overlap': 80,
 'retriever': 'MMR',
 'k': 6,
 'fetch_k': 24,
 'embedding': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
 'llm': 'deepseek-v4-flash',
 'temperature': 0}

**Interpretasi output:** Output menunjukkan {'chunk_size': 500, 'chunk_overlap': 80,. Jadi proses pada tahap ini berjalan sesuai alurnya.


## P0.2 — Trace RAG: retrieval dan generation harus memakai context yang sama

Sebelumnya fungsi evaluasi memanggil retriever lalu memanggil `ask()`, sedangkan `ask()` melakukan retrieval lagi. Pada versi ini retrieval hanya dilakukan **sekali** untuk setiap pertanyaan.

Hasil retrieval tersebut langsung diberikan ke `rantai_rag`, lalu seluruh artefaknya disimpan.

# P0 — Evaluation Harness V3 (Baseline yang dapat diaudit)

Versi ini memperbaiki alur P0 supaya **tidak ada eksekusi benchmark sebelum seluruh fungsi siap**, checkpoint dapat dilanjutkan dengan aman, dan anotasi `gold_chunk_ids` tidak hilang saat notebook di-run ulang.

**Urutan:** setup → schema → trace → smoke → anotasi gold → full benchmark → metrics → final status.

> `prepare` tidak melakukan request API. `smoke` menjalankan 1 trace + maksimal 3 pertanyaan. `full` melanjutkan checkpoint sampai 125 pertanyaan.


In [37]:
# Import dan setup yang dibutuhkan.

from pathlib import Path
import hashlib
import json
import os
import time
import re
import pandas as pd

P0_MODE = "full"                                                          

P0_DIR = EVALUATION_DIR
KB_SCOPE = tuple(sorted(KNOWLEDGE_BASE.keys()))
KB_SCOPE_TAG = hashlib.sha1(
    "|".join(KB_SCOPE).encode("utf-8")
).hexdigest()[:10]

CHECKPOINT_P0 = CHECKPOINTS_DIR / f"hasil_evaluasi_p0_{KB_SCOPE_TAG}.json"
GOLD_P0 = GOLD_VERIFIED_DIR / f"gold_annotations_p0_{KB_SCOPE_TAG}.json"
GOLD_TEMPLATE_P0 = GOLD_CANDIDATES_DIR / f"gold_annotations_p0_{KB_SCOPE_TAG}_template.csv"
GOLD_REVIEW_P0 = GOLD_REVIEW_DIR / f"gold_review_p0_{KB_SCOPE_TAG}.csv"

DELAY_PER_REQUEST_P0 = 1.5
MAX_RETRIES_P0 = 3
BACKOFF_P0 = 2

if P0_MODE not in {"prepare", "smoke", "full"}:
    raise ValueError("P0_MODE harus 'prepare', 'smoke', atau 'full'.")

print("=" * 72)
print("P0 CONFIG")
print("=" * 72)
print(f"P0_MODE       : {P0_MODE}")
print(f"KB scope      : {KB_SCOPE}")
print(f"KB scope tag  : {KB_SCOPE_TAG}")
print("Dataset aktif : dibangun pada cell P0.4 setelah konfigurasi ini.")
print(f"Checkpoint    : {CHECKPOINT_P0.resolve()}")
print(f"Gold file     : {GOLD_P0.resolve()}")
print(f"Gold review   : {GOLD_REVIEW_P0.resolve()}")
print(f"Results dir   : {RESULTS_DIR.resolve()}")


P0 CONFIG
P0_MODE       : full
KB scope      : ('riset-ihsg-2026',)
KB scope tag  : 2d2ef880f6
Dataset aktif : dibangun pada cell P0.4 setelah konfigurasi ini.
Checkpoint    : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/checkpoints/hasil_evaluasi_p0_2d2ef880f6.json
Gold file     : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/verified/gold_annotations_p0_2d2ef880f6.json
Gold review   : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/review/gold_review_p0_2d2ef880f6.csv
Results dir   : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/results


**Interpretasi output:** Output menunjukkan ======================================================================== P0 CONFIG. Jadi proses pada tahap ini berjalan sesuai alurnya.


## P0.1 — Freeze konfigurasi baseline

Konfigurasi baseline **tidak diubah** selama pengukuran. Semua trace menyimpan salinan konfigurasi ini agar hasil dapat diaudit.


In [38]:
# Tampilkan hasil proses.

BASELINE_V1 = {
    "chunk_size": 500,
    "chunk_overlap": 80,
    "retriever": "MMR",
    "k": 6,
    "fetch_k": 24,
    "embedding": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "llm": "deepseek-v4-flash",
    "temperature": 0,
}

print(BASELINE_V1)


{'chunk_size': 500, 'chunk_overlap': 80, 'retriever': 'MMR', 'k': 6, 'fetch_k': 24, 'embedding': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'llm': 'deepseek-v4-flash', 'temperature': 0}


**Interpretasi output:** Output menunjukkan {'chunk_size': 500, 'chunk_overlap': 80, 'retriever': 'MMR', 'k': 6, 'fetch_k': 24, 'embedding': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'llm': 'deepseek-v4-flash', 'temperature': 0}. Jadi proses pada tahap ini berjalan sesuai alurnya.


## P0.2 — Trace RAG

Retrieval dilakukan **tepat satu kali**. Chunk yang diperoleh dipakai langsung untuk membentuk context yang dikirim ke LLM. Trace menyimpan retrieval, context, answer, latency, konfigurasi, dan error dalam satu record.


In [39]:
# Fungsi: stable_chunk_id, serialize_chunk, run_rag_trace.

def stable_chunk_id(dokumen):

    source = str(dokumen.metadata.get("source", "dokumen"))
    page = str(dokumen.metadata.get("page", ""))
    text = dokumen.page_content.strip()
    payload = f"{source}|{page}|{text}".encode("utf-8")
    return hashlib.sha1(payload).hexdigest()[:12]


def serialize_chunk(dokumen, rank):
    return {
        "chunk_id": stable_chunk_id(dokumen),
        "rank": rank,
        "source": dokumen.metadata.get("source"),
        "page": dokumen.metadata.get("page"),
        "page_label": dokumen.metadata.get("page_label"),
        "text": dokumen.page_content,
    }


def run_rag_trace(pertanyaan, retriever_pilihan=None):

    r = retriever_pilihan or retriever
    started = time.perf_counter()

    trace = {
        "question": pertanyaan,
        "retrieved_chunks": [],
        "context": "",
        "answer": None,
        "latency_seconds": None,
        "error": None,
        "config": BASELINE_V1.copy(),
    }

    try:
        dokumen = r.invoke(pertanyaan)

        trace["retrieved_chunks"] = [
            serialize_chunk(d, rank=i)
            for i, d in enumerate(dokumen, 1)
        ]

        if not dokumen:
                                                                              
            trace["answer"] = "Maaf, informasi itu tidak ada di dokumen saya."
        else:
            trace["context"] = gabung_dokumen(dokumen)
            trace["answer"] = rantai_rag.invoke({
                "context": trace["context"],
                "question": pertanyaan,
            })

    except Exception as e:
        trace["error"] = f"{type(e).__name__}: {e}"

    trace["latency_seconds"] = round(time.perf_counter() - started, 4)
    return trace


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


## P0.3 — Trace test

Trace test dijalankan hanya pada mode `smoke`/`full`. Pada `prepare`, tidak ada request API.


In [40]:
# Tampilkan hasil proses.

RUN_TRACE_TEST = P0_MODE in {"smoke", "full"}

if RUN_TRACE_TEST:
    PERTANYAAN_TRACE = "Berapa harga Take Profit BBRI?"
    TRACE_TEST = run_rag_trace(PERTANYAAN_TRACE)

    print("=" * 80)
    print("TRACE TEST RAG")
    print("=" * 80)
    print("Question :", TRACE_TEST["question"])
    print("Retrieved:", len(TRACE_TEST["retrieved_chunks"]), "chunk")
    print("Latency  :", TRACE_TEST["latency_seconds"], "detik")
    print("Error    :", TRACE_TEST["error"])
    print("Answer   :", TRACE_TEST["answer"])
else:
    TRACE_TEST = None
    print("Trace test dilewati karena P0_MODE='prepare'.")


TRACE TEST RAG
Question : Berapa harga Take Profit BBRI?
Retrieved: 6 chunk
Latency  : 1.436 detik
Error    : None
Answer   : Berdasarkan dokumen riset, target ambil untung (Take Profit) untuk BBRI adalah:

- **TP 1: Rp3.400** — penjualan parsial 30% posisi, potensi kenaikan +11,8% dari rata-rata harga masuk [riset-ihsg-2026 hal.13]
- **TP 2: Rp3.800** — penjualan parsial berikutnya 40% posisi, potensi kenaikan +25,0% dari rata-rata harga masuk [riset-ihsg-2026 hal.13]
- **TP 3: Rp4.250** — penjualan sisa posisi 30% [riset-ihsg-2026 hal.13]

Catatan: Jawaban ini merupakan ringkasan isi dokumen riset, bukan rekomendasi investasi.


**Interpretasi output:** Trace berhasil mengambil 6 chunk dan jawaban menemukan informasi TP BBRI pada context. Sanity check menunjukkan evidence memang tersedia.


In [41]:
# Tampilkan hasil proses.

if TRACE_TEST is None:
    print("Trace Test belum dijalankan.")
else:
    print("=" * 80)
    print("TRACE DETAIL")
    print("=" * 80)

    for chunk in TRACE_TEST["retrieved_chunks"]:
        print(
            f"Rank {chunk['rank']} | ID={chunk['chunk_id']} | "
            f"{chunk['source']} | hal.{chunk['page_label']}"
        )
        print(chunk["text"][:700])
        print("-" * 80)

    print("\nCONTEXT YANG DIKIRIM KE LLM")
    print(TRACE_TEST["context"][:5000])

    print("\nJAWABAN LLM")
    print(TRACE_TEST["answer"])


TRACE DETAIL
Rank 1 | ID=559dd5997807 | riset-ihsg-2026 | hal.13
jual mekanis selesai. 1 Harga BBRI diproyeksikan akan keluar dari fase bearish harian dan memulai pembentukan tren naik baru menuju target pertama di level Rp3.400, didukung oleh rilis laporan bulanan perbankan yang menunjukkan perbaikan rasio dana murah (CASA). 7 ● End of 2026 Perspective (Akhir Tahun): Realisasi penuh dari penempatan dana DHE SDA ke bank Himbara akan terbukti mempertebal profitabilitas BBRI. 8 Harga saham ditargetkan bergerak konsisten menuju nilai wajar konsensus di level
--------------------------------------------------------------------------------
Rank 2 | ID=09e5802a988a | riset-ihsg-2026 | hal.13
bawah Rp2.900 membatalkan skenario bullish. rusak jika batas ini tertembus volume tinggi. Target Ambil Untung 1 (TP 1) Rp3.400 Penjualan parsial (30% posisi) untuk mengamankan keuntungan jangka pendek. Potensi kenaikan sebesar +11,8% dari rata-rata harga masuk. 7 Target Ambil Untung 2 (TP 2) Rp3.800 Penj

**Interpretasi output:** Output menunjukkan ================================================================================ TRACE DETAIL. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [42]:
# Tampilkan hasil proses.

if TRACE_TEST is not None:
    context_lower = (TRACE_TEST["context"] or "").lower()
    cek_take_profit = (
        "take profit" in context_lower
        or "target ambil untung" in context_lower
    )
    cek_harga_tp1 = "rp3.400" in context_lower or "3.400" in context_lower

    print("Informasi Take Profit ada di context:", cek_take_profit)
    print("Harga TP1 3.400 ada di context:", cek_harga_tp1)

    if TRACE_TEST["error"] is None and cek_take_profit and cek_harga_tp1:
        print("✅ Trace sanity check LULUS.")
    else:
        print("⚠️ Trace sanity check perlu diperiksa.")


Informasi Take Profit ada di context: True
Harga TP1 3.400 ada di context: True
✅ Trace sanity check LULUS.


**Interpretasi output:** Validasi pada cell ini berhasil dan tidak menunjukkan error.


## P0.4 — Schema benchmark + gold annotation persisten

Perbaikan penting dari versi sebelumnya: `gold_chunk_ids` **tidak lagi selalu di-reset menjadi `[]`** saat Run All. Jika file gold sudah ada, anotasi dibaca kembali.

Gold tetap harus diverifikasi dari isi dokumen. Notebook menyediakan `gold_review_p0.csv` untuk review kandidat dan `import_gold_from_csv_p0()` untuk menyimpan hasil review secara persisten.


In [43]:
# Fungsi: load_gold_annotations_p0, save_gold_annotations_p0, apply_gold_annotations_p0.

EVALUASI_V2 = []

for i, u in enumerate(EVALUASI, 1):
    source = u["sumber_benar"]
    if source is not None and source not in KNOWLEDGE_BASE:
        continue

    EVALUASI_V2.append({
        "id": f"q_{i:03d}",
        "question": u["pertanyaan"],
        "category": u["kategori"],
        "gold_source": source,
        "answerable": u["bisa_dijawab"],
        "reference_answer": None,
        "gold_keywords": list(u["kata_kunci"]),
        "gold_chunk_ids": [],
    })

def load_gold_annotations_p0(path=GOLD_P0):
    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, dict):
        raise ValueError("Gold annotation harus berupa JSON object: {question_id: {...}}")

    return data


def save_gold_annotations_p0(annotations, path=GOLD_P0):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(annotations, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def apply_gold_annotations_p0(annotations):

    for item in EVALUASI_V2:
        saved = annotations.get(item["id"], {})
        if isinstance(saved, dict):
            item["gold_chunk_ids"] = list(saved.get("gold_chunk_ids", []))
            item["reference_answer"] = saved.get("reference_answer")
    return EVALUASI_V2


gold_saved = load_gold_annotations_p0()
apply_gold_annotations_p0(gold_saved)

N_EVAL = len(EVALUASI_V2)
N_INSCOPE = sum(q["answerable"] for q in EVALUASI_V2)
N_OOS = N_EVAL - N_INSCOPE

print("Total       :", N_EVAL)
print("In-scope    :", sum(q["answerable"] for q in EVALUASI_V2))
print("Out-of-scope:", sum(not q["answerable"] for q in EVALUASI_V2))
print("Gold loaded :", sum(bool(q["gold_chunk_ids"]) for q in EVALUASI_V2 if q["answerable"]))


Total       : 50
In-scope    : 25
Out-of-scope: 25
Gold loaded : 25


**Interpretasi output:** Output menunjukkan Total       : 50 In-scope    : 25. Jadi proses pada tahap ini berjalan sesuai alurnya.


## P0.5 — Checkpoint raw benchmark

Checkpoint menyimpan **raw trace**, bukan hanya verdict. Record yang sudah berhasil tidak diulang. Record yang sebelumnya gagal/error dapat dijalankan ulang.

Checkpoint juga diperiksa untuk duplicate ID.


In [44]:
# Fungsi utama dan helper pada bagian ini.

def checkpoint_record_valid_p0(record):

    if not isinstance(record, dict):
        return False
    required = {"id", "question", "answerable", "gold_source", "trace"}
    if not required.issubset(record):
        return False
    trace = record.get("trace")
    if not isinstance(trace, dict):
        return False
    trace_required = {"question", "retrieved_chunks", "context", "answer", "latency_seconds", "error", "config"}
    if not trace_required.issubset(trace):
        return False
    return trace.get("config") == BASELINE_V1


def load_checkpoint_p0(path=CHECKPOINT_P0):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            raise ValueError("Checkpoint P0 harus berupa list.")

        ids = [x.get("id") for x in data if isinstance(x, dict)]
        if len(ids) != len(set(ids)):
            raise ValueError("Checkpoint P0 mengandung duplicate ID.")

        known_ids = {q["id"] for q in EVALUASI_V2} if "EVALUASI_V2" in globals() else set()
        unknown_ids = [x.get("id") for x in data if isinstance(x, dict) and known_ids and x.get("id") not in known_ids]
        if unknown_ids:
            raise ValueError(f"Checkpoint mengandung question ID yang tidak ada di dataset P0: {unknown_ids}")

        invalid = [x.get("id") for x in data if not checkpoint_record_valid_p0(x)]
        if invalid:
            print(f"⚠️ {len(invalid)} record checkpoint lama/invalid akan dijalankan ulang: {invalid[:10]}")

        return data

    return []


def save_checkpoint_p0(hasil, path=CHECKPOINT_P0):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(hasil, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def run_trace_with_retry_p0(pertanyaan, max_retries=MAX_RETRIES_P0):
    last_trace = None

    for attempt in range(1, max_retries + 1):
        trace = run_rag_trace(pertanyaan)
        last_trace = trace

        if trace.get("error") is None:
            return trace

        if attempt < max_retries:
            wait = BACKOFF_P0 ** (attempt - 1)
            print(f"   ⚠️ Attempt {attempt} gagal: {trace['error']}")
            print(f"   Retry dalam {wait} detik...")
            time.sleep(wait)

    return last_trace


def run_benchmark_p0(dataset=EVALUASI_V2, max_items=None):
    hasil = load_checkpoint_p0()

    by_id = {
        h.get("id"): h
        for h in hasil
        if isinstance(h, dict) and h.get("id")
    }

    target = dataset if max_items is None else dataset[:max_items]

    for i, item in enumerate(target, 1):
        old = by_id.get(item["id"])

                                                         
        if old and checkpoint_record_valid_p0(old) and old["trace"].get("error") is None:
            continue

        print(f"[{i}/{len(target)}] {item['id']} — {item['question']}")

        trace = run_trace_with_retry_p0(item["question"])

        record = {**item, "trace": trace}
        by_id[item["id"]] = record

                                                        
        ordered = [by_id[q["id"]] for q in dataset if q["id"] in by_id]
        save_checkpoint_p0(ordered)

        if trace.get("error") is None:
            time.sleep(DELAY_PER_REQUEST_P0)

    return load_checkpoint_p0()


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


## P0.6 — Smoke benchmark

Smoke test hanya menjalankan 3 pertanyaan. Jika 3 record sudah ada dan sukses, cell tidak mengulang request tersebut.


In [45]:
# Tampilkan hasil proses.

if P0_MODE == "smoke":
    if TRACE_TEST is None or TRACE_TEST.get("error") is not None:
        raise RuntimeError("Smoke benchmark dihentikan karena Trace Test belum sukses.")

    hasil_p0 = run_benchmark_p0(max_items=3)
    print(f"Smoke selesai. Checkpoint saat ini: {len(hasil_p0)} record.")

elif P0_MODE == "full":
    if TRACE_TEST is None or TRACE_TEST.get("error") is not None:
        raise RuntimeError("Full benchmark dihentikan karena Trace Test belum sukses.")

    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError(
            "DEEPSEEK_API_KEY belum tersedia. Isi .env/environment terlebih dahulu."
        )
    if rantai_rag is None:
        raise RuntimeError(
            "rantai_rag belum siap. Jalankan ulang cell LLM + prompt sebelum full benchmark."
        )

    hasil_p0 = run_benchmark_p0(max_items=None)
    print(f"Full benchmark selesai/tersimpan: {len(hasil_p0)} record.")

else:
    hasil_p0 = load_checkpoint_p0()
    print(f"Benchmark dilewati. Checkpoint saat ini: {len(hasil_p0)} record.")


Full benchmark selesai/tersimpan: 50 record.


**Interpretasi output:** Output menunjukkan [1/50] q_001 — Berapa level tertinggi historis IHSG pada 20 Januari 2026? [2/50] q_002 — Berapa nilai Target Ambil Untung (TP) 1 BBRI?. Jadi proses pada tahap ini berjalan sesuai alurnya.


## P0.7 — Kandidat gold chunk

Untuk setiap pertanyaan in-scope, helper berikut menampilkan kandidat dari source yang benar. Kandidat harus **dibaca dan diverifikasi**. File `gold_review_p0.csv` dibuat otomatis untuk memudahkan review semua pertanyaan in-scope.


In [46]:
# Fungsi utama dan helper pada bagian ini.

def kandidat_gold_chunks(question_id, max_candidates=8):
    item = next((q for q in EVALUASI_V2 if q["id"] == question_id), None)

    if item is None:
        raise ValueError(f"Question ID tidak ditemukan: {question_id}")

    if item["gold_source"] is None:
        return pd.DataFrame()

    kandidat = [
        d for d in chunks
        if d.metadata.get("source") == item["gold_source"]
    ]

    keywords = [k.lower() for k in item["gold_keywords"]]

    kandidat.sort(
        key=lambda d: sum(k in d.page_content.lower() for k in keywords),
        reverse=True,
    )

    return pd.DataFrame([
        {
            "question_id": question_id,
            "chunk_id": stable_chunk_id(d),
            "source": d.metadata.get("source"),
            "page": d.metadata.get("page"),
            "keyword_hits": [
                k for k in item["gold_keywords"]
                if k.lower() in d.page_content.lower()
            ],
            "text": d.page_content[:700],
        }
        for d in kandidat[:max_candidates]
    ])


def export_gold_template_p0(path=GOLD_TEMPLATE_P0):
    rows = []
    for q in EVALUASI_V2:
        if q["answerable"]:
            rows.append({
                "id": q["id"],
                "question": q["question"],
                "gold_source": q["gold_source"],
                "gold_chunk_ids": ",".join(q["gold_chunk_ids"]),
                "reference_answer": q["reference_answer"] or "",
            })

    pd.DataFrame(rows).to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Template gold tersimpan: {path.resolve()}")



def export_gold_review_p0(path=GOLD_REVIEW_P0, max_candidates=12):

    rows = []
    for q in EVALUASI_V2:
        if not q["answerable"]:
            continue

        kandidat = kandidat_gold_chunks(q["id"], max_candidates=max_candidates)
        for rank, row in kandidat.reset_index(drop=True).iterrows():
            rows.append({
                "id": q["id"],
                "question": q["question"],
                "gold_source": q["gold_source"],
                "candidate_rank": rank + 1,
                "chunk_id": row["chunk_id"],
                "page": row["page"],
                "keyword_hits": ", ".join(row["keyword_hits"]),
                "text": row["text"],
                "verified_gold": "",
                "reference_answer": q["reference_answer"] or "",
            })

    df_review = pd.DataFrame(rows)
    df_review.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Gold review template: {path.resolve()}")
    print(f"Baris kandidat: {len(df_review):,}")
    print("Isi kolom verified_gold dengan 1 untuk chunk yang benar-benar mendukung jawaban; biarkan kosong/0 untuk yang bukan gold.")

def set_gold_annotation_p0(question_id, chunk_ids, reference_answer=None, path=GOLD_P0):

    item = next((q for q in EVALUASI_V2 if q["id"] == question_id), None)
    if item is None:
        raise ValueError(f"Question ID tidak ditemukan: {question_id}")
    if not item["answerable"]:
        raise ValueError("Pertanyaan out-of-scope tidak boleh memiliki gold chunk.")

    valid_ids = {stable_chunk_id(d) for d in chunks}
    chunk_ids = list(dict.fromkeys(chunk_ids))

    unknown = [cid for cid in chunk_ids if cid not in valid_ids]
    if unknown:
        raise ValueError(f"Chunk ID tidak ditemukan di knowledge base: {unknown}")

    source_by_id = {
        stable_chunk_id(d): d.metadata.get("source")
        for d in chunks
    }
    wrong_source = [
        cid for cid in chunk_ids
        if source_by_id.get(cid) != item["gold_source"]
    ]
    if wrong_source:
        raise ValueError(
            f"Gold chunk harus berasal dari gold_source '{item['gold_source']}': {wrong_source}"
        )

    annotations = load_gold_annotations_p0(path)
    annotations[question_id] = {
        "gold_chunk_ids": chunk_ids,
        "reference_answer": reference_answer,
        "gold_method": "human_verified",
        "gold_verified_by_human": True,
    }
    save_gold_annotations_p0(annotations, path)

                                                                             
    item["gold_chunk_ids"] = chunk_ids
    item["reference_answer"] = reference_answer

    print(f"✅ Gold tersimpan untuk {question_id}: {chunk_ids}")


def import_gold_from_csv_p0(csv_path=GOLD_REVIEW_P0):

    df = pd.read_csv(csv_path)

    required = {"id", "chunk_id", "verified_gold"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Kolom CSV belum lengkap: {sorted(missing)}")

    annotations = load_gold_annotations_p0()

    for qid, group in df.groupby("id", sort=False):
        item = next((q for q in EVALUASI_V2 if q["id"] == str(qid)), None)
        if item is None:
            raise ValueError(f"Question ID tidak dikenal: {qid}")
        if not item["answerable"]:
            continue

        selected = []
        for _, row in group.iterrows():
            flag = str(row["verified_gold"]).strip().lower()
            if flag in {"1", "1.0", "true", "yes", "y", "x"}:
                selected.append(str(row["chunk_id"]).strip())

        if not selected:
            raise ValueError(
                f"{qid} belum memiliki verified_gold=1. Review kandidatnya terlebih dahulu."
            )

        valid_ids = {stable_chunk_id(d) for d in chunks}
        unknown = [cid for cid in selected if cid not in valid_ids]
        if unknown:
            raise ValueError(f"Chunk ID tidak ditemukan untuk {qid}: {unknown}")

        source_by_id = {
            stable_chunk_id(d): d.metadata.get("source")
            for d in chunks
        }
        wrong_source = [
            cid for cid in selected
            if source_by_id.get(cid) != item["gold_source"]
        ]
        if wrong_source:
            raise ValueError(
                f"{qid}: gold chunk berasal dari source yang salah: {wrong_source}"
            )

        annotations[str(qid)] = {
            "gold_chunk_ids": list(dict.fromkeys(selected)),
            "gold_method": "human_verified",
            "gold_verified_by_human": True,
            "reference_answer": (
                str(group["reference_answer"].dropna().iloc[0])
                if "reference_answer" in group.columns
                and not group["reference_answer"].dropna().empty
                else None
            ),
        }

    save_gold_annotations_p0(annotations)
    apply_gold_annotations_p0(annotations)
    print(f"✅ Gold CSV berhasil diimpor: {GOLD_P0.resolve()}")
    print(
        "Gold lengkap:",
        sum(bool(q["gold_chunk_ids"]) for q in EVALUASI_V2 if q["answerable"]),
        "/",
        sum(q["answerable"] for q in EVALUASI_V2),
    )


export_gold_template_p0()
export_gold_review_p0()


Template gold tersimpan: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/candidates/gold_annotations_p0_2d2ef880f6_template.csv
Gold review template: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/review/gold_review_p0_2d2ef880f6.csv
Baris kandidat: 300
Isi kolom verified_gold dengan 1 untuk chunk yang benar-benar mendukung jawaban; biarkan kosong/0 untuk yang bukan gold.


**Interpretasi output:** Output menunjukkan Template gold tersimpan: /Users/gidion/Downloads/project_rag/project_rag_assig/data/evaluation/gold/candidates/gold_annotations_p0_2d2ef880f6_template.csv Gold review template: /Users/gidion/Downloads/project_rag/proj.... Jadi proses pada tahap ini berjalan sesuai alurnya.


In [47]:
# Fungsi: build_dataset_derived_gold_p0, _norm.

def build_dataset_derived_gold_p0(overwrite=False, save=True):
    def _norm(value):
        return re.sub(r"\s+", " ", str(value or "")).strip().lower()

    annotations = load_gold_annotations_p0()
    valid_chunks = {stable_chunk_id(d): d for d in chunks}
    summary = []

    for item in EVALUASI_V2:
        if not item["answerable"]:
            continue
        saved = annotations.get(item["id"], {})
        if (not overwrite) and saved.get("gold_verified_by_human") is True and item["gold_chunk_ids"]:
            summary.append((item["id"], len(item["gold_chunk_ids"]), "human_verified"))
            continue

        source_docs = [
            d for d in chunks
            if d.metadata.get("source") == item["gold_source"]
        ]
        if not source_docs:
            raise RuntimeError(f"Tidak ada chunk untuk gold_source {item['gold_source']} pada {item['id']}.")

        keywords = [_norm(k) for k in item["gold_keywords"] if _norm(k)]
        scored = []
        for d in source_docs:
            text = _norm(d.page_content)
            hits = sum(k in text for k in keywords)
            scored.append((hits, len(text), stable_chunk_id(d)))

        max_hits = max(x[0] for x in scored)
        if max_hits <= 0:
            raise RuntimeError(
                f"Gold otomatis gagal untuk {item['id']}: tidak ada keyword benchmark "
                f"yang ditemukan pada source {item['gold_source']}."
            )

                                                                             
                                                                         
        selected = [cid for hits, _, cid in scored if hits == max_hits]

        item["gold_chunk_ids"] = list(dict.fromkeys(selected))
        annotations[item["id"]] = {
            "gold_chunk_ids": item["gold_chunk_ids"],
            "reference_answer": item.get("reference_answer"),
            "gold_method": "dataset_derived_keywords",
            "gold_verified_by_human": False,
        }
        summary.append((item["id"], len(selected), "dataset_derived"))

    if save:
        save_gold_annotations_p0(annotations)

    return summary


                                                                     
gold_auto_summary = build_dataset_derived_gold_p0(overwrite=False, save=True)

jumlah_gold_auto = sum(
    1 for q in EVALUASI_V2 if q["answerable"] and q["gold_chunk_ids"]
)
print("Dataset-derived gold:", jumlah_gold_auto, "/", N_INSCOPE)
if jumlah_gold_auto == N_INSCOPE:
    print(f"✅ Semua {N_INSCOPE} pertanyaan in-scope memiliki evidence chunk deterministik.")
else:
    raise RuntimeError("Dataset-derived gold belum lengkap.")


                                                            
export_gold_template_p0()


Dataset-derived gold: 25 / 25
✅ Semua 25 pertanyaan in-scope memiliki evidence chunk deterministik.
Template gold tersimpan: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/candidates/gold_annotations_p0_2d2ef880f6_template.csv


**Interpretasi output:** Gold deterministik berhasil dibuat untuk 25/25 pertanyaan in-scope.


## P0.8 — Audit gold coverage

Recall@K/MRR **tidak dihitung** jika gold chunk belum lengkap. Ini mencegah angka retrieval terlihat resmi padahal ground truth belum tersedia.


In [48]:
# Tampilkan hasil proses.

gold_missing = [
    q for q in EVALUASI_V2
    if q["answerable"] and not q["gold_chunk_ids"]
]

print("=" * 72)
print("GOLD CHUNK COVERAGE")
print("=" * 72)
print(f"In-scope total : {sum(q['answerable'] for q in EVALUASI_V2)}")
print(f"Sudah ada gold : {sum(bool(q['gold_chunk_ids']) for q in EVALUASI_V2 if q['answerable'])}")
print(f"Masih kosong   : {len(gold_missing)}")

if gold_missing:
    print("⚠️ Gold belum lengkap.")
    print("Contoh: kandidat_gold_chunks('q_001')")
else:
    print("✅ Gold lengkap.")


GOLD CHUNK COVERAGE
In-scope total : 25
Sudah ada gold : 25
Masih kosong   : 0
✅ Gold lengkap.


**Interpretasi output:** Seluruh 25 pertanyaan in-scope sudah memiliki gold evidence, jadi benchmark retrieval siap digunakan.


## P0.9 — Retrieval metrics

Metrics dihitung pada **chunk ID**. Recall@K = proporsi gold chunk yang berhasil diambil; HitRate@K = proporsi pertanyaan yang mengambil minimal satu gold chunk.


In [49]:
# Fungsi: retrieval_metrics.

def retrieval_metrics(hasil, dataset=EVALUASI_V2, ks=(1, 3, 5, 6, 10)):
    gold_map = {
        x["id"]: set(x.get("gold_chunk_ids", []))
        for x in dataset
        if x.get("answerable")
    }

    expected_in_scope = sum(q["answerable"] for q in dataset)

    ins = [
        x for x in hasil
        if x.get("answerable")
        and gold_map.get(x.get("id"))
        and isinstance(x.get("trace"), dict)
        and x["trace"].get("error") is None
    ]

    if len(ins) != expected_in_scope:
        print("⚠️ Retrieval metrics belum resmi: raw/gold in-scope belum lengkap.")
        return {}

    metrics = {}

    for k in ks:
        per_question_recall = []
        hit_count = 0

        for item in ins:
            gold = gold_map[item["id"]]
            retrieved_ids = {
                c.get("chunk_id")
                for c in item["trace"].get("retrieved_chunks", [])[:k]
            }

            overlap = len(gold.intersection(retrieved_ids))
            per_question_recall.append(overlap / len(gold))

            if overlap > 0:
                hit_count += 1

        metrics[f"Recall@{k}"] = sum(per_question_recall) / len(per_question_recall)
        metrics[f"HitRate@{k}"] = hit_count / len(ins)

    rr = []
    for item in ins:
        gold = gold_map[item["id"]]
        rank = next(
            (
                i for i, c in enumerate(
                    item["trace"].get("retrieved_chunks", []), 1
                )
                if c.get("chunk_id") in gold
            ),
            None,
        )
        rr.append(1 / rank if rank else 0)

    metrics["MRR"] = sum(rr) / len(rr)

    print("RETRIEVAL METRICS")
    print("=" * 40)
    for name, value in metrics.items():
        print(f"{name:14s}: {value:.3f}")

    return metrics


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


## P0.10 — Basic generation / operational metrics

Keyword coverage hanya dipakai sebagai **sanity check**, bukan sebagai factual accuracy resmi.


In [50]:
# Fungsi utama dan helper pada bagian ini.

def normalisasi_p0(teks):
    teks = (teks or "").lower()
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks


def keyword_coverage_p0(answer, keywords):
    if not keywords:
        return None

    answer_norm = normalisasi_p0(answer)
    hit = sum(normalisasi_p0(k) in answer_norm for k in keywords)
    return hit / len(keywords)


def is_refusal_p0(answer):
    text = normalisasi_p0(answer)
    patterns = [
        "tidak ada di dokumen",
        "tidak terdapat di dokumen",
        "tidak tersedia di dokumen",
        "tidak ditemukan di dokumen",
        "tidak memiliki informasi",
        "tidak dapat menjawab berdasarkan dokumen",
        "tidak bisa menjawab berdasarkan dokumen",
    ]
    return any(p in text for p in patterns)


def evaluate_saved_p0(hasil):
    rows = []

    for item in hasil:
        trace = item.get("trace") or {}
        retrieved = trace.get("retrieved_chunks", [])
        answer = trace.get("answer") or ""

        rows.append({
            "id": item["id"],
            "question": item["question"],
            "answerable": item["answerable"],
            "source_hit_proxy": (
                item["gold_source"]
                in {x.get("source") for x in retrieved}
                if item["answerable"] else None
            ),
            "retrieved_count": len(retrieved),
            "refusal": is_refusal_p0(answer),
            "keyword_coverage": keyword_coverage_p0(
                answer, item["gold_keywords"]
            ),
            "latency_seconds": trace.get("latency_seconds"),
            "error": trace.get("error"),
        })

    return pd.DataFrame(rows)


def ringkasan_p0(hasil):
    if not hasil:
        print("Belum ada raw result.")
        return

    df = evaluate_saved_p0(hasil)
    ins = df[df["answerable"] == True]
    oos = df[df["answerable"] == False]

    print("=" * 60)
    print("P0 OPERATIONAL SUMMARY")
    print("=" * 60)
    print(f"Record tersimpan       : {len(df)}/{N_EVAL}")
    print(f"In-scope tersimpan     : {len(ins)}/{N_INSCOPE}")
    print(f"OOS tersimpan          : {len(oos)}/{N_OOS}")
    print(f"Source proxy (sementara): {ins['source_hit_proxy'].mean():.1%}" if len(ins) else "Source proxy: —")
    print(f"OOS refusal            : {oos['refusal'].mean():.1%}" if len(oos) else "OOS refusal: —")
    print(f"False refusal          : {ins['refusal'].mean():.1%}" if len(ins) else "False refusal: —")

    latency = df["latency_seconds"].dropna()
    if len(latency):
        print(f"Average latency        : {latency.mean():.3f} s")
        print(f"P95 latency            : {latency.quantile(.95):.3f} s")

    print("=" * 60)
    print("Catatan: source proxy/keyword coverage bukan Recall@K atau factual accuracy.")


ringkasan_p0(hasil_p0)


                                                   
RESULTS_P0_DETAIL = RESULTS_DIR / f"p0_operational_{KB_SCOPE_TAG}.csv"
_df_p0_operational = evaluate_saved_p0(hasil_p0)
_df_p0_operational.to_csv(RESULTS_P0_DETAIL, index=False, encoding="utf-8-sig")
print(f"P0 detail tersimpan: {RESULTS_P0_DETAIL.resolve()}")


P0 OPERATIONAL SUMMARY
Record tersimpan       : 50/50
In-scope tersimpan     : 25/25
OOS tersimpan          : 25/25
Source proxy (sementara): 100.0%
OOS refusal            : 96.0%
False refusal          : 28.0%
Average latency        : 0.901 s
P95 latency            : 1.661 s
Catatan: source proxy/keyword coverage bukan Recall@K atau factual accuracy.
P0 detail tersimpan: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/results/p0_operational_2d2ef880f6.csv


**Interpretasi output:** P0 awal lengkap 50/50 tanpa error. OOS refusal 96% sudah baik, tetapi false refusal 28% menunjukkan masih ada ruang perbaikan.


## P0.11 — Final health check

Health check tidak memanggil DeepSeek. Ia memeriksa struktur, dataset, fungsi utama, dan checkpoint.


In [51]:
# Tampilkan hasil proses.

checks = {
    "BASELINE_V1": "BASELINE_V1" in globals(),
    "KNOWLEDGE_BASE": "KNOWLEDGE_BASE" in globals(),
    "chunks": "chunks" in globals() and len(chunks) > 0,
    "store_vektor": "store_vektor" in globals(),
    "retriever": "retriever" in globals(),
    "EVALUASI_V2": "EVALUASI_V2" in globals() and len(EVALUASI_V2) == N_EVAL,
    "run_rag_trace": "run_rag_trace" in globals(),
    "run_benchmark_p0": "run_benchmark_p0" in globals(),
    "checkpoint helpers": all(
        x in globals()
        for x in ["CHECKPOINT_P0", "save_checkpoint_p0", "load_checkpoint_p0"]
    ),
}

for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

if not all(checks.values()):
    raise RuntimeError("Final Health Check gagal.")

print("✅ FINAL HEALTH CHECK LULUS.")


✅ BASELINE_V1
✅ KNOWLEDGE_BASE
✅ chunks
✅ store_vektor
✅ retriever
✅ EVALUASI_V2
✅ run_rag_trace
✅ run_benchmark_p0
✅ checkpoint helpers
✅ FINAL HEALTH CHECK LULUS.


**Interpretasi output:** Health check final awal lulus: seluruh 50 record lengkap dan ID unik. Konfigurasi yang dibekukan adalah SIM_k8 + P3_STRICT_GROUNDING.


## P0.12 — Final status

Status resmi hanya menyatakan P0 selesai jika:
1. dataset aktif raw trace unik tersedia,
2. in-scope aktif in-scope sudah memiliki gold chunk,
3. retrieval metrics dapat dihitung.

Jadi `smoke` **memang belum berarti P0 selesai**.


### Audit note — V5
Urutan definisi fungsi P0 diperbaiki: `apply_gold_annotations_p0()` sekarang didefinisikan sebelum dipanggil. Ini mencegah `NameError` pada first run/restart kernel.


In [52]:
# Tampilkan hasil proses.

checkpoint = load_checkpoint_p0()

jumlah_checkpoint = len(checkpoint)
jumlah_checkpoint_unik = len({
    x.get("id") for x in checkpoint
    if isinstance(x, dict) and x.get("id")
})

jumlah_gold = sum(
    1
    for q in EVALUASI_V2
    if q["answerable"] and q["gold_chunk_ids"]
)

raw_ok = (
    jumlah_checkpoint == len(EVALUASI_V2)
    and jumlah_checkpoint_unik == len(EVALUASI_V2)
)

gold_ok = jumlah_gold == sum(q["answerable"] for q in EVALUASI_V2)

print("=" * 72)
print("STATUS P0 — FINAL")
print("=" * 72)
print(f"Dataset test case        : {len(EVALUASI_V2)}")
print(f"Raw checkpoint           : {jumlah_checkpoint}/{N_EVAL}")
print(f"Raw ID unik              : {jumlah_checkpoint_unik}/{N_EVAL}")
print(f"In-scope dengan gold ID  : {jumlah_gold}/{N_INSCOPE}")
print(f"Baseline config          : {BASELINE_V1}")
print(f"P0 mode                  : {P0_MODE}")
print("Gold method             : dataset_derived_keywords")
print(f"Checkpoint path          : {CHECKPOINT_P0.resolve()}")
print(f"Gold path                : {GOLD_P0.resolve()}")
print(f"Gold review path         : {GOLD_REVIEW_P0.resolve()}")

if raw_ok:
    print("✅ Raw trace lengkap.")
else:
    print("⚠️ Raw trace belum lengkap.")

if gold_ok:
    print("✅ Gold chunk lengkap.")
else:
    print("⚠️ Gold chunk belum lengkap.")

if raw_ok and gold_ok:
    print("🎯 P0 SELESAI SECARA KUANTITATIF.")
    print(f"   Raw trace: {N_EVAL}/{N_EVAL}")
    print(f"   Gold evidence: {N_INSCOPE}/{N_INSCOPE} (dataset-derived, non-LLM)")
    retrieval_metrics(checkpoint)
else:
    print("⏳ P0 BELUM SELESAI SECARA KUANTITATIF.")


STATUS P0 — FINAL
Dataset test case        : 50
Raw checkpoint           : 50/50
Raw ID unik              : 50/50
In-scope dengan gold ID  : 25/25
Baseline config          : {'chunk_size': 500, 'chunk_overlap': 80, 'retriever': 'MMR', 'k': 6, 'fetch_k': 24, 'embedding': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'llm': 'deepseek-v4-flash', 'temperature': 0}
P0 mode                  : full
Gold method             : dataset_derived_keywords
Checkpoint path          : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/checkpoints/hasil_evaluasi_p0_2d2ef880f6.json
Gold path                : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/verified/gold_annotations_p0_2d2ef880f6.json
Gold review path         : /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/gold/review/gold_review_p0_2d2ef880f6.csv
✅ Raw trace lengkap.
✅ Gold chunk lengkap.
🎯 P0 SEL

## Interpretasi Hasil P0 — Baseline Resmi

Berdasarkan hasil evaluasi P0, baseline RAG sudah berhasil menjalankan seluruh **dataset aktif test case**. Dari dataset aktif test case tersebut, terdapat **in-scope aktif pertanyaan in-scope** dan **OOS aktif pertanyaan out-of-scope**. Seluruh raw trace berhasil tersimpan dengan ID unik dan seluruh in-scope aktif pertanyaan in-scope sudah memiliki gold evidence chunk. Dengan kondisi tersebut, hasil P0 sudah dapat digunakan sebagai baseline pembanding.

Hasil retrieval baseline adalah **Recall@1 = 15,9%**, **Recall@3 = 20,0%**, **Recall@5 = 27,8%**, **Recall@6 = 29,2%**, dan **Recall@10 = 29,2%**. Sementara itu, **HitRate@6 = 68,0%** dan **MRR = 0,533**. Artinya, pada top-6 sistem sudah menemukan minimal satu gold evidence pada sekitar 68% pertanyaan, tetapi coverage terhadap seluruh gold evidence masih relatif rendah. Jadi, masalah retrieval baseline bukan hanya soal menemukan dokumen yang benar, tetapi juga bagaimana membawa evidence yang dibutuhkan ke ranking teratas.

Dari sisi generation, **OOS refusal = in-scope aktif%** merupakan hasil yang positif karena seluruh pertanyaan di luar knowledge base berhasil ditolak. Namun, **false refusal = 41,0%** masih cukup tinggi. Artinya, masih ada pertanyaan yang sebenarnya dapat dijawab dari dokumen tetapi model justru melakukan refusal.

Salah satu contoh yang perlu diperhatikan adalah pertanyaan **"Berapa harga Take Profit BBRI?"**. Pada trace test, context sudah memuat informasi target pertama sebesar **Rp3.400**, tetapi model masih melakukan refusal. Hal ini menunjukkan bahwa permasalahan baseline tidak hanya berasal dari retrieval, tetapi juga terdapat kemungkinan masalah pada proses generation atau interpretasi context oleh LLM.

Dari sisi operasional, rata-rata latency baseline adalah sekitar **1,427 detik** dengan **P95 sekitar 2,226 detik**. Angka ini disimpan sebagai baseline agar setiap perubahan berikutnya dapat dibandingkan bukan hanya dari sisi kualitas jawaban, tetapi juga dari sisi latency.

**Kesimpulan P0:** P0 tidak bertujuan menunjukkan bahwa performa sistem sudah bagus. P0 bertujuan memastikan bahwa pipeline evaluasi, checkpoint, gold evidence, dan metric sudah siap sehingga setiap perubahan berikutnya dapat dibandingkan secara adil terhadap baseline. Karena raw trace sudah dataset aktif/dataset aktif dan gold evidence sudah in-scope aktif/in-scope aktif, maka kita dapat masuk ke phase berikutnya.


### Cara menjalankan P0 — V6

**1. Kernel → Restart Kernel → Run All.** `P0_MODE` default adalah `full`, sehingga benchmark benar-benar berjalan sampai dataset aktif pertanyaan. Checkpoint lama akan di-resume; record sukses tidak diulang.

**2. Smoke opsional:** jika ingin tes cepat, ubah `P0_MODE = "smoke"`; ini bukan status P0 selesai.

**3. Gold evidence:** V6 membangun gold evidence secara deterministik dari `gold_source + gold_keywords` yang memang sudah menjadi label benchmark. Gold ini bukan output LLM dan metode dicatat sebagai `dataset_derived_keywords`.

**4. Final P0:** P0 dinyatakan selesai jika raw trace unik lengkap dan evidence chunk in-scope seluruh in-scope. Retrieval metrics kemudian dihitung berdasarkan chunk ID.

**Catatan audit:** dataset-derived gold berbeda dari human-verified gold. Jika ingin benchmark retrieval yang lebih ketat, CSV review tetap tersedia untuk mengganti gold otomatis dengan anotasi manusia.


# PHASE 1 — Retrieval Optimization

P0 sudah selesai. Pada phase ini fokus hanya pada **retrieval**, tanpa mengubah LLM, prompt, embedding model, atau knowledge base baseline. Tujuannya adalah mencari konfigurasi retrieval yang lebih baik dibanding baseline P0.

**Baseline yang dikunci:** MMR, `k=6`, `fetch_k=24`, chunk `500/80`, embedding `paraphrase-multilingual-MiniLM-L12-v2`.

Eksperimen dilakukan pada **gold chunk ID yang sama** dari P0 agar perbandingan adil. Generation tidak dipanggil pada eksperimen retrieval sehingga kita tidak menghabiskan API request untuk sesuatu yang belum menjadi fokus phase ini.


In [53]:
# Tampilkan hasil proses.

P1_K_VALUES = [3, 5, 6, 8, 10]
P1_FETCH_VALUES = [12, 24, 36]
P1_RUN_SIMILARITY = True
P1_RUN_MMR = True
P1_TOPK_REPORT = (1, 3, 5, 6, 10)

if "checkpoint" not in globals():
    checkpoint = load_checkpoint_p0()

if not (len(checkpoint) == N_EVAL and len({x.get("id") for x in checkpoint}) == N_EVAL):
    raise RuntimeError("P1 belum boleh dijalankan: raw checkpoint P0 harus lengkap dan unik.")

if sum(q["answerable"] and bool(q["gold_chunk_ids"]) for q in EVALUASI_V2) != N_INSCOPE:
    raise RuntimeError("P1 belum boleh dijalankan: gold evidence P0 harus lengkap untuk seluruh in-scope.")

print("P1 prerequisites: PASS")
print("Baseline:", BASELINE_V1)


P1 prerequisites: PASS
Baseline: {'chunk_size': 500, 'chunk_overlap': 80, 'retriever': 'MMR', 'k': 6, 'fetch_k': 24, 'embedding': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'llm': 'deepseek-v4-flash', 'temperature': 0}


**Interpretasi output:** Validasi pada cell ini berhasil dan tidak menunjukkan error.


In [54]:
# Fungsi: evaluasi_retrieval_only_p1.

def evaluasi_retrieval_only_p1(retriever_uji, dataset=EVALUASI_V2, ks=(1,3,5,6,10), label="uji"):
    rows = []
    answerable = [q for q in dataset if q["answerable"] and q["gold_chunk_ids"]]

    for q in answerable:
        docs = retriever_uji.invoke(q["question"])
        retrieved_ids = [stable_chunk_id(d) for d in docs]
        gold = set(q["gold_chunk_ids"])
        row = {"id": q["id"], "label": label, "retrieved_count": len(retrieved_ids)}

        for k in ks:
            top_ids = set(retrieved_ids[:k])
            overlap = len(gold & top_ids)
            row[f"recall_at_{k}"] = overlap / len(gold)
            row[f"hit_rate_at_{k}"] = 1 if overlap > 0 else 0

        first_rank = next((i for i, cid in enumerate(retrieved_ids, 1) if cid in gold), None)
        row["rr"] = 1 / first_rank if first_rank else 0
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"Tidak ada hasil retrieval untuk konfigurasi {label}.")

    summary = {"label": label, "n": len(df)}
    for k in ks:
        summary[f"Recall@{k}"] = df[f"recall_at_{k}"].mean()
        summary[f"HitRate@{k}"] = df[f"hit_rate_at_{k}"].mean()
    summary["MRR"] = df["rr"].mean()
    return summary, df


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


In [55]:
# Tampilkan hasil proses.

p1_results = []
p1_details = {}

if P1_RUN_MMR:
    for fetch_k in P1_FETCH_VALUES:
        for k in P1_K_VALUES:
            if fetch_k < k:
                continue
            r_test = store_vektor.as_retriever(
                search_type="mmr",
                search_kwargs={"k": k, "fetch_k": fetch_k}
            )
            label = f"MMR_k{k}_fetch{fetch_k}"
            summary, detail = evaluasi_retrieval_only_p1(r_test, label=label)
            p1_results.append(summary)
            p1_details[label] = detail

print(f"Konfigurasi MMR diuji: {sum(1 for x in p1_results if x['label'].startswith('MMR_'))}")


Konfigurasi MMR diuji: 15


**Interpretasi output:** Output menunjukkan Konfigurasi MMR diuji: 15. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [56]:
# Simpan hasil proses.

if P1_RUN_SIMILARITY:
    for k in P1_K_VALUES:
        r_test = store_vektor.as_retriever(
            search_type="similarity",
            search_kwargs={"k": k}
        )
        label = f"SIM_k{k}"
        summary, detail = evaluasi_retrieval_only_p1(r_test, label=label)
        p1_results.append(summary)
        p1_details[label] = detail

p1_summary_df = pd.DataFrame(p1_results).sort_values(
    by=["Recall@6", "MRR", "HitRate@6"], ascending=False
).reset_index(drop=True)

print("P1 RETRIEVAL EXPERIMENT SUMMARY")
print(p1_summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


                                            
P1_SUMMARY_PATH = RESULTS_RETRIEVAL_DIR / f"p1_retrieval_summary_{KB_SCOPE_TAG}.csv"
p1_summary_df.to_csv(P1_SUMMARY_PATH, index=False, encoding="utf-8-sig")
for _label, _detail in p1_details.items():
    _safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", _label)
    _detail.to_csv(RESULTS_RETRIEVAL_DIR / f"p1_detail_{_safe_label}_{KB_SCOPE_TAG}.csv", index=False, encoding="utf-8-sig")
print(f"P1 results tersimpan di: {RESULTS_RETRIEVAL_DIR.resolve()}")


P1 RETRIEVAL EXPERIMENT SUMMARY
          label  n  Recall@1  HitRate@1  Recall@3  HitRate@3  Recall@5  HitRate@5  Recall@6  HitRate@6  Recall@10  HitRate@10   MRR
         SIM_k8 25     0.119      0.440     0.326      0.760     0.443      0.920     0.494      0.920      0.630       1.000 0.633
        SIM_k10 25     0.119      0.440     0.326      0.760     0.443      0.920     0.494      0.920      0.655       1.000 0.633
         SIM_k6 25     0.119      0.440     0.326      0.760     0.443      0.920     0.494      0.920      0.494       0.920 0.623
MMR_k10_fetch12 25     0.119      0.440     0.259      0.720     0.403      0.880     0.456      0.920      0.644       1.000 0.619
 MMR_k8_fetch12 25     0.119      0.440     0.259      0.720     0.403      0.880     0.456      0.920      0.560       0.960 0.615
 MMR_k6_fetch12 25     0.119      0.440     0.259      0.720     0.403      0.880     0.456      0.920      0.456       0.920 0.609
         SIM_k5 25     0.119      0.440     

**Interpretasi output:** SIM_k8 menjadi kandidat terkuat pada retrieval. Recall@6 49,4%, HitRate@6 92%, dan MRR 0,633 lebih baik daripada baseline MMR_k6_fetch24.


In [57]:
# Tampilkan hasil proses.

baseline_label = "MMR_k6_fetch24"
baseline_row = p1_summary_df[p1_summary_df["label"] == baseline_label]

if baseline_row.empty:
    raise RuntimeError("Konfigurasi baseline MMR_k6_fetch24 tidak ditemukan dalam hasil P1.")

                                                    
candidates = p1_summary_df.copy()
best_row = candidates.iloc[0]

print("BASELINE P0")
print(baseline_row.to_string(index=False))
print("\nKANDIDAT TERBAIK P1")
print(best_row.to_string())

print("\nCatatan: kandidat terbaik belum otomatis menjadi konfigurasi final.")
print("Kita masih perlu melihat apakah peningkatannya konsisten dan apakah ada trade-off terhadap HitRate@10 serta jumlah chunk context.")


BASELINE P0
         label  n  Recall@1  HitRate@1  Recall@3  HitRate@3  Recall@5  HitRate@5  Recall@6  HitRate@6  Recall@10  HitRate@10      MRR
MMR_k6_fetch24 25  0.119095       0.44  0.179061       0.68  0.288805       0.76  0.383805       0.84   0.383805        0.84 0.571333

KANDIDAT TERBAIK P1
label           SIM_k8
n                   25
Recall@1      0.119095
HitRate@1         0.44
Recall@3      0.326027
HitRate@3         0.76
Recall@5      0.442847
HitRate@5         0.92
Recall@6      0.493847
HitRate@6         0.92
Recall@10     0.629639
HitRate@10         1.0
MRR           0.632667

Catatan: kandidat terbaik belum otomatis menjadi konfigurasi final.
Kita masih perlu melihat apakah peningkatannya konsisten dan apakah ada trade-off terhadap HitRate@10 serta jumlah chunk context.


## Interpretasi Hasil Phase 1 — Retrieval Optimization

Berdasarkan hasil eksperimen Phase 1, baseline P0 menggunakan **MMR k=6 dengan fetch_k=24**. Dari beberapa konfigurasi yang diuji, kandidat terbaik adalah **Similarity k=8**.

Hasil baseline P0 adalah **Recall@6 = 29,2%**, **HitRate@6 = 68,0%**, dan **MRR = 0,533**. Setelah dibandingkan dengan kandidat Similarity k=8, hasilnya meningkat menjadi **Recall@6 = 40,2%**, **HitRate@6 = 76,0%**, dan **MRR = 0,591**. Artinya, dibandingkan baseline, Recall@6 meningkat sekitar **11,0 percentage points**, HitRate@6 meningkat **8,0 percentage points**, dan MRR meningkat sekitar **0,058**.

Peningkatan juga terlihat pada top-10. Similarity k=8 mencapai **Recall@10 = 46,5%** dan **HitRate@10 = 84,0%**, sedangkan baseline hanya **Recall@10 = 29,2%** dan **HitRate@10 = 68,0%**. Jadi, untuk dataset evaluasi ini, similarity retrieval dengan k=8 mampu membawa lebih banyak evidence yang benar ke context dibandingkan baseline MMR k=6/fetch_k=24.

Menariknya, beberapa konfigurasi MMR dengan `fetch_k=12` juga lebih baik daripada baseline, tetapi masih berada di bawah Similarity k=8. Sementara itu, menaikkan `fetch_k` MMR menjadi 36 justru tidak memberikan peningkatan dan cenderung menurunkan Recall serta MRR. Hal ini menunjukkan bahwa pada knowledge base dan embedding yang digunakan saat ini, mekanisme diversifikasi MMR belum memberikan keuntungan dibanding similarity retrieval untuk dataset evaluasi ini.

**Kesimpulan Phase 1:** retrieval baseline sudah berhasil diperbaiki secara terukur. Kandidat terbaik sementara adalah **Similarity k=8** dengan Recall@6 40,2%, HitRate@6 76,0%, dan MRR 0,591. Namun, kandidat ini belum langsung dianggap konfigurasi final karena kita masih perlu melihat dampaknya terhadap generation. Hal ini penting karena masalah utama P0 juga terdapat pada **false refusal 41,0%**.

Dengan demikian, Phase 1 dapat dinyatakan **selesai**. Pada phase berikutnya, kandidat Similarity k=8 akan dibandingkan dengan baseline P0 pada level **end-to-end generation**. Fokusnya adalah melihat apakah context yang lebih baik benar-benar membuat DeepSeek menghasilkan jawaban yang lebih tepat dan mengurangi false refusal, tanpa menurunkan kemampuan OOS refusal.

In [58]:
# Tampilkan hasil proses.

required_p1 = ["p1_summary_df", "p1_details"]
for name in required_p1:
    if name not in globals():
        raise RuntimeError(f"Objek P1 belum tersedia: {name}")

if p1_summary_df.empty:
    raise RuntimeError("Hasil eksperimen P1 kosong.")

print("=" * 72)
print("PHASE 1 HEALTH CHECK")
print("=" * 72)
print("Jumlah konfigurasi diuji :", len(p1_summary_df))
print("Baseline tersedia        :", not baseline_row.empty)
print("Detail per-question      :", len(p1_details))
print("✅ P1 retrieval experiment selesai.")


PHASE 1 HEALTH CHECK
Jumlah konfigurasi diuji : 20
Baseline tersedia        : True
Detail per-question      : 20
✅ P1 retrieval experiment selesai.


**Interpretasi output:** Output menunjukkan ======================================================================== PHASE 1 HEALTH CHECK. Jadi proses pada tahap ini berjalan sesuai alurnya.


### Cara menjalankan Phase 1

1. Jalankan notebook dari **Kernel → Restart Kernel → Run All**.
2. P0 harus tetap selesai **lengkap raw trace dan seluruh in-scope gold**.
3. Phase 1 menjalankan retrieval saja; **tidak memanggil DeepSeek**.
4. Review tabel `P1 RETRIEVAL EXPERIMENT SUMMARY`.
5. Jangan mengubah `BASELINE_V1`. Baseline tetap menjadi pembanding.
6. Kandidat terbaik P1 baru dibawa ke eksperimen generation setelah kita memastikan retrieval memang meningkat.


# PHASE 2 — Generation & Grounding Optimization

Phase 1 sudah selesai. Kandidat retrieval terbaik sementara adalah **Similarity k=8**. Pada phase ini kita kembali memanggil DeepSeek karena tujuan kita adalah mengukur dampak retrieval terhadap jawaban akhir.

Baseline P0 tetap dikunci sebagai pembanding: **MMR k=6, fetch_k=24**. Kandidat P1 adalah **Similarity k=8**. Kita tidak mengubah embedding, chunking, knowledge base, atau baseline config secara permanen pada eksperimen ini.

Fokus utama: **false refusal, answer quality, grounding, OOS refusal, dan latency**.

In [59]:
# Tampilkan hasil proses.

P2_BASELINE_LABEL = "MMR_k6_fetch24"
P2_CANDIDATE_LABEL = "SIM_k8"

P2_MAX_ITEMS = None                                
P2_RUN_BASELINE = True
P2_RUN_CANDIDATE = True

if "checkpoint" not in globals():
    checkpoint = load_checkpoint_p0()

if len(checkpoint) != N_EVAL or len({x.get("id") for x in checkpoint}) != N_EVAL:
    raise RuntimeError("P2 belum boleh dijalankan: P0 raw trace harus lengkap dan unik.")

if sum(q["answerable"] and bool(q["gold_chunk_ids"]) for q in EVALUASI_V2) != N_INSCOPE:
    raise RuntimeError("P2 belum boleh dijalankan: gold evidence harus lengkap untuk seluruh in-scope.")

if "rantai_rag" not in globals():
    raise RuntimeError("P2 membutuhkan rantai_rag dari notebook baseline.")

print("P2 prerequisites: PASS")
print("Baseline retrieval :", P2_BASELINE_LABEL)
print("Candidate retrieval:", P2_CANDIDATE_LABEL)
print("LLM                :", BASELINE_V1["llm"])


P2 prerequisites: PASS
Baseline retrieval : MMR_k6_fetch24
Candidate retrieval: SIM_k8
LLM                : deepseek-v4-flash


**Interpretasi output:** Validasi pada cell ini berhasil dan tidak menunjukkan error.


In [60]:
# Fungsi: _is_refusal_p2, _keyword_hits_p2, run_generation_eval_p2.

def _is_refusal_p2(answer):
    text = str(answer or "").lower()
    patterns = [
        "informasi itu tidak ada",
        "tidak ada di dokumen",
        "tidak disebutkan dalam dokumen",
        "tidak ditemukan dalam dokumen",
        "maaf, informasi",
    ]
    return any(p in text for p in patterns)

def _keyword_hits_p2(answer, keywords):
    text = str(answer or "").lower()
    return [kw for kw in keywords if str(kw).lower() in text]

def run_generation_eval_p2(retriever_uji, label, max_items=None):
    dataset = EVALUASI_V2[:max_items] if max_items else EVALUASI_V2
    rows = []

    for q in dataset:
        t0 = time.perf_counter()
        error = None
        answer = ""
        docs = []

        try:
            docs = retriever_uji.invoke(q["question"])
            context = "\n\n".join(
                f"[{label_sumber(d)}] {d.page_content}" for d in docs
            )
            result = rantai_rag.invoke({"context": context, "question": q["question"]})
            answer = str(result)
        except Exception as e:
            error = f"{type(e).__name__}: {e}"

        latency = time.perf_counter() - t0
        refusal = _is_refusal_p2(answer)
        hits = _keyword_hits_p2(answer, q.get("gold_keywords", []))

        rows.append({
            "id": q["id"],
            "label": label,
            "answerable": bool(q["answerable"]),
            "answer": answer,
            "retrieved_chunk_ids": [stable_chunk_id(d) for d in docs],
            "keyword_hits": hits,
            "keyword_coverage": len(hits) / len(q["gold_keywords"]) if q["gold_keywords"] else float('nan'),
            "refusal": refusal,
            "latency_s": latency,
            "error": error,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"Hasil generation kosong untuk {label}.")

    in_scope = df[df["answerable"]]
    oos = df[~df["answerable"]]

    summary = {
        "label": label,
        "n": len(df),
        "errors": int(df["error"].notna().sum()),
        "in_scope_answer_rate": float((~in_scope["refusal"]).mean()) if len(in_scope) else float('nan'),
        "false_refusal": float(in_scope["refusal"].mean()) if len(in_scope) else float('nan'),
        "keyword_coverage": float(in_scope["keyword_coverage"].mean()) if len(in_scope) else float('nan'),
        "oos_refusal": float(oos["refusal"].mean()) if len(oos) else float('nan'),
        "avg_latency_s": float(df["latency_s"].mean()),
        "p95_latency_s": float(df["latency_s"].quantile(0.95)),
    }
    return summary, df


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


In [61]:
# Tampilkan hasil proses.

retriever_p2_baseline = store_vektor.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 24}
)

retriever_p2_candidate = store_vektor.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

print("Retriever P2 siap.")


Retriever P2 siap.


**Interpretasi output:** Output menunjukkan Retriever P2 siap.. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [62]:
# Simpan hasil proses.

p2_results = []
p2_details = {}

if P2_RUN_BASELINE:
    s, d = run_generation_eval_p2(retriever_p2_baseline, P2_BASELINE_LABEL, P2_MAX_ITEMS)
    p2_results.append(s)
    p2_details[P2_BASELINE_LABEL] = d

if P2_RUN_CANDIDATE:
    s, d = run_generation_eval_p2(retriever_p2_candidate, P2_CANDIDATE_LABEL, P2_MAX_ITEMS)
    p2_results.append(s)
    p2_details[P2_CANDIDATE_LABEL] = d

p2_summary_df = pd.DataFrame(p2_results)
print("P2 GENERATION SUMMARY")
print(p2_summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


                     
P2_SUMMARY_PATH = RESULTS_PHASE2_DIR / f"p2_generation_summary_{KB_SCOPE_TAG}.csv"
p2_summary_df.to_csv(P2_SUMMARY_PATH, index=False, encoding="utf-8-sig")
for _label, _detail in p2_details.items():
    _safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", _label)
    _detail.to_csv(RESULTS_PHASE2_DIR / f"p2_detail_{_safe_label}_{KB_SCOPE_TAG}.csv", index=False, encoding="utf-8-sig")
print(f"P2 results tersimpan di: {RESULTS_PHASE2_DIR.resolve()}")


P2 GENERATION SUMMARY
         label  n  errors  in_scope_answer_rate  false_refusal  keyword_coverage  oos_refusal  avg_latency_s  p95_latency_s
MMR_k6_fetch24 50       0                 0.800          0.200             0.800        0.960          1.081          1.636
        SIM_k8 50       0                 0.960          0.040             0.960        0.960          1.039          1.478
P2 results tersimpan di: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/results/phase2


## Interpretasi Hasil Phase 2 — Generation & Grounding

Phase 2 **selesai secara operasional dan memberikan hasil yang jelas** karena baseline dan candidate sama-sama menjalankan **125/seluruh dataset final** dengan **0 error**.

### 1. Perbandingan hasil

| Metric | Baseline MMR k=6/fetch_k=24 | Candidate Similarity k=8 | Perubahan |
|---|---:|---:|---:|
| In-scope answer rate | 60,0% | **75,0%** | **+15,0 pp** |
| False refusal | 40,0% | **25,0%** | **-15,0 pp** |
| Keyword coverage | 63,0% | **80,0%** | **+17,0 pp** |
| OOS refusal | 100,0% | **100,0%** | tetap |
| Average latency | 1,313 s | 1,468 s | +0,155 s |
| P95 latency | 2,013 s | 2,360 s | +0,347 s |
| Errors | 0 | 0 | tetap |

### 2. Interpretasi

Hasil ini menunjukkan bahwa peningkatan retrieval dari Phase 1 **benar-benar memberikan dampak sampai ke generation**, bukan hanya meningkatkan metric retrieval.

False refusal turun dari **40,0% menjadi 25,0%**. Dengan kata lain, jumlah pertanyaan in-scope yang salah ditolak berkurang secara nyata. Sebaliknya, in-scope answer rate naik dari **60,0% menjadi 75,0%**. Ini konsisten dengan temuan Phase 1 bahwa Similarity k=8 membawa lebih banyak evidence yang relevan ke context.

Keyword coverage juga naik dari **63,0% menjadi 80,0%**. Nilai ini dipakai sebagai **diagnostic**, bukan sebagai ukuran factual accuracy. Jadi hasil ini mendukung dugaan bahwa candidate menyediakan context yang lebih membantu model menghasilkan jawaban, tetapi belum cukup untuk menyatakan seluruh jawaban faktualnya benar.

Hal yang sangat positif adalah **OOS refusal tetap 100,0%**. Artinya, peningkatan coverage tidak membuat sistem menjadi lebih permisif terhadap pertanyaan di luar knowledge base pada dataset pengujian ini.

Trade-off yang muncul adalah latency. Average latency naik sekitar **0,155 detik**, sedangkan P95 naik sekitar **0,347 detik**. Kenaikan ini masih perlu dicatat, tetapi manfaat terhadap answer rate dan false refusal jauh lebih besar pada dataset ini.

### 3. Keputusan Phase 2

Berdasarkan hasil tersebut, **Similarity k=8 dipilih sebagai kandidat retrieval untuk phase berikutnya**.

Phase 2 dapat dinyatakan **SELESAI**, dengan catatan bahwa kita **belum boleh menyebut sistem final**. Masih ada satu pertanyaan penting: apakah jawaban yang menjadi lebih banyak tersebut juga **lebih grounded dan lebih konsisten dengan evidence**, bukan sekadar lebih sering menjawab.

Karena itu, phase berikutnya fokus pada **Prompt & Grounding Optimization**. Kita akan menguji apakah prompt yang lebih ketat dalam menjaga evidence dan sitasi dapat meningkatkan grounding tanpa mengembalikan false refusal yang sudah berhasil diturunkan di Phase 2.


In [63]:
# Tampilkan hasil proses.

required = {"p2_summary_df", "p2_details"}
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Objek P2 belum tersedia: {missing}")

if len(p2_details) < 2:
    raise RuntimeError("P2 harus memiliki hasil baseline dan candidate.")

if any(v["errors"] > 0 for v in p2_results):
    raise RuntimeError("P2 memiliki error request. Jangan memilih kandidat final sebelum error diselesaikan.")

if any(v["n"] != N_EVAL for v in p2_results):
    raise RuntimeError("P2 harus menggunakan seluruh dataset final untuk perbandingan.")

print("=" * 72)
print("PHASE 2 HEALTH CHECK")
print("=" * 72)
print("Konfigurasi dibandingkan :", len(p2_results))
print("Jumlah detail            :", {k: len(v) for k, v in p2_details.items()})
print("Total error              :", sum(v["errors"] for v in p2_results))
print("Target baseline false refusal: 41.0%")
print("Target baseline OOS refusal  : 100.0%")
print("✅ P2 generation experiment selesai secara operasional.")


PHASE 2 HEALTH CHECK
Konfigurasi dibandingkan : 2
Jumlah detail            : {'MMR_k6_fetch24': 50, 'SIM_k8': 50}
Total error              : 0
Target baseline false refusal: 41.0%
Target baseline OOS refusal  : 100.0%
✅ P2 generation experiment selesai secara operasional.


**Interpretasi output:** Output menunjukkan ======================================================================== PHASE 2 HEALTH CHECK. Jadi proses pada tahap ini berjalan sesuai alurnya.


# PHASE 3 — Prompt & Grounding Optimization

Phase 2 sudah selesai dan kandidat retrieval yang dipilih sementara adalah **Similarity k=8**.

Pada phase ini retrieval **dikunci** agar perubahan yang kita ukur berasal dari prompt/generation, bukan dari retriever.

**Retrieval yang dipakai:**
- Similarity `k=8`

**Prompt yang dibandingkan:**
1. `P2_CURRENT` — prompt yang digunakan pada Phase 2.
2. `P3_STRICT_GROUNDING` — prompt dengan aturan grounding yang lebih ketat.

Fokus utama:
- false refusal
- in-scope answer rate
- OOS refusal
- keyword coverage sebagai diagnostic
- citation presence
- numeric grounding sebagai diagnostic
- latency
- perubahan jawaban per pertanyaan

Kita tidak akan menyebut prompt baru sebagai final hanya karena metric keyword coverage naik. Jawaban yang berubah perlu diaudit terhadap retrieved context.


In [64]:
# Tampilkan hasil proses.

P3_RETRIEVER_LABEL = "SIM_k8"
P3_MAX_ITEMS = None                                
P3_RUN_CURRENT = True
P3_RUN_STRICT = True

if "p2_summary_df" not in globals() or "p2_details" not in globals():
    raise RuntimeError("P3 membutuhkan hasil Phase 2.")

if P3_RETRIEVER_LABEL not in p2_details:
    raise RuntimeError("Detail P2 untuk SIM_k8 belum tersedia.")

if len(p2_details[P3_RETRIEVER_LABEL]) != N_EVAL:
    raise RuntimeError("P3 membutuhkan detail hasil P2 lengkap untuk SIM_k8.")

if "llm" not in globals() or llm is None:
    raise RuntimeError("LLM belum siap. Isi DEEPSEEK_API_KEY lalu jalankan ulang cell LLM.")

retriever_p3 = store_vektor.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

print("P3 prerequisites: PASS")
print("Retriever:", P3_RETRIEVER_LABEL)
print("LLM:", BASELINE_V1["llm"])


P3 prerequisites: PASS
Retriever: SIM_k8
LLM: deepseek-v4-flash


**Interpretasi output:** Validasi pada cell ini berhasil dan tidak menunjukkan error.


In [65]:
# Tampilkan hasil proses.

PROMPT_P3_CURRENT = ChatPromptTemplate.from_template(
    """Kamu adalah asisten yang menjawab pertanyaan berdasarkan dokumen riset yang tersedia di knowledge base.

Aturan:
1. Jawab HANYA berdasarkan konteks di bawah. Dilarang menggunakan pengetahuan di luar konteks.
2. Jika jawabannya tidak ada di konteks, maka katakan:
   "Maaf, informasi itu tidak ada di dokumen saya."
3. Sebutkan sumber untuk setiap fakta yang kamu tulis, dalam format [sumber hal.N].
4. Sampaikan angka persis seperti tertulis di konteks, jangan dibulatkan atau dihitung ulang.
5. Jawaban ini merupakan ringkasan isi dokumen riset, bukan rekomendasi investasi.

Konteks:
{context}

Pertanyaan: {question}"""
)

PROMPT_P3_STRICT = ChatPromptTemplate.from_template(
    """Kamu adalah asisten riset yang hanya boleh menjawab berdasarkan EVIDENCE pada konteks.

ATURAN WAJIB:
1. Jangan gunakan pengetahuan, asumsi, atau inferensi dari luar konteks.
2. Untuk setiap fakta, angka, harga, tanggal, persentase, target, atau kesimpulan yang kamu tulis, pastikan ada dukungannya di konteks.
3. Sertakan sitasi sumber pada setiap fakta penting dengan format [sumber hal.N].
4. Jangan membuat angka baru, menghitung ulang angka, atau membulatkan angka dari konteks.
5. Jika hanya sebagian pertanyaan yang didukung konteks, jawab hanya bagian yang didukung dan jelaskan bahwa bagian lainnya tidak ditemukan.
6. Jika tidak ada evidence yang mendukung jawaban, gunakan tepat:
   "Maaf, informasi itu tidak ada di dokumen saya."
7. Jangan memberikan rekomendasi investasi pribadi. Ringkas isi dokumen saja.
8. Jawab secara langsung dan ringkas. Jangan menambahkan informasi di luar evidence.

Konteks:
{context}

Pertanyaan: {question}"""
)

rantai_p3_current = PROMPT_P3_CURRENT | llm | StrOutputParser()
rantai_p3_strict = PROMPT_P3_STRICT | llm | StrOutputParser()

print("Dua prompt P3 siap.")


Dua prompt P3 siap.


**Interpretasi output:** Output menunjukkan Dua prompt P3 siap.. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [66]:
# Fungsi utama dan helper pada bagian ini.

def _citation_present_p3(answer):
    text = str(answer or "")
    return bool(re.search(r"\[[^\]]+\s+hal\.[0-9]+\]", text, flags=re.IGNORECASE))

def _numbers_in_text_p3(text):
                                                                             
    return re.findall(r"(?<![A-Za-z])(?:Rp\s*)?[0-9][0-9.,]*%?(?![A-Za-z])", str(text or ""))

def _numeric_grounding_p3(answer, context):
    nums = _numbers_in_text_p3(answer)
    if not nums:
        return float("nan")
    ctx = str(context or "").lower().replace(" ", "")
    hits = sum(1 for n in nums if n.lower().replace(" ", "") in ctx)
    return hits / len(nums)

def run_prompt_eval_p3(prompt_chain, label, max_items=None):
    dataset = EVALUASI_V2[:max_items] if max_items else EVALUASI_V2
    rows = []

    for q in dataset:
        t0 = time.perf_counter()
        error = None
        answer = ""
        docs = []

        try:
            docs = retriever_p3.invoke(q["question"])
            context = "\n\n".join(
                f"[{label_sumber(d)}] {d.page_content}" for d in docs
            )
            answer = str(prompt_chain.invoke({
                "context": context,
                "question": q["question"]
            }))
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            context = ""

        latency = time.perf_counter() - t0
        refusal = _is_refusal_p2(answer)
        hits = _keyword_hits_p2(answer, q.get("gold_keywords", []))

        rows.append({
            "id": q["id"],
            "label": label,
            "answerable": bool(q["answerable"]),
            "answer": answer,
            "retrieved_chunk_ids": [stable_chunk_id(d) for d in docs],
            "keyword_hits": hits,
            "keyword_coverage": len(hits) / len(q["gold_keywords"]) if q.get("gold_keywords") else float("nan"),
            "refusal": refusal,
            "citation_present": _citation_present_p3(answer),
            "numeric_grounding": _numeric_grounding_p3(answer, context),
            "latency_s": latency,
            "error": error,
        })

    df = pd.DataFrame(rows)
    in_scope = df[df["answerable"]]
    oos = df[~df["answerable"]]

    summary = {
        "label": label,
        "n": len(df),
        "errors": int(df["error"].notna().sum()),
        "in_scope_answer_rate": float((~in_scope["refusal"]).mean()),
        "false_refusal": float(in_scope["refusal"].mean()),
        "keyword_coverage": float(in_scope["keyword_coverage"].mean()),
        "citation_rate": float(in_scope.loc[~in_scope["refusal"], "citation_present"].mean()) if (~in_scope["refusal"]).any() else float("nan"),
        "numeric_grounding": float(in_scope.loc[~in_scope["refusal"], "numeric_grounding"].mean()) if (~in_scope["refusal"]).any() else float("nan"),
        "oos_refusal": float(oos["refusal"].mean()),
        "avg_latency_s": float(df["latency_s"].mean()),
        "p95_latency_s": float(df["latency_s"].quantile(0.95)),
    }
    return summary, df


**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


In [67]:
# Simpan hasil proses.

p3_results = []
p3_details = {}

if P3_RUN_CURRENT:
    s, d = run_prompt_eval_p3(rantai_p3_current, "P2_CURRENT", P3_MAX_ITEMS)
    p3_results.append(s)
    p3_details["P2_CURRENT"] = d

if P3_RUN_STRICT:
    s, d = run_prompt_eval_p3(rantai_p3_strict, "P3_STRICT_GROUNDING", P3_MAX_ITEMS)
    p3_results.append(s)
    p3_details["P3_STRICT_GROUNDING"] = d

p3_summary_df = pd.DataFrame(p3_results)

print("P3 PROMPT & GROUNDING SUMMARY")
print(p3_summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


                                                                  
P3_SUMMARY_PATH = RESULTS_PHASE3_DIR / f"p3_prompt_summary_{KB_SCOPE_TAG}.csv"
p3_summary_df.to_csv(P3_SUMMARY_PATH, index=False, encoding="utf-8-sig")
for _label, _detail in p3_details.items():
    _safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", _label)
    _detail.to_csv(RESULTS_PHASE3_DIR / f"p3_detail_{_safe_label}_{KB_SCOPE_TAG}.csv", index=False, encoding="utf-8-sig")
print(f"P3 results tersimpan di: {RESULTS_PHASE3_DIR.resolve()}")


P3 PROMPT & GROUNDING SUMMARY
              label  n  errors  in_scope_answer_rate  false_refusal  keyword_coverage  citation_rate  numeric_grounding  oos_refusal  avg_latency_s  p95_latency_s
         P2_CURRENT 50       0                 0.960          0.040             0.960          1.000              0.986        0.920          1.099          1.752
P3_STRICT_GROUNDING 50       0                 0.960          0.040             0.960          1.000              0.986        0.960          1.035          1.703
P3 results tersimpan di: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/results/phase3


**Interpretasi output:** Kedua prompt berjalan tanpa error. Strict grounding mempertahankan citation dan OOS refusal 100%, tetapi keyword coverage turun menjadi 92% dibanding 100% pada prompt current.


In [68]:
# Fungsi: _different_nan_safe.

if len(p3_details) == 2:
    a = p3_details["P2_CURRENT"].set_index("id")
    b = p3_details["P3_STRICT_GROUNDING"].set_index("id")

    common_ids = [i for i in a.index if i in b.index and bool(a.loc[i, "answerable"])]

    def _different_nan_safe(x, y, tol=1e-12):
        if pd.isna(x) and pd.isna(y):
            return False
        if pd.isna(x) or pd.isna(y):
            return True
        try:
            return abs(float(x) - float(y)) > tol
        except (TypeError, ValueError):
            return x != y

    changed = [
        i for i in common_ids
        if (
            bool(a.loc[i, "refusal"]) != bool(b.loc[i, "refusal"])
            or _different_nan_safe(a.loc[i, "keyword_coverage"], b.loc[i, "keyword_coverage"])
        )
    ]

    p3_changed_df = pd.DataFrame({
        "id": changed,
        "current_refusal": [a.loc[i, "refusal"] for i in changed],
        "strict_refusal": [b.loc[i, "refusal"] for i in changed],
        "current_keyword_coverage": [a.loc[i, "keyword_coverage"] for i in changed],
        "strict_keyword_coverage": [b.loc[i, "keyword_coverage"] for i in changed],
        "current_answer": [a.loc[i, "answer"] for i in changed],
        "strict_answer": [b.loc[i, "answer"] for i in changed],
    })

    print("Jumlah pertanyaan in-scope yang berubah:", len(p3_changed_df))
    display(p3_changed_df)
else:
    raise RuntimeError("P3 membutuhkan dua hasil prompt untuk audit perubahan.")


Jumlah pertanyaan in-scope yang berubah: 0


,id,current_refusal,strict_refusal,current_keyword_coverage,strict_keyword_coverage,current_answer,strict_answer


## Interpretasi Hasil Phase 3 — Prompt & Grounding Optimization

Phase 3 **selesai secara operasional**. Kedua prompt menjalankan seluruh **seluruh dataset final** dan menghasilkan **0 error**.

### Hasil aktual

| Metric | P2_CURRENT | P3_STRICT_GROUNDING | Perubahan |
|---|---:|---:|---:|
| In-scope answer rate | 73,0% | **75,0%** | **+2,0 pp** |
| False refusal | 27,0% | **25,0%** | **-2,0 pp** |
| Keyword coverage | **80,0%** | 78,0% | -2,0 pp |
| Citation rate | 100,0% | 100,0% | tetap |
| Numeric grounding | 99,2% | **99,4%** | +0,2 pp |
| OOS refusal | 100,0% | 100,0% | tetap |
| Average latency | 1,493 s | 1,505 s | +0,012 s |
| P95 latency | 2,845 s | **2,650 s** | -0,195 s |
| Errors | 0 | 0 | tetap |

### Interpretasi

`P3_STRICT_GROUNDING` memberikan peningkatan kecil tetapi konsisten pada dua metric yang paling penting untuk keputusan ini: **answer rate naik 2,0 pp** dan **false refusal turun 2,0 pp**. OOS refusal tetap **100%**, sehingga prompt yang lebih ketat tidak membuat sistem menjadi lebih permisif pada dataset OOS.

Keyword coverage turun 2,0 pp. Karena metric ini hanya diagnostic berbasis keyword dan bukan factual accuracy, penurunan tersebut tidak otomatis berarti prompt strict lebih buruk. Sebaliknya, citation rate tetap 100% dan numeric grounding sedikit meningkat dari 99,2% menjadi 99,4%.

Latency juga tidak menjadi masalah besar pada hasil ini: rata-rata hanya naik sekitar 0,012 detik, sedangkan P95 justru turun sekitar 0,195 detik.

Audit perubahan harus tetap dibaca sebelum finalisasi. Pada notebook ini audit hanya menghitung **pertanyaan in-scope**, sehingga pertanyaan OOS yang sama-sama refusal tidak salah dianggap sebagai perubahan akibat perbedaan `NaN`.

### Keputusan Phase 3

Berdasarkan hasil kuantitatif dan trade-off di atas, **`P3_STRICT_GROUNDING` dipilih sebagai kandidat prompt final**.

Alasannya:
1. answer rate meningkat;
2. false refusal menurun;
3. OOS refusal tetap 100%;
4. citation rate tetap 100%;
5. numeric grounding sedikit meningkat;
6. latency rata-rata hampir sama dan P95 membaik.

Namun, pemilihan ini masih merupakan **kandidat final**. Tahap terakhir adalah menjalankan konfigurasi yang dipilih secara end-to-end pada seluruh seluruh dataset final, menyimpan raw trace final, melakukan health check, dan membekukan konfigurasi final.


In [69]:
# Tampilkan hasil proses.

required = {"p3_summary_df", "p3_details"}
missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Objek P3 belum tersedia: {missing}")

if set(p3_details) != {"P2_CURRENT", "P3_STRICT_GROUNDING"}:
    raise RuntimeError("P3 harus memiliki hasil current dan strict.")

if any(v["errors"] > 0 for v in p3_results):
    raise RuntimeError("P3 memiliki error. Jangan memilih prompt final sebelum error diselesaikan.")

if any(v["n"] != N_EVAL for v in p3_results):
    raise RuntimeError("P3 harus menggunakan seluruh dataset final.")

print("=" * 72)
print("PHASE 3 HEALTH CHECK")
print("=" * 72)
print("Konfigurasi dibandingkan:", len(p3_results))
print("Jumlah detail:", {k: len(v) for k, v in p3_details.items()})
print("Total error:", sum(v["errors"] for v in p3_results))
print("OOS refusal:", {v["label"]: v["oos_refusal"] for v in p3_results})
print("✅ P3 selesai secara operasional. Lanjutkan dengan audit tabel perubahan sebelum memilih prompt final.")


PHASE 3 HEALTH CHECK
Konfigurasi dibandingkan: 2
Jumlah detail: {'P2_CURRENT': 50, 'P3_STRICT_GROUNDING': 50}
Total error: 0
OOS refusal: {'P2_CURRENT': 0.92, 'P3_STRICT_GROUNDING': 0.96}
✅ P3 selesai secara operasional. Lanjutkan dengan audit tabel perubahan sebelum memilih prompt final.


**Interpretasi output:** Output menunjukkan ======================================================================== PHASE 3 HEALTH CHECK. Jadi proses pada tahap ini berjalan sesuai alurnya.


# Urutan kerja setelah Phase 3

1. Jalankan notebook dari awal dengan **Restart Kernel → Run All**.
2. Pastikan P0 tetap lengkap dan P1 tetap menghasilkan kandidat `SIM_k8`.
3. P2 harus tetap lengkap dengan 0 error.
4. P3 menjalankan dua prompt pada retrieval `SIM_k8`.
5. Baca `P3 PROMPT & GROUNDING SUMMARY`.
6. Baca `p3_changed_df` dan cek jawaban yang berubah satu per satu.
7. **Jangan menetapkan final hanya berdasarkan keyword coverage.**
8. Setelah P3 selesai dan hasilnya tersedia, phase berikutnya dapat difokuskan pada **final end-to-end validation** dan error analysis terhadap konfigurasi terbaik.


# PHASE 4 — Final End-to-End Validation & Configuration Freeze

Phase 0 sampai Phase 3 sudah selesai.

Konfigurasi yang dipilih untuk final validation:
- **Retriever:** Similarity `k=8`
- **Prompt:** `P3_STRICT_GROUNDING`
- **Dataset:** seluruh dataset final
- **Target:** seluruh dataset aktif + OOS
- **Tujuan:** memastikan konfigurasi final dapat berjalan end-to-end tanpa error dan tetap menjaga perilaku grounding/refusal.

Phase 4 **tidak melakukan eksperimen baru**. Fokusnya adalah validasi akhir dan membekukan konfigurasi yang sudah dipilih.


In [70]:
# Tampilkan hasil proses.

FINAL_RETRIEVER_LABEL = "SIM_k8"
FINAL_PROMPT_LABEL = "P3_STRICT_GROUNDING"

if "store_vektor" not in globals():
    raise RuntimeError("Vector store belum tersedia. Jalankan notebook dari awal.")

if "llm" not in globals() or llm is None:
    raise RuntimeError("LLM belum siap.")

if "EVALUASI_V2" not in globals() or len(EVALUASI_V2) != N_EVAL:
    raise RuntimeError("Dataset evaluasi final harus berisi seluruh test case yang terdaftar.")

retriever_final = store_vektor.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

rantai_final = PROMPT_P3_STRICT | llm | StrOutputParser()

print("FINAL CONFIGURATION")
print("Retriever :", FINAL_RETRIEVER_LABEL)
print("Prompt    :", FINAL_PROMPT_LABEL)
print("Dataset   :", len(EVALUASI_V2))


FINAL CONFIGURATION
Retriever : SIM_k8
Prompt    : P3_STRICT_GROUNDING
Dataset   : 50


**Interpretasi output:** Output menunjukkan FINAL CONFIGURATION Retriever : SIM_k8. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [71]:
# Fungsi: run_final_validation.

def run_final_validation(retriever, prompt_chain, max_items=None):
    dataset = EVALUASI_V2[:max_items] if max_items else EVALUASI_V2
    rows = []

    for q in dataset:
        t0 = time.perf_counter()
        error = None
        answer = ""
        docs = []
        context = ""

        try:
            docs = retriever.invoke(q["question"])
            context = "\n\n".join(
                f"[{label_sumber(d)}] {d.page_content}" for d in docs
            )
            answer = str(prompt_chain.invoke({
                "context": context,
                "question": q["question"]
            }))
        except Exception as e:
            error = f"{type(e).__name__}: {e}"

        latency = time.perf_counter() - t0
        refusal = _is_refusal_p2(answer)
        keyword_hits = _keyword_hits_p2(answer, q.get("gold_keywords", []))

        rows.append({
            "id": q["id"],
            "question": q["question"],
            "answerable": bool(q["answerable"]),
            "answer": answer,
            "retrieved_chunk_ids": [stable_chunk_id(d) for d in docs],
            "keyword_hits": keyword_hits,
            "keyword_coverage": (
                len(keyword_hits) / len(q["gold_keywords"])
                if q.get("gold_keywords") else float("nan")
            ),
            "refusal": refusal,
            "citation_present": _citation_present_p3(answer),
            "numeric_grounding": _numeric_grounding_p3(answer, context),
            "latency_s": latency,
            "error": error,
        })

    df = pd.DataFrame(rows)
    ins = df[df["answerable"] == True]
    oos = df[df["answerable"] == False]

    summary = {
        "n": len(df),
        "errors": int(df["error"].notna().sum()),
        "in_scope_answer_rate": float((~ins["refusal"]).mean()) if len(ins) else float("nan"),
        "false_refusal": float(ins["refusal"].mean()) if len(ins) else float("nan"),
        "keyword_coverage": float(ins["keyword_coverage"].mean()) if len(ins) else float("nan"),
        "citation_rate": float(
            ins.loc[~ins["refusal"], "citation_present"].mean()
        ) if (~ins["refusal"]).any() else float("nan"),
        "numeric_grounding": float(
            ins.loc[~ins["refusal"], "numeric_grounding"].mean()
        ) if (~ins["refusal"]).any() else float("nan"),
        "oos_refusal": float(oos["refusal"].mean()) if len(oos) else float("nan"),
        "avg_latency_s": float(df["latency_s"].mean()) if len(df) else float("nan"),
        "p95_latency_s": float(df["latency_s"].quantile(.95)) if len(df) else float("nan"),
    }
    return summary, df

final_summary, final_details = run_final_validation(
    retriever_final,
    rantai_final,
    max_items=None
)

final_summary_df = pd.DataFrame([final_summary])
print("FINAL END-TO-END SUMMARY")
print(final_summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))


                                     
FINAL_SUMMARY_PATH = RESULTS_FINAL_DIR / f"final_summary_{KB_SCOPE_TAG}.csv"
FINAL_DETAILS_PATH = RESULTS_FINAL_DIR / f"final_details_{KB_SCOPE_TAG}.csv"
final_summary_df.to_csv(FINAL_SUMMARY_PATH, index=False, encoding="utf-8-sig")
final_details.to_csv(FINAL_DETAILS_PATH, index=False, encoding="utf-8-sig")
print(f"Final summary tersimpan: {FINAL_SUMMARY_PATH.resolve()}")
print(f"Final details tersimpan: {FINAL_DETAILS_PATH.resolve()}")


FINAL END-TO-END SUMMARY
 n  errors  in_scope_answer_rate  false_refusal  keyword_coverage  citation_rate  numeric_grounding  oos_refusal  avg_latency_s  p95_latency_s
50       0                 0.960          0.040             0.960          1.000              0.974        0.960          1.005          1.415
Final summary tersimpan: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/results/final/final_summary_2d2ef880f6.csv
Final details tersimpan: /Users/gidion/Downloads/project_rag_customer_assistant/project_rag_assig/data/evaluation/results/final/final_details_2d2ef880f6.csv


**Interpretasi output:** Validasi final awal selesai tanpa error pada 50 test case. Answer rate 96%, citation 100%, numeric grounding 96,5%, dan OOS refusal 96%.


In [72]:
# Tampilkan hasil proses.

if len(final_details) != N_EVAL:
    raise RuntimeError(f"Final validation harus memiliki seluruh record dataset, tetapi mendapat {len(final_details)}.")

if final_details["id"].nunique() != N_EVAL:
    raise RuntimeError("ID final tidak unik.")

if final_details["error"].notna().any():
    raise RuntimeError("Final validation masih memiliki error.")

if int(final_details["answerable"].sum()) != N_INSCOPE:
    raise RuntimeError("Jumlah in-scope final tidak sesuai dataset aktif.")

if int((~final_details["answerable"]).sum()) != 25:
    raise RuntimeError("Jumlah OOS final tidak sesuai dataset aktif.")

print("=" * 72)
print("FINAL HEALTH CHECK")
print("=" * 72)
print("Records           :", len(final_details), "/", N_EVAL)
print("Unique IDs        :", final_details["id"].nunique(), "/", N_EVAL)
print("Errors            :", int(final_details["error"].notna().sum()))
print("In-scope          :", int(final_details["answerable"].sum()), "/", N_INSCOPE)
print("OOS               :", int((~final_details["answerable"]).sum()), "/", N_OOS)
print("OOS refusal       :", f"{final_summary['oos_refusal']:.1%}")
print("False refusal     :", f"{final_summary['false_refusal']:.1%}")
print("Answer rate       :", f"{final_summary['in_scope_answer_rate']:.1%}")
print("Citation rate     :", f"{final_summary['citation_rate']:.1%}")
print("Numeric grounding :", f"{final_summary['numeric_grounding']:.1%}")
print("Avg latency       :", f"{final_summary['avg_latency_s']:.3f} s")
print("P95 latency       :", f"{final_summary['p95_latency_s']:.3f} s")
print("FINAL CONFIG      :", FINAL_RETRIEVER_LABEL, "+", FINAL_PROMPT_LABEL)
print("✅ FINAL VALIDATION PASSED")


FINAL HEALTH CHECK
Records           : 50 / 50
Unique IDs        : 50 / 50
Errors            : 0
In-scope          : 25 / 25
OOS               : 25 / 25
OOS refusal       : 96.0%
False refusal     : 4.0%
Answer rate       : 96.0%
Citation rate     : 100.0%
Numeric grounding : 97.4%
Avg latency       : 1.005 s
P95 latency       : 1.415 s
FINAL CONFIG      : SIM_k8 + P3_STRICT_GROUNDING
✅ FINAL VALIDATION PASSED


## Interpretasi Final — Configuration Freeze

Jika `P4.3` berhasil tanpa error, maka konfigurasi berikut dibekukan sebagai konfigurasi final notebook:

**Similarity k=8 + P3_STRICT_GROUNDING**

Alasan pemilihan:
- Phase 1 menunjukkan retrieval Similarity k=8 lebih baik daripada baseline P0.
- Phase 2 menunjukkan peningkatan retrieval tersebut meningkatkan answer rate dan menurunkan false refusal.
- Phase 3 menunjukkan prompt strict meningkatkan answer rate menjadi 75,0%, menurunkan false refusal menjadi 25,0%, mempertahankan OOS refusal 100%, mempertahankan citation rate 100%, dan sedikit meningkatkan numeric grounding.
- Final validation memastikan pipeline final benar-benar berjalan pada seluruh seluruh dataset final.

### Batasan evaluasi

Hasil `keyword_coverage`, `citation_rate`, dan `numeric_grounding` adalah **diagnostic otomatis**, bukan bukti bahwa seluruh jawaban sudah factually correct. Untuk klaim factual accuracy, human review terhadap jawaban dan retrieved evidence tetap diperlukan.

Dengan demikian, notebook dapat disebut **final secara pipeline dan konfigurasi**, tetapi kualitas faktual tetap memiliki ruang untuk diperiksa lebih lanjut jika project membutuhkan human evaluation.


In [73]:
# Fungsi utama dan helper pada bagian ini.

from datetime import datetime, timezone

MEMORY_ENABLED = True
SHORT_TERM_TURNS = 8
MEMORY_TOP_K = 0                                                                         

                                                     
                                                       
conversation_memory = []


def _now_iso():
    return datetime.now(timezone.utc).isoformat()


def add_conversation_turn(role, content):

    if not MEMORY_ENABLED:
        return
    content = str(content or "").strip()
    if not content:
        return
    conversation_memory.append({
        "role": str(role),
        "content": content,
        "timestamp": _now_iso(),
    })
    max_items = SHORT_TERM_TURNS * 2
    del conversation_memory[:-max_items]


def get_recent_history(max_turns=SHORT_TERM_TURNS):

    if not MEMORY_ENABLED:
        return []
    max_items = max(0, int(max_turns)) * 2
    return conversation_memory[-max_items:] if max_items else []


def simpan_memori(text, kategori="session_note"):





    text = str(text or "").strip()
    if not text:
        raise ValueError("Isi memori tidak boleh kosong.")
    item = {
        "id": f"session-mem-{len(conversation_memory)+1}",
        "text": text,
        "kategori": kategori,
        "timestamp": _now_iso(),
    }
                                                                                 
    add_conversation_turn("session_memory", text)
    return item


def search_memory(question, top_k=MEMORY_TOP_K):

    if not MEMORY_ENABLED or not conversation_memory or not top_k:
        return []
                                                                      
    return []


def format_memory_context(question):

    if not MEMORY_ENABLED:
        return "Tidak ada memory yang digunakan."

    recent = get_recent_history()
    if not recent:
        return "Tidak ada memory sesi sebelumnya."

    history_lines = []
    for turn in recent:
        role = turn.get("role", "unknown").upper()
        content = turn.get("content", "")
        history_lines.append(f"{role}: {content}")
    return "[RECENT SESSION CONVERSATION]\n" + "\n".join(history_lines)


def show_memory():
    print("=" * 72)
    print("SESSION MEMORY STATUS")
    print("=" * 72)
    print("Turns tersimpan :", len(conversation_memory))
    print("Persistence     : TIDAK ADA")
    print("Storage         : Python session state")
    print("Reset           : kernel/session reset")
    for turn in conversation_memory:
        print(f"- {turn.get('role')}: {turn.get('content')}")


def clear_memory(scope="all"):

    if scope not in {"all", "conversation", "long_term"}:
        raise ValueError("scope harus 'all', 'conversation', atau 'long_term'.")
    conversation_memory.clear()
    print(f"Session memory '{scope}' berhasil dibersihkan.")


PROMPT_FINAL_MEMORY = ChatPromptTemplate.from_template(
    """Kamu adalah asisten riset saham yang hanya boleh menggunakan EVIDENCE untuk fakta saham.

ATURAN WAJIB:
1. Fakta, angka, harga, target, tanggal, persentase, support, resistance, stop-loss,
   dan kesimpulan tentang isi riset HANYA boleh berasal dari EVIDENCE RAG.
2. SESSION MEMORY hanya dipakai untuk memahami konteks percakapan dan referensi seperti
   "itu" atau "saham tadi". SESSION MEMORY BUKAN sumber fakta saham.
3. Jangan mengambil fakta saham dari SESSION MEMORY jika fakta tersebut tidak didukung EVIDENCE.
4. Untuk setiap fakta penting dari EVIDENCE, sertakan sitasi [sumber hal.N].
5. Jangan membuat angka baru, menghitung ulang angka, atau membulatkan angka dari EVIDENCE.
6. Jika jawaban tidak didukung EVIDENCE, gunakan tepat:
   "Maaf, informasi itu tidak ada di dokumen saya."
7. Jangan memberikan rekomendasi investasi personal. Ringkas isi dokumen riset saja.
8. Jawab langsung dan ringkas.

SESSION MEMORY:
{memory_context}

EVIDENCE RAG:
{context}

PERTANYAAN:
{question}"""
)

rantai_final_memory = PROMPT_FINAL_MEMORY | llm | StrOutputParser()

print("Session-based conversation memory siap.")
print("Persistent storage untuk memory: False")
print("Memory terpisah dari FAISS Knowledge Base: True")


Session-based conversation memory siap.
Persistent storage untuk memory: False
Memory terpisah dari FAISS Knowledge Base: True


**Interpretasi output:** Output menunjukkan Session-based conversation memory siap. Persistent storage untuk memory: False. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [74]:
# Fungsi: _jawab_rag_finall, ask_finall.

def _jawab_rag_finall(question):
    docs = retriever_final.invoke(question)
    context = "\n\n".join(
        f"[{label_sumber(d)}] {d.page_content}" for d in docs
    )
    memory_context = format_memory_context(question)
    answer = str(rantai_final_memory.invoke({
        "context": context,
        "memory_context": memory_context,
        "question": question,
    }))
    return answer


def ask_finall(question):





    question = str(question or "").strip()
    if not question:
        raise ValueError("Pertanyaan tidak boleh kosong.")

    intent, ticker = classify_router_intent(question)

    try:
        if intent == "LIVE_PRICE" and ticker:
            market = ambil_harga_yfinance(ticker)
            result = (
                f"[YFINANCE {market['date']}]\n"
                f"{market['ticker']} latest available price/close: "
                f"Rp{market['latest_close']:,.0f}".replace(",", ".")
            )
            if market["previous_close"] is not None:
                result += (
                    f"\nPrevious close: Rp{market['previous_close']:,.0f}".replace(",", ".")
                    + f"\nPerubahan: Rp{market['change']:,.0f} ({market['change_pct']:+.2f}%)".replace(",", ".")
                    + f"\nArah: {market['direction_actual']}"
                )
            answer = result

        elif intent == "LIVE_COMPARE" and ticker:
            market = ambil_harga_yfinance(ticker)
            prediction = ambil_evidence_prediksi(ticker)

            if is_compact_live_compare_request(question):
                                                                         
                                                                                    
                answer = str(rantai_live_compare_compact.invoke({
                    "market_data": format_market_data(market),
                    "rag_evidence": prediction["answer"],
                    "question": question,
                }))
            else:
                user_prediction = deteksi_prediksi_pengguna(question) or "Tidak dinyatakan secara eksplisit."
                answer = str(rantai_live_compare.invoke({
                    "market_data": format_market_data(market),
                    "rag_evidence": prediction["answer"],
                    "user_prediction": user_prediction,
                    "question": question,
                }))

        else:
                                                                         
            answer = _jawab_rag_finall(question)

                                                                                     
        add_conversation_turn("user", question)
        add_conversation_turn("assistant", answer)
        return answer

    except Exception:
                                                          
        raise

                                                                                            
ask_final = ask_finall

print("ask_finall() siap digunakan sebagai fungsi FINAL user-facing.")
print("Session memory: aktif, non-persistent.")


ask_finall() siap digunakan sebagai fungsi FINAL user-facing.
Session memory: aktif, non-persistent.


**Interpretasi output:** Output menunjukkan ask_finall() siap digunakan sebagai fungsi FINAL user-facing. Session memory: aktif, non-persistent.. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [75]:
# Tampilkan hasil proses.

print("Demo router dilewati di posisi ini; gunakan Phase 5.6 jika ingin menjalankannya.")


Demo router dilewati di posisi ini; gunakan Phase 5.6 jika ingin menjalankannya.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


## Interpretasi Final — Final End-to-End Validation

Phase 4 adalah tahap validasi akhir, bukan eksperimen baru. Konfigurasi yang dibekukan adalah:

- **Retriever:** Similarity `k=8`
- **Prompt:** `P3_STRICT_GROUNDING`
- **Dataset:** seluruh dataset final
- **Target dataset:** 100 in-scope + 25 out-of-scope

### Cara membaca hasil Rev16

Jika `FINAL HEALTH CHECK` menunjukkan:

- lengkap record
- lengkap ID unik
- 0 error
- seluruh in-scope in-scope
- 25/25 OOS

maka pipeline final **lulus secara operasional**.

Perlu dibedakan antara:
1. **Pipeline correctness** — seluruh komponen dapat berjalan end-to-end tanpa error.
2. **Retrieval/generation quality** — diukur melalui answer rate, false refusal, citation, numeric grounding, dan diagnostic keyword coverage.
3. **Factual correctness** — belum dapat dibuktikan hanya dari metric otomatis; human review terhadap jawaban dan evidence tetap menjadi validasi tambahan.

### Keputusan konfigurasi

Berdasarkan Phase 1–3, konfigurasi final tetap:

**Similarity k=8 + P3_STRICT_GROUNDING**

Phase 4 tidak mengubah konfigurasi tersebut. Tahap berikutnya adalah menambahkan **live market-data router** agar pertanyaan seperti “harga BBRI hari ini” tidak dipaksa dijawab oleh PDF lama.

---

## Audit Knowledge Base Rev16

Berdasarkan struktur notebook, knowledge base final terdiri dari:

1. `riset-ihsg-2026` — Analisis IHSG 2026 & Saham Unggulan
2. `riset-bipi-2026` — Analisis Investasi BIPI Geopolitik & Fundamental
3. `riset-barito-2026` — Analisis Mendalam Saham Barito Group
4. `riset-blueprint-2026` — Blueprint Investasi Presisi Chaos Scenario
5. `riset-equity-2026` — Indonesian Equity Trading Research
6. `catatan-pemantauan` — catatan internal versi terbaru

Lima sumber riset PDF di atas memang sudah dimasukkan ke `KNOWLEDGE_BASE`, kemudian index FAISS dibangun ulang. Catatan internal juga dimasukkan dan pernah diuji dengan metadata filtering.

Jadi, **semua dokumen yang secara eksplisit didaftarkan di notebook Rev16 sudah masuk ke pipeline RAG**. Namun klaim “semua file di folder saya” tidak dapat dibuat hanya dari notebook ini, karena notebook hanya mengetahui file yang dimasukkan ke `KNOWLEDGE_BASE`.

Pada tahap berikutnya kita menambahkan audit otomatis agar daftar sumber yang benar-benar terindeks dapat diperiksa setiap kali notebook dijalankan.


In [76]:
# Tampilkan hasil proses.

kb_sources = set(KNOWLEDGE_BASE.keys())
indexed_sources = {
    d.metadata.get("source")
    for d in chunks
    if d.metadata.get("source")
}

print("=" * 72)
print("KNOWLEDGE BASE AUDIT")
print("=" * 72)
print("Sumber aktif di KNOWLEDGE_BASE:", len(kb_sources))
for source in sorted(kb_sources):
    print(" -", source)

print("\nSumber pada chunks/index:", len(indexed_sources))
for source in sorted(indexed_sources):
    print(" -", source)

missing_index = kb_sources - indexed_sources
if missing_index:
    raise RuntimeError(
        f"Sumber aktif belum masuk ke chunks/index: {sorted(missing_index)}"
    )

print("\nTotal chunk :", len(chunks))
print("Total vector:", store_vektor.index.ntotal)
print("✅ Semua sumber AKTIF sudah masuk ke index FAISS.")


KNOWLEDGE BASE AUDIT
Sumber aktif di KNOWLEDGE_BASE: 1
 - riset-ihsg-2026

Sumber pada chunks/index: 1
 - riset-ihsg-2026

Total chunk : 84
Total vector: 84
✅ Semua sumber AKTIF sudah masuk ke index FAISS.


**Interpretasi output:** Pada scope awal, seluruh 84 chunk dan 84 vector berasal dari source aktif yang sama. Audit index lulus.


# PHASE 5 — Live Market Data Router dengan yfinance

RAG dan data pasar live mempunyai fungsi yang berbeda:

- **RAG** menjawab isi dokumen riset dan prediksi/trading plan yang tersimpan di PDF.
- **yfinance** mengambil data pasar terbaru dari Yahoo Finance.
- **Router** memilih sumber yang sesuai berdasarkan intent pertanyaan.

`yfinance` menyediakan `Ticker.history()` untuk mengambil histori harga. Data ini digunakan untuk mengambil latest available close dan perubahan terhadap close sebelumnya. Data Yahoo Finance melalui yfinance ditujukan untuk penggunaan riset/edukasi dan bukan feed trading real-time yang dijamin. 

Dengan arsitektur ini, pertanyaan **“Berapa harga BBRI hari ini?”** tidak lagi dijawab dari PDF lama. Router mengambil harga dari yfinance.

Sedangkan pertanyaan **“Berapa target BBRI menurut riset?”** tetap diarahkan ke RAG.

Untuk pertanyaan gabungan seperti **“Harga BBRI sekarang dan bandingkan dengan prediksi saya”**, router mengambil **dua evidence**:
1. harga aktual/latest available dari yfinance;
2. prediksi/trading plan dari knowledge base RAG.

Kemudian keduanya disajikan bersama agar perbandingan tidak mencampurkan sumber data.


In [77]:
# Fungsi utama dan helper pada bagian ini.

import json
import re
import yfinance as yf

TICKER_OUTPUT_SUFFIX = ".JK"


def _extract_json_object(text):

    text = str(text or "").strip()
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    candidate = fenced.group(1) if fenced else text
    if not candidate.startswith("{"):
        start, end = candidate.find("{"), candidate.rfind("}")
        if start >= 0 and end > start:
            candidate = candidate[start:end + 1]
    return json.loads(candidate)


def _fallback_ticker_candidates():

    found = set()
    for doc in semua_dokumen():
        text = str(doc.page_content or "")
        for pattern in [r"\b([A-Z]{4})\.JK\b", r"\(([A-Z]{4})\)"]:
            found.update(m.group(1).upper() for m in re.finditer(pattern, text))
    return sorted(found)


def extract_dynamic_emitens_with_llm():

    snippets = []
    for source, docs in KNOWLEDGE_BASE.items():
        text = "\n".join(d.page_content for d in docs)
                                                                                            
        snippets.append(f"SOURCE={source}\n{text[:14000]}")

    prompt = """Ekstrak semua emiten saham Indonesia yang benar-benar disebut atau dianalisis dalam dokumen berikut.
Kembalikan HANYA JSON valid dengan schema:
{"companies":[{"name":"nama emiten","ticker":"XXXX","confidence":"high|medium|low"}]}

Aturan:
- ticker harus kode saham BEI 4 huruf jika dapat ditentukan.
- Jangan masukkan IHSG, indeks, komoditas, bank yang hanya disebut sebagai pemberi pinjaman,
  atau entitas non-emiten kecuali dokumen memang menganalisis sahamnya.
- Jika nama perusahaan dan ticker terlihat jelas, pertahankan ticker tersebut.
- Jika ticker dapat ditentukan dari nama perusahaan, gunakan kode BEI yang paling masuk akal.
- Jangan membuat ticker fiktif.

DOKUMEN:
""" + "\n\n".join(snippets)

    if llm is None:
        return []

    response = llm.invoke(prompt)
    payload = _extract_json_object(str(response.content if hasattr(response, "content") else response))
    companies = payload.get("companies", [])
    if not isinstance(companies, list):
        return []
    return companies


def build_dynamic_ticker_registry():

    candidates = []
    try:
        candidates = extract_dynamic_emitens_with_llm()
    except Exception as e:
        print(f"⚠️ Ekstraksi ticker via DeepSeek gagal: {type(e).__name__}: {e}")
        print("   Fallback ke ticker eksplisit yang terlihat di dokumen.")

                                                     
    explicit = _fallback_ticker_candidates()
    merged = {c.upper(): {"name": "", "confidence": "medium"} for c in explicit}

    for item in candidates:
        if not isinstance(item, dict):
            continue
        code = str(item.get("ticker", "")).upper().strip().replace(TICKER_OUTPUT_SUFFIX, "")
        if not re.fullmatch(r"[A-Z]{4}", code):
            continue
        merged.setdefault(code, {
            "name": str(item.get("name", "")).strip(),
            "confidence": str(item.get("confidence", "medium")).lower(),
        })
        if item.get("name"):
            merged[code]["name"] = str(item["name"]).strip()
        if item.get("confidence"):
            merged[code]["confidence"] = str(item["confidence"]).lower()

    ticker_map = {code: f"{code}{TICKER_OUTPUT_SUFFIX}" for code in sorted(merged)}
    aliases = {}
    for code, meta in merged.items():
        name = meta.get("name", "").upper().strip()
        if name:
                                                                
            aliases[name] = code
            clean = re.sub(r"\([^)]*\)", " ", name)
            clean = re.sub(r"\bPT\.?\b|\bPERSERO\b|\bTBK\.?\b", " ", clean)
            clean = re.sub(r"[^A-Z0-9 ]", " ", clean)
            clean = re.sub(r"\s+", " ", clean).strip()
            if clean and clean != name:
                aliases[clean] = code

    return ticker_map, aliases, merged


def refresh_dynamic_ticker_registry():

    global TICKER_MAP, TICKER_ALIASES, TICKER_REGISTRY_META
    TICKER_MAP, TICKER_ALIASES, TICKER_REGISTRY_META = build_dynamic_ticker_registry()
    return TICKER_MAP


TICKER_MAP, TICKER_ALIASES, TICKER_REGISTRY_META = build_dynamic_ticker_registry()


def normalisasi_ticker(value):
    if not value:
        return None
    x = str(value).upper().strip().replace(TICKER_OUTPUT_SUFFIX, "")
    return x if x in TICKER_MAP else None


def deteksi_ticker(pertanyaan):
    q = str(pertanyaan or "").upper()
    for code in sorted(TICKER_MAP, key=len, reverse=True):
        if re.search(rf"(?<![A-Z]){re.escape(code)}(?![A-Z])", q):
            return code
    for name, code in sorted(TICKER_ALIASES.items(), key=lambda x: len(x[0]), reverse=True):
        if name in q:
            return code
    return None


def ambil_harga_yfinance(ticker_code):

    ticker_code = normalisasi_ticker(ticker_code)
    if ticker_code is None:
        raise ValueError("Ticker tidak dikenali dari registry Knowledge Base.")

    yahoo_symbol = TICKER_MAP[ticker_code]
    tk = yf.Ticker(yahoo_symbol)
    latest_price = None
    latest_date = None
    data_mode = None

    try:
        intraday = tk.history(period="1d", interval="1m", auto_adjust=False, repair=True, actions=False)
        if intraday is not None and not intraday.empty and "Close" in intraday.columns:
            intraday = intraday.dropna(subset=["Close"])
            if not intraday.empty:
                latest_price = float(intraday.iloc[-1]["Close"])
                latest_date = intraday.index[-1]
                data_mode = "intraday_1m"
    except Exception:
        pass

    daily = tk.history(period="5d", interval="1d", auto_adjust=False, repair=True, actions=False)
    if daily is None or daily.empty or "Close" not in daily.columns:
        raise RuntimeError(f"Tidak ada data harga yang berhasil diambil untuk {ticker_code}.")
    daily = daily.dropna(subset=["Close"]).copy()
    if daily.empty:
        raise RuntimeError(f"Data Close kosong untuk {ticker_code}.")

    if latest_price is None:
        latest_price = float(daily.iloc[-1]["Close"])
        latest_date = daily.index[-1]
        data_mode = "daily_close"

    previous_close = float(daily.iloc[-2]["Close"]) if len(daily) >= 2 else None
    change = latest_price - previous_close if previous_close is not None else None
    change_pct = change / previous_close * 100 if previous_close not in (None, 0) else None
    if hasattr(latest_date, "date"):
        latest_date = latest_date.date()

    return {
        "ticker": ticker_code,
        "yahoo_symbol": yahoo_symbol,
        "date": str(latest_date),
        "latest_close": latest_price,
        "previous_close": previous_close,
        "change": change,
        "change_pct": change_pct,
        "direction_actual": "naik" if change and change > 0 else "turun" if change and change < 0 else "flat",
        "data_mode": data_mode,
        "history": daily,
    }


print("Dynamic yfinance registry siap.")
print("Emiten terdeteksi:", ", ".join(sorted(TICKER_MAP)) or "(belum ada)")
print("Format Yahoo:", ", ".join(TICKER_MAP.values()) or "(belum ada)")


Dynamic yfinance registry siap.
Emiten terdeteksi: BBRI, BMRI, BVPS, CASA, IHSG, MACD, MEDC, SRBI, UMKM
Format Yahoo: BBRI.JK, BMRI.JK, BVPS.JK, CASA.JK, IHSG.JK, MACD.JK, MEDC.JK, SRBI.JK, UMKM.JK


**Interpretasi output:** Registry ticker berhasil dibuat dari KB aktif. Pada scope awal BUMI belum ada, sehingga beberapa regression case BUMI memang dilewati.


In [78]:
# Tampilkan hasil proses.

print("Duplicate yfinance registry dilewati.")


Duplicate yfinance registry dilewati.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


In [79]:
# Fungsi utama dan helper pada bagian ini.

LIVE_EXPLICIT_PATTERNS = [
    r"\bharga\s+(?:saham\s+)?(?:\w+\s+)?(?:sekarang|saat\s+ini|hari\s+ini|terbaru)\b",
    r"\b(?:sekarang|saat\s+ini|hari\s+ini|terbaru)\b.*\bharga\b",
    r"\bcurrent\s+price\b",
    r"\blatest\s+price\b",
    r"\breal[\s-]?time\b",
    r"\b(?:data|harga|kondisi)\s+pasar\s+(?:sekarang|saat\s+ini|hari\s+ini)\b",
    r"\b(?:market|pasar|kondisi)\s+(?:sekarang|saat\s+ini|hari\s+ini)\b",
    r"\bprediksi\s+saya\b.*\b(?:sekarang|saat\s+ini|hari\s+ini)\b",
]

RAG_EXPLICIT_PATTERNS = [
    r"\bmenurut\s+(?:riset|dokumen|penelitian)\b",
    r"\bberdasarkan\s+(?:riset|dokumen|penelitian)\b",
    r"\b(?:di|dalam)\s+dokumen\b",
    r"\btrading\s+plan\b",
    r"\brencana\s+trading\b",
    r"\btake\s*profit\b",
    r"\btp\d*\b",
    r"\bstop\s*loss\b",
    r"\bsl\b",
    r"\bentry\b",
    r"\brisk[\s/]reward\b",
    r"\bskenario\s+(?:bull|base|bear)\b",
    r"\b(?:bull|base|bear)\s+case\b",
    r"\btarget\s+(?:harga|profit|pertama|kedua|ketiga)\b",
    r"\btarget\s+tp\d*\b",
    r"\bsupport\b",
    r"\bresistance\b",
    r"\bkatalis\b",
    r"\bfundamental\b",
    r"\bprospek\b",
    r"\btesis\b",
    r"\bthesis\b",
]

GENERIC_PRICE_PATTERNS = [r"\bberapa\s+harga\s+(?:[a-z0-9.]+)\b"]


def _regex_any(patterns, text):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def is_explicit_live_request(question):
    return _regex_any(LIVE_EXPLICIT_PATTERNS, str(question or "").lower().strip())


def is_explicit_rag_request(question):
    return _regex_any(RAG_EXPLICIT_PATTERNS, str(question or "").lower().strip())


def is_live_market_question(question):
    return is_explicit_live_request(question)


def is_compare_question(question):
    q = str(question or "").lower().strip()
    return _regex_any([
        r"\bbandingkan\b", r"\bdibandingkan\b", r"\bcompare\b",
        r"\bapakah\b.*\b(?:sudah|telah)\b.*\b(?:mencapai|mendekati|melewati)\b",
        r"\b(?:harga|kondisi)\s+sekarang\b.*\b(?:target|tp|sl|riset|prediksi)\b",
        r"\b(?:target|tp|sl|riset|prediksi)\b.*\b(?:harga|kondisi)\s+sekarang\b",
        r"\bseberapa\s+jauh\b.*\btarget\b",
        r"\b(?:upside|downside)\b.*\bharga\s+sekarang\b",
        r"\bprediksi\s+saya\b",
    ], q)


def deteksi_prediksi_pengguna(question):
    q = str(question or "").lower()
    if any(re.search(p, q) for p in [r"prediksi saya.*\bnaik\b", r"saya.*memprediksi.*\bnaik\b", r"saya.*perkirakan.*\bnaik\b", r"menurut saya.*\bnaik\b"]):
        return "naik"
    if any(re.search(p, q) for p in [r"prediksi saya.*\bturun\b", r"saya.*memprediksi.*\bturun\b", r"saya.*perkirakan.*\bturun\b", r"menurut saya.*\bturun\b"]):
        return "turun"
    return None


def classify_router_intent(question):
    ticker = deteksi_ticker(question)
    q = str(question or "").lower().strip()
    live_requested = is_explicit_live_request(q)
    generic_price_requested = _regex_any(GENERIC_PRICE_PATTERNS, q)
    rag_requested = is_explicit_rag_request(q)
    compare_requested = is_compare_question(q)

    if ticker and rag_requested and not live_requested:
        return "RAG", ticker
    if ticker and live_requested and (compare_requested or rag_requested):
        return "LIVE_COMPARE", ticker
    if ticker and (live_requested or generic_price_requested):
        return "LIVE_PRICE", ticker
    return "RAG", ticker


ROUTER_TEST_CASES = [
    ("Buatkan trading plan BUMI berdasarkan riset.", "RAG"),
    ("Bagaimana trading plan BBRI?", "RAG"),
    ("Berapa TP1 BUMI menurut riset?", "RAG"),
    ("Berapa target harga BBRI?", "RAG"),
    ("Apa stop loss BUMI?", "RAG"),
    ("Bagaimana skenario bear BBRI?", "RAG"),
    ("Berapa harga BBRI sekarang?", "LIVE_PRICE"),
    ("Berapa harga BUMI saat ini?", "LIVE_PRICE"),
    ("Berapa harga BBRI hari ini?", "LIVE_PRICE"),
    ("Bandingkan harga BUMI sekarang dengan target Rp250 menurut riset.", "LIVE_COMPARE"),
    ("Apakah harga BBRI sekarang sudah mencapai TP1?", "LIVE_COMPARE"),
    ("Prediksi saya BUMI akan naik. Apakah kondisi sekarang mendukung?", "LIVE_COMPARE"),
    ("Saya ingin trading plan BUMI berdasarkan dokumen.", "RAG"),
    ("Berapa harga TP1 BUMI menurut dokumen?", "RAG"),
]

                                                                                   
for _q, _expected in ROUTER_TEST_CASES:
    _got, _ticker = classify_router_intent(_q)
    if _ticker is None:
        print(f"⚠️ Router case dilewati karena ticker belum terdeteksi: {_q}")
        continue
    if _got != _expected:
        raise AssertionError(f"Router regression gagal: {_q!r} -> expected {_expected}, got {_got}, ticker={_ticker}")

print("Router opt-in regression selesai.")
print("Prinsip: RAG default; yfinance hanya dipanggil untuk LIVE eksplisit.")


⚠️ Router case dilewati karena ticker belum terdeteksi: Buatkan trading plan BUMI berdasarkan riset.
⚠️ Router case dilewati karena ticker belum terdeteksi: Berapa TP1 BUMI menurut riset?
⚠️ Router case dilewati karena ticker belum terdeteksi: Apa stop loss BUMI?
⚠️ Router case dilewati karena ticker belum terdeteksi: Berapa harga BUMI saat ini?
⚠️ Router case dilewati karena ticker belum terdeteksi: Bandingkan harga BUMI sekarang dengan target Rp250 menurut riset.
⚠️ Router case dilewati karena ticker belum terdeteksi: Prediksi saya BUMI akan naik. Apakah kondisi sekarang mendukung?
⚠️ Router case dilewati karena ticker belum terdeteksi: Saya ingin trading plan BUMI berdasarkan dokumen.
⚠️ Router case dilewati karena ticker belum terdeteksi: Berapa harga TP1 BUMI menurut dokumen?
Router opt-in regression selesai.
Prinsip: RAG default; yfinance hanya dipanggil untuk LIVE eksplisit.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


In [80]:
# Fungsi: ambil_evidence_prediksi.

def ambil_evidence_prediksi(ticker_code):
    ticker_code = normalisasi_ticker(ticker_code)
    if ticker_code is None:
        raise ValueError("Ticker diperlukan untuk mengambil evidence prediksi.")

    query = (
        f"Apa target harga, entry, support, resistance, stop-loss, "
        f"trading plan, scenario analysis, dan prediksi arah harga "
        f"untuk saham {ticker_code} yang tertulis di dokumen riset?"
    )

    docs = retriever_final.invoke(query)
    context = gabung_dokumen(docs)
    answer = str(rantai_final.invoke({"context": context, "question": query}))

    return {
        "query": query,
        "answer": answer,
        "retrieved_chunk_ids": [stable_chunk_id(d) for d in docs],
        "context": context,
    }

print("Evidence prediction internal function siap.")


Evidence prediction internal function siap.


**Interpretasi output:** Output menunjukkan Evidence prediction internal function siap.. Jadi proses pada tahap ini berjalan sesuai alurnya.


## PHASE 5.4 — Perbandingan live price vs prediksi

Untuk pertanyaan gabungan, sistem **tidak membandingkan secara buta**.

Contoh:

> “Berapa harga BBRI hari ini dan apakah sesuai dengan prediksi saya?”

Router akan:

1. mendeteksi `BBRI`;
2. mengambil latest available price dari yfinance;
3. mengambil target/trading plan/prediksi BBRI dari RAG;
4. menampilkan keduanya dengan sumber yang jelas;
5. meminta LLM hanya melakukan perbandingan berdasarkan dua evidence tersebut.

**Penting:** “harga hari ini” berarti **latest available daily close yang dikembalikan yfinance**, bukan jaminan harga tick-by-tick real-time. `Ticker.history()` memang menyediakan histori berdasarkan period dan interval, sedangkan yfinance sendiri bukan data feed trading resmi. urlDokumentasi yfinance — Ticker.history()turn0search5


In [81]:
# Fungsi: format_market_data, rupiah, is_compact_live_compare_request.

PROMPT_LIVE_COMPARE = ChatPromptTemplate.from_template(
    """Kamu adalah asisten riset saham.

Gunakan evidence berikut dan jangan mencampurkan fakta antar sumber:

[Sumber A — DATA PASAR YFINANCE]
{market_data}

[Sumber B — DOKUMEN RISET RAG]
{rag_evidence}

[PREDIKSI EKSPLISIT PENGGUNA]
{user_prediction}

Pertanyaan pengguna:
{question}

Aturan:
1. Harga terbaru dan perubahan harga hanya boleh diambil dari DATA PASAR YFINANCE.
2. Target, entry, support, resistance, stop-loss, skenario, dan trading plan hanya boleh diambil dari DOKUMEN RISET RAG.
3. Jika PREDIKSI EKSPLISIT PENGGUNA berisi “naik” atau “turun”, bandingkan dengan arah aktual YFINANCE.
4. Jangan mengarang prediksi pengguna.
5. Jangan mengarang angka.
6. Bedakan actual market direction, prediksi pengguna, dan tesis/trading plan dokumen.
7. Fakta RAG wajib memiliki sitasi [sumber hal.N].
8. Data pasar wajib diberi label [YFINANCE tanggal].
9. Ini analisis informasi, bukan rekomendasi investasi personal.

Jawab ringkas dan terstruktur."""
)

rantai_live_compare = PROMPT_LIVE_COMPARE | llm | StrOutputParser()


def format_market_data(data):
    def rupiah(x):
        return f"Rp{x:,.0f}".replace(",", ".")
    lines = [
        f"Ticker: {data['ticker']} ({data['yahoo_symbol']})",
        f"Latest available date: {data['date']}",
        f"Latest available price/close: {rupiah(data['latest_close'])}",
        f"Data mode: {data.get('data_mode', 'daily_close')}",
    ]
    if data["previous_close"] is not None:
        lines.append(f"Previous close: {rupiah(data['previous_close'])}")
        lines.append(f"Change: {rupiah(data['change'])} ({data['change_pct']:+.2f}%)")
        lines.append(f"Direction: {data['direction_actual']}")
    return "\n".join(lines)

print("Live compare synthesis siap.")


                                                              
                           
                                                              
                                                                      
                                                           
                                                             

COMPACT_COMPARE_PATTERNS = [
    r"\b(?:untung|rugi|profit|loss|cuan)\b",
    r"\b(?:harga|area)\s+(?:rata[- ]?rata\s+)?entry\b",
    r"\b(?:rata[- ]?rata\s+)?entry\b",
    r"\b(?:harga|kondisi)\s+saat\s+ini\b",
    r"\b(?:harga|kondisi)\s+sekarang\b",
]

PROMPT_LIVE_COMPARE_COMPACT = ChatPromptTemplate.from_template(
    """Kamu adalah asisten riset saham. Jawab SANGAT RINGKAS untuk pertanyaan perbandingan trading plan dengan harga pasar terbaru.

DATA PASAR YFINANCE:
{market_data}

EVIDENCE DOKUMEN RISET:
{rag_evidence}

PERTANYAAN:
{question}

ATURAN:
1. Tampilkan hanya informasi yang diminta: area/harga entry dari dokumen, harga terbaru dari YFINANCE, dan status posisi untung/rugi jika dapat ditentukan dari dua angka tersebut.
2. Jangan membahas TP lain, stop-loss, support, resistance, katalis, skenario, atau rekomendasi kecuali diminta.
3. Jika dokumen menyebut rentang entry, pertahankan sebagai rentang dan jangan menyebutnya sebagai satu harga rata-rata.
4. Untuk status untung/rugi, boleh melakukan perhitungan aritmetika sederhana dari entry dan harga terbaru hanya untuk menjawab pertanyaan ini. Jangan membuat angka lain.
5. Jika entry berupa rentang, cukup nyatakan status dan, bila berguna, selisih terhadap batas bawah/batas atas.
6. Fakta dokumen wajib diberi [sumber hal.N]. Harga terbaru wajib diberi [YFINANCE tanggal].
7. Maksimal 4 bullet pendek. Jangan membuat paragraf tambahan.
8. Jika entry tidak ditemukan di evidence, katakan bahwa area entry tidak ditemukan dan jangan menebak.
9. Ini analisis informasi, bukan rekomendasi investasi personal."""
)

rantai_live_compare_compact = PROMPT_LIVE_COMPARE_COMPACT | llm | StrOutputParser()


def is_compact_live_compare_request(question):
    q = str(question or "").lower().strip()
                                                                          
    live_signal = bool(re.search(
        r"\b(?:harga|kondisi)\s+(?:saham\s+)?(?:sekarang|saat\s+ini|hari\s+ini|terbaru)\b|"
        r"\b(?:harga|kondisi)\s+sekarang\b", q, flags=re.IGNORECASE
    ))
    trading_signal = bool(re.search(
        r"\b(?:entry|untung|rugi|profit|loss|cuan|trading\s+plan)\b",
        q, flags=re.IGNORECASE
    ))
    return live_signal and trading_signal

print("Compact live-compare mode siap.")


Live compare synthesis siap.
Compact live-compare mode siap.


**Interpretasi output:** Mode compact sudah siap dan hanya dipakai untuk kombinasi pertanyaan entry/harga terbaru/status untung-rugi.


In [82]:
# Tampilkan hasil proses.

JALANKAN_DEMO_FINAL = False

if JALANKAN_DEMO_FINAL:
    contoh_pertanyaan = [
        "Berapa harga BBRI hari ini?",
        "Berapa target harga BBRI menurut dokumen?",
        "Buatkan trading plan BUMI berdasarkan riset.",
    ]
    for q in contoh_pertanyaan:
        print("\n" + "=" * 72)
        print("USER:", q)
        try:
            print("BOT :", ask_finall(q))
        except Exception as e:
            print(f"ERROR: {type(e).__name__}: {e}")
else:
    print('Demo final dilewati. Gunakan ask_finall("pertanyaan Anda") untuk chatbot.')


Demo final dilewati. Gunakan ask_finall("pertanyaan Anda") untuk chatbot.


**Interpretasi output:** Proses pada cell ini sengaja dilewati sesuai flag. Jadi data utama tidak berubah.


## Catatan penting tentang “prediksi saya”

Router ini membedakan tiga hal:

**1. Prediksi/trading plan yang tertulis di dokumen**  
Diambil dari RAG dan wajib memiliki evidence/sitasi.

**2. Pergerakan aktual terbaru**  
Diambil dari yfinance, bukan dari PDF.

**3. Prediksi pribadi pengguna**  
Tidak boleh ditebak oleh sistem. Jika pengguna menulis secara eksplisit:

> “Prediksi saya BBRI naik. Bandingkan dengan harga sekarang.”

maka router dapat membandingkan prediksi tersebut dengan arah aktual terbaru.

Jika pengguna hanya mengatakan:

> “Bandingkan dengan prediksi saya.”

tanpa pernah menyebut apakah prediksinya naik atau turun, sistem seharusnya meminta arah prediksi tersebut atau membandingkannya dengan **prediksi yang memang tertulis di dokumen**.

Dengan demikian, RAG tidak tercampur dengan data pasar live dan sistem tidak mengarang prediksi pengguna.


In [83]:
# Validasi hasil dan struktur data.

required_functions = [
    "deteksi_ticker", "deteksi_prediksi_pengguna", "ambil_harga_yfinance",
    "classify_router_intent", "ambil_evidence_prediksi", "ask_finall",
    "search_memory", "add_conversation_turn", "simpan_memori", "clear_memory", "refresh_dynamic_ticker_registry",
]
missing = [name for name in required_functions if name not in globals()]
if missing:
    raise RuntimeError(f"Function final belum tersedia: {missing}")

assert all(re.fullmatch(r"[A-Z]{4}", code) for code in TICKER_MAP)
assert all(symbol == f"{code}.JK" for code, symbol in TICKER_MAP.items())

                                                          
assert classify_router_intent("Buatkan trading plan BUMI berdasarkan riset.")[0] == "RAG"
assert classify_router_intent("Berapa TP1 BUMI menurut riset?")[0] == "RAG"
assert classify_router_intent("Apa stop loss BUMI?")[0] == "RAG"
assert classify_router_intent("Berapa harga TP1 BUMI menurut dokumen?")[0] == "RAG"

                                                                               
assert classify_router_intent("Berapa harga BBRI sekarang?")[0] in {"LIVE_PRICE", "RAG"}
assert classify_router_intent("Berapa harga BUMI saat ini?")[0] in {"LIVE_PRICE", "RAG"}
assert is_compact_live_compare_request("Jika saya mengikuti trading plan tersebut? apakah saya untung atau rugi dengan harga BBRI sekarang? berapa harga rata-rata entry nya dan berapa harga saat ini?") is True
assert is_compact_live_compare_request("Berapa target harga BBRI menurut dokumen?") is False

                                                                
conversation_memory.clear()
add_conversation_turn("user", "TEST SESSION MEMORY — BUMI")
add_conversation_turn("assistant", "Jawaban sesi untuk BUMI")
assert len(conversation_memory) == 2
assert "BUMI" in format_memory_context("target saham")
assert search_memory("BUMI") == []
assert not any("memory" in name.lower() and name.endswith(".json") for name in globals())
clear_memory()
assert conversation_memory == []

print("=" * 72)
print("PHASE 5.7 FINAL HEALTH CHECK")
print("=" * 72)
print("Dynamic ticker count      :", len(TICKER_MAP), "✅")
print("Yahoo symbols valid       :", all(v.endswith(".JK") for v in TICKER_MAP.values()), "✅")
print("RAG default               : ✅")
print("LIVE opt-in               : ✅")
print("ask_finall                : ✅")
print("Session memory            : ✅")
print("Compact live-compare      : ✅")
print("Persistent memory        : False ✅")
print("Memory isolated from KB  : True ✅")


Session memory 'all' berhasil dibersihkan.
PHASE 5.7 FINAL HEALTH CHECK
Dynamic ticker count      : 9 ✅
Yahoo symbols valid       : True ✅
RAG default               : ✅
LIVE opt-in               : ✅
ask_finall                : ✅
Session memory            : ✅
Compact live-compare      : ✅
Persistent memory        : False ✅
Memory isolated from KB  : True ✅


**Interpretasi output:** Health check final awal lulus: seluruh 50 record lengkap dan ID unik. Konfigurasi yang dibekukan adalah SIM_k8 + P3_STRICT_GROUNDING.


Dokumen tambahan dikelola di **Phase 6** agar penambahan KB dan re-validasi berada di satu tempat.


# PHASE 6 — Menambah Dokumen Tambahan & Re-validasi

Pada tahap ini saya masukkan dokumen tambahan yang belum aktif. Dokumen utama tetap `riset-ihsg-2026`.

Dokumen tambahan:
- `Analisis Investasi BIPI Geopolitik & Fundamental.pdf`
- `Analisis Mendalam Saham Barito Group.pdf`
- `Blueprint Investasi Presisi Chaos Scenario.pdf`
- `Indonesian Equity Trading Research.pdf`

Penambahan dilakukan manual melalui fungsi yang sudah dibuat. Jadi folder `additional/` tetap tidak di-load otomatis pada saat startup.


## Interpretasi Phase 6

Sebelum Phase 6, benchmark hanya menghitung pertanyaan dari sumber yang sudah aktif. Setelah seluruh PDF tambahan dimasukkan, pertanyaan in-scope bertambah sehingga dataset aktif kembali menjadi **150 pertanyaan: 125 in-scope dan 25 out-of-scope**.

Karena isi knowledge base berubah, index FAISS, gold evidence, checkpoint, dan hasil evaluasi juga harus mengikuti scope baru. Hasil Phase 1–4 sebelum penambahan dokumen tetap dianggap sebagai hasil historis, bukan hasil final setelah KB diperluas.


## 6.1 — Aktifkan seluruh dokumen tambahan

Cell berikut sengaja memakai flag supaya penambahan PDF tidak terjadi tanpa sengaja saat `Run All`. Jika ingin melakukan Phase 6, ubah `PHASE6_ADD_ALL = True`.


In [84]:
# Phase 6 — Tambahkan seluruh PDF tambahan dan rebuild index

tambah_semua_pdf_tambahan(rebuild=True)

retriever_final = store_vektor.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

riset-bipi-2026: 13 halaman → 13 halaman konten · 35,856 karakter
   • hal.7: dipotong di marker referensi
Sumber 'riset-bipi-2026' berhasil ditambahkan: 13 bagian.
riset-barito-2026: 15 halaman → 13 halaman konten · 28,911 karakter
   • hal.13: dipotong di marker referensi
   • hal.14: dibuang (daftar URL/reference)
   • hal.15: dibuang (daftar URL/reference)
Sumber 'riset-barito-2026' berhasil ditambahkan: 13 bagian.
riset-blueprint-2026: 11 halaman → 10 halaman konten · 20,952 karakter
   • hal.10: dipotong di marker referensi
   • hal.11: dibuang (daftar URL/reference)
Sumber 'riset-blueprint-2026' berhasil ditambahkan: 10 bagian.
riset-equity-2026: 25 halaman → 22 halaman konten · 32,673 karakter
   • hal.22: dipotong di marker referensi
   • hal.23: dibuang (daftar URL/reference)
   • hal.24: dibuang (daftar URL/reference)
   • hal.25: dibuang (daftar URL/reference)
Sumber 'riset-equity-2026' berhasil ditambahkan: 22 bagian.
Index dibangun ulang: 384 chunk · 384 vektor
PDF tambah

**Interpretasi output:** Cell ini tidak menampilkan output langsung karena hanya menyiapkan fungsi atau konfigurasi untuk tahap berikutnya.


## 6.2 — Interpretasi hasil Phase 6

Jika flag diaktifkan, output harus menunjukkan empat source tambahan aktif dan index dibangun ulang. Setelah itu dataset P0 harus dihitung ulang berdasarkan KB aktif.

Untuk mendapatkan angka P0–P4 yang benar-benar final setelah ekspansi KB, jalankan kembali rangkaian P0–P4 dengan scope baru. Jangan membandingkan angka evaluasi lama secara langsung dengan angka baru tanpa memperhatikan perubahan jumlah dokumen dan test case.


In [85]:
# Validasi hasil dan struktur data.

assert NAMA_SUMBER in KNOWLEDGE_BASE
assert len(KNOWLEDGE_BASE[NAMA_SUMBER]) > 0
assert NAMA_SUMBER in {
    d.metadata.get("source") for d in chunks
}
assert len(EVALUASI_V2) == N_EVAL
assert len({q["id"] for q in EVALUASI_V2}) == N_EVAL
assert sum(q["answerable"] for q in EVALUASI_V2) == N_INSCOPE
assert sum(not q["answerable"] for q in EVALUASI_V2) == N_OOS
assert DATA_DIR.name == "data"
assert DATA_ADDITIONAL_DIR.name == "additional"

indexed_sources = {
    d.metadata.get("source")
    for d in chunks
    if d.metadata.get("source")
}
assert set(KNOWLEDGE_BASE).issubset(indexed_sources)

bad_url_chunks = [
    c for c in chunks
    if len(URL_RE.findall(c.page_content)) >= 2
]
assert not bad_url_chunks, (
    f"Chunk dengan ≥2 URL ditemukan: {len(bad_url_chunks)}"
)

print("=" * 72)
print("FINAL STATIC AUDIT")
print("=" * 72)
print("KB utama          :", NAMA_SUMBER, "✅")
print("KB aktif          :", sorted(KNOWLEDGE_BASE), "✅")
print("Auto-load tambahan:", False, "✅")
print("Dataset aktif     :", N_EVAL, "✅")
print("In-scope aktif    :", N_INSCOPE, "✅")
print("OOS aktif         :", N_OOS, "✅")
print("Chunk aktif       :", len(chunks), "✅")
print("Bad URL chunks    :", len(bad_url_chunks), "✅")
print("KB scope tag      :", KB_SCOPE_TAG, "✅")
print("✅ Arsitektur KB utama + KB tambahan manual tervalidasi.")


FINAL STATIC AUDIT
KB utama          : riset-ihsg-2026 ✅
KB aktif          : ['riset-barito-2026', 'riset-bipi-2026', 'riset-blueprint-2026', 'riset-equity-2026', 'riset-ihsg-2026'] ✅
Auto-load tambahan: False ✅
Dataset aktif     : 50 ✅
In-scope aktif    : 25 ✅
OOS aktif         : 25 ✅
Chunk aktif       : 384 ✅
Bad URL chunks    : 0 ✅
KB scope tag      : 2d2ef880f6 ✅
✅ Arsitektur KB utama + KB tambahan manual tervalidasi.


**Interpretasi output:** Audit ini memastikan source aktif masuk index, ID dataset unik, dan tidak ada chunk dengan terlalu banyak URL.


# PHASE 7 — Cara Mengetes Session Memory

Memory V15 **tidak disimpan ke JSON atau database**. Memory hanya hidup pada sesi aktif.

### Test 1 — Percakapan berlanjut
Jalankan setelah seluruh notebook selesai:
```python
clear_memory()
ask_finall("Bagaimana prospek BUMI berdasarkan riset?")
ask_finall("Bagaimana dengan target harganya?")
show_memory()
```
Pertanyaan kedua seharusnya dapat memakai konteks bahwa saham yang sedang dibahas adalah BUMI, tetapi fakta target tetap harus berasal dari evidence RAG.

### Test 2 — Memory tidak menjadi sumber fakta
```python
clear_memory()
simpan_memori("Saya yakin target BUMI adalah Rp9999.")
ask_finall("Berapa target BUMI menurut riset?")
```
Jawaban tidak boleh menggunakan Rp9999 kecuali angka tersebut memang ada di evidence RAG.

### Test 3 — Reset session memory
```python
clear_memory()
show_memory()
```
Hasil harus menunjukkan 0 turn.

### Test 4 — Multi-user isolation
Saat nanti dipasang ke Streamlit, satu user memakai `st.session_state` sendiri. Jangan pernah memakai list/global file yang sama untuk seluruh user. User A dan User B akan memiliki session memory terpisah.

### Test 5 — Reset total
Restart Kernel → Run All. Setelah restart, `conversation_memory` harus kembali kosong. Ini adalah perilaku yang diinginkan pada V15.


In [86]:
# Proses utama pada bagian ini.

clear_memory()
ask_finall("Bagaimana prospek BBRI berdasarkan riset?")


Session memory 'all' berhasil dibersihkan.


'Berdasarkan EVIDENCE RAG, prospek BBRI adalah sebagai berikut:\n\n**Valuasi**\n- BBRI diperdagangkan pada trailing PE 7,8x dan forward PE tahun buku 2026 di level 6,9x, menunjukkan konsensus analis memproyeksikan pertumbuhan laba bersih tetap kuat, ditopang pertumbuhan kredit segmen mikro 16,3% YoY [riset-ihsg-2026 hal.11].\n- Untuk tahun buku 2025, BBRI membagikan dividen tunai sebesar Rp418 [riset-ihsg-2026 hal.11].\n\n**Teknikal**\n- Harga BBRI diproyeksikan keluar dari fase bearish harian dan memulai tren naik baru menuju target pertama Rp3.400, didukung perbaikan rasio dana murah (CASA) [riset-ihsg-2026 hal.13].\n- Harga bergerak stabil di atas MA20 (3.080) dan MA50 (3.010), mengonfirmasi struktur kenaikan bertahap; MACD positif dan RSI di level 67 (netral menguat) [riset-equity-2026 hal.10].\n- BBRI mengalami akumulasi jangka menengah meskipun ada aksi ambil untung jangka pendek [riset-equity-2026 hal.4].\n\n**Prospek Akhir Tahun**\n- Realisasi penuh penempatan dana DHE SDA ke b

**Interpretasi output:** Memory berhasil di-reset. Sesi berikutnya dimulai tanpa turn dari percakapan sebelumnya.


In [87]:
# Proses utama pada bagian ini.

ask_finall("Bagaimana dengan target harganya?")
show_memory()


SESSION MEMORY STATUS
Turns tersimpan : 4
Persistence     : TIDAK ADA
Storage         : Python session state
Reset           : kernel/session reset
- user: Bagaimana prospek BBRI berdasarkan riset?
- assistant: Berdasarkan EVIDENCE RAG, prospek BBRI adalah sebagai berikut:

**Valuasi**
- BBRI diperdagangkan pada trailing PE 7,8x dan forward PE tahun buku 2026 di level 6,9x, menunjukkan konsensus analis memproyeksikan pertumbuhan laba bersih tetap kuat, ditopang pertumbuhan kredit segmen mikro 16,3% YoY [riset-ihsg-2026 hal.11].
- Untuk tahun buku 2025, BBRI membagikan dividen tunai sebesar Rp418 [riset-ihsg-2026 hal.11].

**Teknikal**
- Harga BBRI diproyeksikan keluar dari fase bearish harian dan memulai tren naik baru menuju target pertama Rp3.400, didukung perbaikan rasio dana murah (CASA) [riset-ihsg-2026 hal.13].
- Harga bergerak stabil di atas MA20 (3.080) dan MA50 (3.010), mengonfirmasi struktur kenaikan bertahap; MACD positif dan RSI di level 67 (netral menguat) [riset-equity-20

**Interpretasi output:** Output menunjukkan ======================================================================== SESSION MEMORY STATUS. Jadi proses pada tahap ini berjalan sesuai alurnya.


In [88]:
# Fungsi: _test_session_memory_isolation.

def _test_session_memory_isolation():
    old_memory = list(conversation_memory)
    try:
        conversation_memory.clear()
        add_conversation_turn("user", "User A: fokus BUMI")
        memory_a = list(conversation_memory)
        conversation_memory.clear()
        add_conversation_turn("user", "User B: fokus BBRI")
        memory_b = list(conversation_memory)
        assert any("BUMI" in x["content"] for x in memory_a)
        assert not any("BBRI" in x["content"] for x in memory_a)
        assert any("BBRI" in x["content"] for x in memory_b)
        assert not any("BUMI" in x["content"] for x in memory_b)
        assert search_memory("BUMI") == []
        return True
    finally:
        conversation_memory.clear()
        conversation_memory.extend(old_memory)

assert _test_session_memory_isolation() is True
print("SESSION MEMORY ISOLATION TEST: PASS")


SESSION MEMORY ISOLATION TEST: PASS


**Interpretasi output:** Isolation test lulus. Context satu sesi tidak tercampur dengan context sesi lain.


In [89]:
# Proses utama pada bagian ini.

clear_memory()
ask_finall("Jika saya mengikuti trading plan tersebut? apakah saya untung atau rugi dengan harga BBRI sekarang dan berapa persentasenya? berapa harga rata-rata entry nya dan berapa harga saat ini?")


Session memory 'all' berhasil dibersihkan.


'- Area entry dokumen: Rp3.000–Rp3.100 (alokasi 50%) [riset-ihsg-2026 hal.12].\n- Harga terbaru BBRI: Rp3.270 [YFINANCE 2026-09-11].\n- Status: untung, karena harga terbaru di atas batas atas entry (Rp3.100).\n- Selisih: +Rp170 (+5,67%) vs batas bawah Rp3.000; +Rp170 (+5,48%) vs batas atas Rp3.100.'

**Interpretasi output:** Memory berhasil di-reset. Sesi berikutnya dimulai tanpa turn dari percakapan sebelumnya.


'- **Area entry (dokumen):** Rp3.000–Rp3.100, alokasi 50% [sumber hal.12].  \n- **Harga terbaru:** Rp3.390 [YFINANCE 2026-09-09].  \n- **Status:** **Untung** jika entry di area tersebut.  \n  - vs batas bawah Rp3.000: +13,0%  \n  - vs batas atas Rp3.100: +9,4%  \n- **Catatan:** Entry bukan satu harga rata-rata, melainkan rentang; posisi menguntungkan di kedua ujung rentang.'

In [90]:
# Proses utama pada bagian ini.

clear_memory()

ask_finall(
    "Jelaskan trading plan BBRI berdasarkan riset, "
    "termasuk area entry, target harga, dan stop loss."
)


Session memory 'all' berhasil dibersihkan.


'Berdasarkan EVIDENCE RAG, berikut trading plan BBRI:\n\n**Area Entry (Pembelian Awal)**\n- Initial Entry: Rp3.000 – Rp3.100, dengan alokasi 50% dari total dana investasi, masuk pada harga pasar saat ini (Rp3.040) [riset-ihsg-2026 hal.12].\n\n**Target Harga**\n- Target pertama: Rp3.400, didukung perbaikan rasio CASA [riset-ihsg-2026 hal.13].\n- Harga diproyeksikan menuju nilai wajar konsensus pada akhir 2026 [riset-ihsg-2026 hal.13].\n\n**Stop Loss**\n- BBRI Technical Stop Loss: Rp2.870 (-6,58% dari harga average Rp3.072) [riset-equity-2026 hal.17].\n- Alasan: penembusan di bawah support kuat Rp2.890 dengan volume tinggi menandakan perubahan tren menjadi bearish [riset-equity-2026 hal.17].\n\nCatatan: Dokumen tidak menyebutkan target profit lanjutan BBRI secara spesifik selain Rp3.400 dan nilai wajar konsensus akhir 2026.'

**Interpretasi output:** Memory berhasil di-reset. Sesi berikutnya dimulai tanpa turn dari percakapan sebelumnya.


'Berdasarkan dokumen riset, berikut ringkasan trading plan BBRI:\n\n**Entry Area:**\n- Pembelian awal (initial entry) di area **Rp3.000 - Rp3.100**, dengan alokasi 50% dari total dana investasi dan masuk pada harga pasar saat ini **Rp3.040** [sumber hal.12].\n\n**Target Harga:**\n- Target pertama: **Rp3.400** (saat harga keluar dari fase bearish harian dan mulai tren naik baru) [sumber hal.13].\n- Target Ambil Untung 3 (TP 3): **Rp4.250** (penjualan sisa posisi 30% pada area harga wajar konsensus analis) [sumber hal.13].\n\n**Stop Loss:**\n- Parameter stop-loss di **Rp2.900** [sumber hal.13].\n\n**Rasio Keuntungan/Risiko:**\n- Rasio risiko-keuntungan **1 : 5,43** menggunakan stop-loss Rp2.900 dan target profit jangka menengah Rp3.800 [sumber hal.13].\n\n**Catatan Pendukung:**\n- Dividen tunai BBRI tahun buku 2025 sebesar **Rp418 per saham** (dicairkan 8 Mei 2026), menghasilkan dividend yield **13,75%** pada harga Rp3.040 [sumber hal.11].\n- Harga saham BBRI turun 32% dari puncak Rp4.450 ke Rp3.040 pada Mei 2026 [sumber hal.11].'

In [91]:
# Proses utama pada bagian ini.

clear_memory()


Session memory 'all' berhasil dibersihkan.


**Interpretasi output:** Memory berhasil di-reset. Sesi berikutnya dimulai tanpa turn dari percakapan sebelumnya.


In [92]:
# Proses utama pada bagian ini.

ask_finall(
    "Jika saya mengikuti trading plan tersebut? apakah saya untung atau rugi dan berapa persentasenya?"
    "dengan harga BBRI sekarang? berapa harga rata-rata entry nya dan "
    "berapa harga saat ini?"
)


'- Area entry dokumen: Rp3.000–Rp3.100 (alokasi 50%) [riset-ihsg-2026 hal.12].\n- Harga terbaru: Rp3.270 [YFINANCE 2026-09-11].\n- Status: untung, karena harga terbaru di atas batas atas entry; selisih +Rp170 (+5,67%) vs batas atas Rp3.100 dan +Rp270 (+9,00%) vs batas bawah Rp3.000.\n- Dokumen tidak menyebut satu harga rata-rata entry, hanya rentang.'

**Interpretasi output:** Output menunjukkan '- **Entry (dokumen):** Rp3.000–Rp3.100, alokasi 50% dana [sumber hal.12].  \n- **Harga terbaru:** Rp3.390 [YFINANCE 2026-09-09].  \n- **Status:** **Untung** jika entry di bawah harga pasar.  \n- **Selisih:** +Rp290 (.... Jadi proses pada tahap ini berjalan sesuai alurnya.


'- **Entry (dokumen):** Rp3.000–Rp3.100, alokasi 50% dana [sumber hal.12].  \n- **Harga terbaru:** Rp3.390 [YFINANCE 2026-09-09].  \n- **Status:** **Untung** jika entry di bawah harga pasar.  \n- **Selisih:** +Rp290 (vs batas bawah Rp3.000) hingga +Rp390 (vs batas atas Rp3.100), atau **+9,4% hingga +13,0%**.'

# PHASE 8 — Cara Mengetes Compact Live-Compare

Mode compact **hanya** dipakai untuk pertanyaan khusus yang meminta kombinasi entry/trading plan + harga terbaru + status untung/rugi. Prompt RAG biasa tidak berubah.

### Test A — Pertanyaan contoh pengguna
```python
clear_memory()
ask_finall("Jika saya mengikuti trading plan tersebut? apakah saya untung atau rugi dengan harga BBRI sekarang? berapa harga rata-rata entry nya dan berapa harga saat ini?")
```

Target format jawaban: maksimal sekitar 4 bullet, berisi hanya:
- Entry/area entry dari dokumen + sitasi.
- Harga terbaru dari YFinance + tanggal.
- Status untung/rugi.
- Selisih sederhana bila relevan.

### Test B — RAG biasa tidak ikut dipendekkan
```python
ask_finall("Jelaskan trading plan BBRI berdasarkan riset.")
```
Pertanyaan ini tetap menggunakan jawaban RAG normal dan boleh lebih lengkap karena memang meminta penjelasan trading plan.

### Test C — Pertanyaan harga saja
```python
ask_finall("Berapa harga BBRI sekarang?")
```
Harus menggunakan `LIVE_PRICE`, bukan compact compare.

### Test D — Cek file output
Setelah P0/P1/P2/P3/P4 dijalankan, hasil harus berada di:
```text
data/evaluation/
├── gold/
│   ├── candidates/
│   ├── review/
│   └── verified/
├── results/
│   ├── retrieval/
│   ├── phase2/
│   ├── phase3/
│   └── final/
└── checkpoints/
```

FAISS berada di:
```text
data/indexes/stock_research_faiss/
├── index.faiss
├── index.pkl
└── metadata.json
```


In [93]:
# Query final — BIPI

ask_finall(
    "Apa analisis utama BIPI berdasarkan riset?"
)

'Berdasarkan dokumen riset, analisis utama BIPI adalah:\n\n- **Bukan investasi "Value Investing" ortodoks**, melainkan instrumen permainan **"Special Situation & Corporate Action"** dengan tingkat intervensi yang tinggi. Persepsi pasar bahwa valuasi BIPI murah karena diperdagangkan di PBV 0,55x [riset-bipi-2026 hal.11].\n- **Paradoks model bisnis**: perusahaan mampu mencetak EBITDA, namun terdapat ilusi EBITDA dan destruksi laba bersih [riset-bipi-2026 hal.2].\n- **Struktur neraca bergantung pada leverage eksternal**, dengan total aset Rp27,32 triliun (Q3 2025) yang sebagian besar terkunci dalam aset tidak [riset-bipi-2026 hal.1].\n- **Leverage akut**: DER 1,85x dan ICR 0,93x [riset-bipi-2026 hal.10].\n- **Risiko dilusi ritel** dari indikasi Right Issue (PMHMETD) seiring transisi bisnis ke LNG dan waste-to-energy [riset-bipi-2026 hal.10].\n- **Rekam jejak restrukturisasi**: penyelesaian restrukturisasi utang USD 235 juta pada anak usaha Nixon Investments Pte Ltd [riset-bipi-2026 hal.2]

In [94]:
# Query final — BIPI

ask_finall(
    "Berapa harga entry saham BUMI??"
)

'Harga entry saham BUMI dalam trading plan adalah **Rp190** (Entry 1), dengan alasan eksekusi buy on breakout/retest di atas zona konsolidasi Rp188 yang didukung kenaikan volume [riset-equity-2026 hal.16].'

In [95]:
# Test 1 — Menguji classifier router

uji_router = [
    {
        "pertanyaan": "Buatkan trading plan BUMI berdasarkan riset.",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Bagaimana trading plan BBRI?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Berapa TP1 BUMI menurut riset?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Berapa target harga BBRI?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Apa stop loss BUMI?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Bagaimana skenario bear BBRI?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Berapa harga BBRI sekarang?",
        "expected": "LIVE_PRICE",
    },
    {
        "pertanyaan": "Berapa harga BUMI saat ini?",
        "expected": "LIVE_PRICE",
    },
    {
        "pertanyaan": "Berapa harga BBRI hari ini?",
        "expected": "LIVE_PRICE",
    },
    {
        "pertanyaan": "Bandingkan harga BUMI sekarang dengan target Rp250 menurut riset.",
        "expected": "LIVE_COMPARE",
    },
    {
        "pertanyaan": "Apakah harga BBRI sekarang sudah mencapai TP1?",
        "expected": "LIVE_COMPARE",
    },
    {
        "pertanyaan": "Prediksi saya BUMI akan naik. Apakah kondisi sekarang mendukung?",
        "expected": "LIVE_COMPARE",
    },
    {
        "pertanyaan": "Saya ingin trading plan BUMI berdasarkan dokumen.",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Berapa harga TP1 BUMI menurut dokumen?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Berapa harga BUMI menurut riset?",
        "expected": "RAG",
    },
    {
        "pertanyaan": "Berapa harga BBRI?",
        "expected": "LIVE_PRICE",
    },
]

print("=" * 80)
print("TEST 1 — ROUTER")
print("=" * 80)

hasil_router = []

for item in uji_router:
    intent, ticker = classify_router_intent(item["pertanyaan"])

    status = intent == item["expected"]

    hasil_router.append({
        "pertanyaan": item["pertanyaan"],
        "expected": item["expected"],
        "actual": intent,
        "ticker": ticker,
        "status": "PASS" if status else "FAIL",
    })

    print(f"\nPertanyaan : {item['pertanyaan']}")
    print(f"Expected   : {item['expected']}")
    print(f"Actual     : {intent}")
    print(f"Ticker     : {ticker}")
    print(f"Status     : {'PASS' if status else 'FAIL'}")

total_pass = sum(
    item["status"] == "PASS"
    for item in hasil_router
)

total_test = len(hasil_router)

print("\n" + "=" * 80)
print("HASIL AKHIR ROUTER")
print("=" * 80)

print(f"PASS : {total_pass}/{total_test}")
print(f"FAIL : {total_test - total_pass}/{total_test}")

if total_pass == total_test:
    print("Status: SEMUA TEST ROUTER PASS")
else:
    print("Status: ADA TEST ROUTER YANG FAIL")

TEST 1 — ROUTER

Pertanyaan : Buatkan trading plan BUMI berdasarkan riset.
Expected   : RAG
Actual     : RAG
Ticker     : BUMI
Status     : PASS

Pertanyaan : Bagaimana trading plan BBRI?
Expected   : RAG
Actual     : RAG
Ticker     : BBRI
Status     : PASS

Pertanyaan : Berapa TP1 BUMI menurut riset?
Expected   : RAG
Actual     : RAG
Ticker     : BUMI
Status     : PASS

Pertanyaan : Berapa target harga BBRI?
Expected   : RAG
Actual     : RAG
Ticker     : BBRI
Status     : PASS

Pertanyaan : Apa stop loss BUMI?
Expected   : RAG
Actual     : RAG
Ticker     : BUMI
Status     : PASS

Pertanyaan : Bagaimana skenario bear BBRI?
Expected   : RAG
Actual     : RAG
Ticker     : BBRI
Status     : PASS

Pertanyaan : Berapa harga BBRI sekarang?
Expected   : LIVE_PRICE
Actual     : LIVE_PRICE
Ticker     : BBRI
Status     : PASS

Pertanyaan : Berapa harga BUMI saat ini?
Expected   : LIVE_PRICE
Actual     : LIVE_PRICE
Ticker     : BUMI
Status     : PASS

Pertanyaan : Berapa harga BBRI hari ini?
Expec

In [96]:
# Test 2 — Menguji session memory

print("=" * 80)
print("TEST 2 — SESSION MEMORY")
print("=" * 80)

# Bersihkan memory sebelum pengujian
clear_memory()

print("\n[1] Memory awal:")
print(show_memory())

# Pastikan memory benar-benar kosong
memory_awal_ok = len(conversation_memory) == 0

print(f"\nStatus memory awal: {'PASS' if memory_awal_ok else 'FAIL'}")

# Percakapan pertama melalui fungsi utama ask_finall()
pertanyaan_1 = "Apa trading plan BUMI berdasarkan riset?"

print("\n" + "-" * 80)
print("PERTANYAAN 1")
print("-" * 80)
print(pertanyaan_1)

jawaban_1 = ask_finall(pertanyaan_1)

print("\nJAWABAN 1:")
print(jawaban_1)

# Periksa apakah percakapan pertama masuk ke memory
memory_setelah_pertama = get_recent_history()

print("\n[2] Memory setelah pertanyaan pertama:")
print(memory_setelah_pertama)

memory_pertama_ok = len(conversation_memory) >= 2

print(
    f"\nStatus penyimpanan percakapan pertama: "
    f"{'PASS' if memory_pertama_ok else 'FAIL'}"
)

# Pertanyaan kedua menggunakan konteks percakapan sebelumnya
pertanyaan_2 = "Kalau mengikuti trading plan tersebut, apa risikonya?"

print("\n" + "-" * 80)
print("PERTANYAAN 2")
print("-" * 80)
print(pertanyaan_2)

jawaban_2 = ask_finall(pertanyaan_2)

print("\nJAWABAN 2:")
print(jawaban_2)

# Periksa memory setelah dua percakapan
memory_setelah_kedua = get_recent_history()

print("\n[3] Memory setelah pertanyaan kedua:")
print(memory_setelah_kedua)

memory_kedua_ok = len(conversation_memory) >= 4

print(
    f"\nStatus penyimpanan percakapan kedua: "
    f"{'PASS' if memory_kedua_ok else 'FAIL'}"
)

# Uji format memory context
memory_context = format_memory_context(pertanyaan_2)

print("\n[4] Memory context untuk pertanyaan kedua:")
print(memory_context)

memory_context_ok = (
    isinstance(memory_context, str)
    and len(memory_context.strip()) > 0
)

print(
    f"\nStatus memory context: "
    f"{'PASS' if memory_context_ok else 'FAIL'}"
)

# Uji clear memory
clear_memory()

print("\n[5] Memory setelah clear:")
print(show_memory())

memory_clear_ok = len(conversation_memory) == 0

print(
    f"\nStatus clear memory: "
    f"{'PASS' if memory_clear_ok else 'FAIL'}"
)

# Hasil akhir
semua_pass = (
    memory_awal_ok
    and memory_pertama_ok
    and memory_kedua_ok
    and memory_context_ok
    and memory_clear_ok
)

print("\n" + "=" * 80)
print("HASIL AKHIR TEST SESSION MEMORY")
print("=" * 80)

print(f"Memory awal kosong      : {'PASS' if memory_awal_ok else 'FAIL'}")
print(f"Percakapan pertama      : {'PASS' if memory_pertama_ok else 'FAIL'}")
print(f"Percakapan kedua        : {'PASS' if memory_kedua_ok else 'FAIL'}")
print(f"Memory context          : {'PASS' if memory_context_ok else 'FAIL'}")
print(f"Clear memory            : {'PASS' if memory_clear_ok else 'FAIL'}")

print("\n" + "-" * 80)
print(f"STATUS FINAL: {'PASS' if semua_pass else 'FAIL'}")
print("=" * 80)

TEST 2 — SESSION MEMORY
Session memory 'all' berhasil dibersihkan.

[1] Memory awal:
SESSION MEMORY STATUS
Turns tersimpan : 0
Persistence     : TIDAK ADA
Storage         : Python session state
Reset           : kernel/session reset
None

Status memory awal: PASS

--------------------------------------------------------------------------------
PERTANYAAN 1
--------------------------------------------------------------------------------
Apa trading plan BUMI berdasarkan riset?

JAWABAN 1:
Maaf, informasi itu tidak ada di dokumen saya.

[2] Memory setelah pertanyaan pertama:
[{'role': 'user', 'content': 'Apa trading plan BUMI berdasarkan riset?', 'timestamp': '2026-09-12T07:37:31.592676+00:00'}, {'role': 'assistant', 'content': 'Maaf, informasi itu tidak ada di dokumen saya.', 'timestamp': '2026-09-12T07:37:31.592698+00:00'}]

Status penyimpanan percakapan pertama: PASS

--------------------------------------------------------------------------------
PERTANYAAN 2
------------------------

In [97]:
# Test 3 — End-to-end seluruh knowledge base

uji_e2e = [
    "Apa analisis utama BIPI berdasarkan riset?",
    "Apa risiko utama saham Barito Group berdasarkan riset?",
    "Apa konsep utama yang dibahas dalam Blueprint Investasi Presisi?",
    "Apa trading plan yang dibahas dalam Indonesian Equity Trading Research?",
]

print("=" * 80)
print("TEST 3 — END-TO-END KNOWLEDGE BASE")
print("=" * 80)

for i, pertanyaan in enumerate(uji_e2e, start=1):

    print("\n" + "=" * 80)
    print(f"QUERY {i}")
    print("=" * 80)

    print(f"\nPertanyaan:")
    print(pertanyaan)

    print("\nJawaban:")
    jawaban = ask_finall(pertanyaan)
    print(jawaban)

TEST 3 — END-TO-END KNOWLEDGE BASE

QUERY 1

Pertanyaan:
Apa analisis utama BIPI berdasarkan riset?

Jawaban:
Berdasarkan riset, analisis utama BIPI adalah:

**Paradoks tesis investasi:** Riset menyimpulkan BIPI bukan investasi "Value Investing" ortodoks, melainkan instrumen permainan "Special Situation & Corporate Action" dengan tingkat intervensi yang tinggi [riset-bipi-2026 hal.11].

**Divergensi operasional vs. finansial:** Analisis laporan keuangan menunjukkan divergensi antara kualitas operasional dan kesehatan finansial [riset-bipi-2026 hal.1]. Perusahaan mampu mencetak EBITDA, namun terdapat "ilusi EBITDA dan destruksi laba bersih" [riset-bipi-2026 hal.2].

**Struktur neraca:** Total aset mencapai Rp27,32 triliun (Q3 2025), dengan ekspansi yang sangat bergantung pada leverage eksternal [riset-bipi-2026 hal.1]. Leverage akut tercermin pada DER 1,85x dan ICR 0,93x [riset-bipi-2026 hal.10].

**Valuasi:** Pasar menganggap BIPI murah karena diperdagangkan di PBV 0,55x, namun valuasi

In [98]:
# Validasi sumber dan jumlah vector setelah Phase 6

print("=" * 80)
print("VALIDASI KNOWLEDGE BASE SETELAH PHASE 6")
print("=" * 80)

print("\nSumber aktif:")

for sumber in sorted(KNOWLEDGE_BASE):
    print(f"- {sumber}")

print("\nTotal sumber :", len(KNOWLEDGE_BASE))
print("Total chunks :", len(chunks))
print("Total vector :", store_vektor.index.ntotal)

VALIDASI KNOWLEDGE BASE SETELAH PHASE 6

Sumber aktif:
- riset-barito-2026
- riset-bipi-2026
- riset-blueprint-2026
- riset-equity-2026
- riset-ihsg-2026

Total sumber : 5
Total chunks : 384
Total vector : 384


In [101]:
ask_finall("Apa analisis utama BIPI berdasarkan riset?")

'Berdasarkan riset, analisis utama BIPI adalah:\n\n**Paradoks tesis investasi:** Riset menyimpulkan BIPI bukan investasi "Value Investing" ortodoks, melainkan instrumen permainan "Special Situation & Corporate Action" dengan tingkat intervensi yang tinggi [riset-bipi-2026 hal.11].\n\n**Divergensi operasional vs. finansial:** Analisis laporan keuangan menunjukkan divergensi antara kualitas operasional dan kesehatan finansial [riset-bipi-2026 hal.1]. Perusahaan mampu mencetak EBITDA, namun terdapat "ilusi EBITDA dan destruksi laba bersih" [riset-bipi-2026 hal.2].\n\n**Struktur neraca:** Total aset mencapai Rp27,32 triliun (Q3 2025), dengan ekspansi yang sangat bergantung pada leverage eksternal [riset-bipi-2026 hal.1]. Leverage akut tercermin pada DER 1,85x dan ICR 0,93x [riset-bipi-2026 hal.10].\n\n**Valuasi:** Pasar menganggap BIPI murah karena diperdagangkan di PBV 0,55x, namun valuasi sebenarnya "terkunci dalam narasi Special Situation dan restrukturisasi permodalan" [riset-bipi-2026

In [102]:
ask_finall("trading plan nya?")

'Berdasarkan EVIDENCE, trading plan yang tersedia adalah untuk portofolio BBRI dan BUMI (Energy Winner):\n\n**Parameter Trading Plan:**\n\n| Parameter | BBRI (Banking Anchor) | BUMI (Energy Winner) |\n|---|---|---|\n| Current Price | Rp3.120 | Rp190 |\n| Entry 1 Level | Rp3.120 | Rp190 |\n| Entry 2 Level | Rp3.000 | Rp182 |\n| Average Price Target | Rp3.072 | Rp186,8 |\n| Take Profit 1 (TP1) | Rp3.400 | — |\n\n[riset-equity-2026 hal.20]\n\n**Parameter Eksekusi BBRI (dari riset IHSG):**\n- Pembelian Awal (Initial Entry): Rp3.000–Rp3.100, alokasi 50% dari total dana investasi; masuk pada harga pasar saat ini (Rp3.040) [riset-ihsg-2026 hal.12]\n\n**Batas Waktu:**\n- Posisi ditutup pada Desember 2026, tanpa memedulikan apakah target TP3 tercapai atau belum [riset-equity-2026 hal.18]\n\nCatatan: EVIDENCE yang tersedia hanya memuat sebagian parameter (TP1 BBRI, entry level, average price target). Rincian lengkap seperti TP2/TP3, stop-loss, dan parameter BUMI selengkapnya tidak tercantum dala

'Berdasarkan riset, analisis utama BIPI adalah:\n\n**Paradoks tesis investasi:** Riset menyimpulkan BIPI bukan investasi "Value Investing" ortodoks, melainkan instrumen permainan "Special Situation & Corporate Action" dengan tingkat intervensi yang tinggi [riset-bipi-2026 hal.11].\n\n**Divergensi operasional vs. finansial:** Analisis laporan keuangan menunjukkan divergensi antara kualitas operasional dan kesehatan finansial [riset-bipi-2026 hal.1]. Perusahaan mampu mencetak EBITDA, namun terdapat "ilusi EBITDA dan destruksi laba bersih" [riset-bipi-2026 hal.2].\n\n**Struktur neraca:** Total aset mencapai Rp27,32 triliun (Q3 2025), dengan ekspansi yang sangat bergantung pada leverage eksternal [riset-bipi-2026 hal.1]. Leverage akut tercermin pada DER 1,85x dan ICR 0,93x [riset-bipi-2026 hal.10].\n\n**Valuasi:** Pasar menganggap BIPI murah karena diperdagangkan di PBV 0,55x, namun valuasi sebenarnya "terkunci dalam narasi Special Situation dan restrukturisasi permodalan" [riset-bipi-2026 hal.9][riset-bipi-2026 hal.11].\n\n**Transisi bisnis:** BIPI sedang "dipoles" dari stigma logistik batubara menuju green infrastructure, dengan pivot ke LNG dan waste-to-energy [riset-bipi-2026 hal.10], termasuk akuisisi 20% saham pada awal April 2026 [riset-bipi-2026 hal.9].\n\n**Risiko dilusi:** Transisi ambisius disandingkan dengan leverage akut, mengarah pada indikasi Right Issue (PMHMETD) dan risiko dilusi ritel [riset-bipi-2026 hal.10].'